In [ ]:
# ============================================================
# 00. ENVIRONMENT CHECK
# ============================================================

import sys
import os
import random
import numpy as np
import pandas as pd
import torch

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("GPU    : Not available")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\nRandom seed:", SEED)
print("=" * 70)

ENVIRONMENT
Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA   : True
GPU    : Tesla T4

Random seed: 42


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# 01. LOAD OFFICIAL TRAINING DATASET
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

print("=" * 70)
print("SEARCHING FOR train2.csv")
print("=" * 70)

# Search common Colab locations
search_paths = [
    "/content/train2.csv",
    "/content/drive/MyDrive/DravidianLangTech_2026/data/train2.csv",
    "/mnt/data/train2.csv",
]

found_paths = []

for path in search_paths:
    if os.path.exists(path):
        found_paths.append(path)

# Also search /content recursively
for path in glob.glob("/content/**/train2.csv", recursive=True):
    if path not in found_paths:
        found_paths.append(path)

print("Found files:")

for p in found_paths:
    print(" -", p)

if len(found_paths) == 0:
    raise FileNotFoundError(
        "train2.csv was not found. Please upload the official train2.csv file."
    )

# Use the first matching file
DATA_PATH = found_paths[0]

print("\nUsing:")
print(DATA_PATH)

# Load
df = pd.read_csv(DATA_PATH)

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df["Class"].value_counts())

print("\nClass proportions:")
print(df["Class"].value_counts(normalize=True))

print("=" * 70)

SEARCHING FOR train2.csv
Found files:
 - /content/drive/MyDrive/DravidianLangTech_2026/data/train2.csv

Using:
/content/drive/MyDrive/DravidianLangTech_2026/data/train2.csv

DATASET INFORMATION
Shape: (25946, 2)
Columns: ['Text', 'Class']

First 5 rows:


,Text,Class
0,இல்லை புரோ மலேசியாவில் டிரெண்ட் ஆயிடுச்சு அதில...,0
1,ரொம்ப சந்தோஷம் இதை பாக்க ஏன்டா தமிழ் பைத்தியங்...,0
2,அவளை மாறானவராக ஆகிறாள் மருத்துவ மனையில் சேர்த்...,1
3,நீ கட்டின புடவையோட வா நான் உன்னை பாத்துக்கறேன்...,0
4,இனி அடுத்தது சூர்யா தேவி நாலு பேர் கூட்டிட்டு ...,1



Data types:
Text     object
Class     int64
dtype: object

Missing values:
Text     0
Class    0
dtype: int64

Class distribution:
Class
0    13361
1    12585
Name: count, dtype: int64

Class proportions:
Class
0    0.514954
1    0.485046
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# 02. DUPLICATE AND CONFLICTING-LABEL AUDIT
# ============================================================

print("=" * 70)
print("DUPLICATE / CONFLICTING LABEL AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Exact duplicate rows: Text + Class
# ------------------------------------------------------------

duplicate_rows = df.duplicated(subset=["Text", "Class"]).sum()

print("Exact duplicate rows (Text + Class):", duplicate_rows)


# ------------------------------------------------------------
# 2. Duplicate TEXTS irrespective of class
# ------------------------------------------------------------

text_duplicate_mask = df.duplicated(subset=["Text"], keep=False)

duplicate_text_rows = text_duplicate_mask.sum()
unique_texts = df["Text"].nunique()

print("Rows belonging to duplicated texts:", duplicate_text_rows)
print("Unique texts:", unique_texts)
print("Total texts:", len(df))


# ------------------------------------------------------------
# 3. Texts with conflicting labels
# ------------------------------------------------------------

label_counts_per_text = (
    df.groupby("Text")["Class"]
      .nunique()
)

conflicting_texts = label_counts_per_text[
    label_counts_per_text > 1
]

print("Texts with conflicting labels:", len(conflicting_texts))


# ------------------------------------------------------------
# 4. Display conflicting texts
# ------------------------------------------------------------

if len(conflicting_texts) > 0:

    print("\n" + "=" * 70)
    print("CONFLICTING TEXTS")
    print("=" * 70)

    conflict_df = (
        df[df["Text"].isin(conflicting_texts.index)]
        .sort_values("Text")
        [["Text", "Class"]]
    )

    display(conflict_df)

else:
    print("\nNo conflicting duplicate texts found.")


# ------------------------------------------------------------
# 5. Duplicate-text summary
# ------------------------------------------------------------

duplicate_text_groups = (
    df[df["Text"].duplicated(keep=False)]
    .groupby("Text")
    .agg(
        occurrences=("Text", "size"),
        labels=("Class", lambda x: sorted(x.unique().tolist()))
    )
    .sort_values("occurrences", ascending=False)
)

print("\n" + "=" * 70)
print("TOP DUPLICATED TEXTS")
print("=" * 70)

display(duplicate_text_groups.head(20))


# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

print("Total rows                    :", len(df))
print("Unique texts                  :", unique_texts)
print("Exact duplicate rows         :", duplicate_rows)
print("Rows in duplicated text groups:", duplicate_text_rows)
print("Conflicting duplicate texts  :", len(conflicting_texts))

print("=" * 70)

DUPLICATE / CONFLICTING LABEL AUDIT
Exact duplicate rows (Text + Class): 57
Rows belonging to duplicated texts: 110
Unique texts: 25888
Total texts: 25946
Texts with conflicting labels: 1

CONFLICTING TEXTS


,Text,Class
2580,ரெண்டு பேரையும் செருப்புலே அடிச்சு தொரத்தரதை த...,0
16025,ரெண்டு பேரையும் செருப்புலே அடிச்சு தொரத்தரதை த...,1



TOP DUPLICATED TEXTS


,occurrences,labels
Text,,
வேற லெவல்,4,[0]
செம்ம,4,[0]
கண்டா வர சொல்லுங்க,3,[0]
ஜாரா நீங்க உங்களின் மேக் அப் காட்ட முடியுமா,3,[0]
அந்த பொண்ணுக்கு பேய் கூட பயமா இருக்கு மேடம்,2,[1]
அப்பாவின் நடத்தையைப் பார்த்தாலே தெரியும் அவர் எவ்வளவு மோசமானவர் என்று அவர் தனது மகளை கைவிட்டு இப்போது நீதிபதி போல் நடிக்கிறார்,2,[1]
அவளாக இருந்தாலும் பரவாயில்லை அவளின் செயல்கள் காதலா,2,[1]
அவளுக்கு சிகிச்சை தேவை பைத்தியம்,2,[1]
அவள் ஆன்லைனில் பணம் சம்பாதித்தாள் ஆனால் என்னால் ஒரு ரூபாய் கூட சம்பாதிக்க முடியவில்லை,2,[1]



AUDIT SUMMARY
Total rows                    : 25946
Unique texts                  : 25888
Exact duplicate rows         : 57
Rows in duplicated text groups: 110
Conflicting duplicate texts  : 1


In [ ]:
# ============================================================
# 03. INSPECT CONFLICTING DUPLICATE
# ============================================================

conflict_texts = (
    df.groupby("Text")["Class"]
      .nunique()
      .loc[lambda x: x > 1]
      .index
)

print("=" * 70)
print("CONFLICTING DUPLICATE TEXT")
print("=" * 70)

conflict_rows = (
    df[df["Text"].isin(conflict_texts)]
    .sort_values("Text")
)

display(conflict_rows)

print("\nNumber of conflicting rows:", len(conflict_rows))
print("Number of conflicting texts:", len(conflict_texts))

CONFLICTING DUPLICATE TEXT


,Text,Class
2580,ரெண்டு பேரையும் செருப்புலே அடிச்சு தொரத்தரதை த...,0
16025,ரெண்டு பேரையும் செருப்புலே அடிச்சு தொரத்தரதை த...,1



Number of conflicting rows: 2
Number of conflicting texts: 1


In [ ]:
from pathlib import Path
import pandas as pd

print("=" * 70)
print("SEARCHING FOR OFFICIAL TEST DATA")
print("=" * 70)

base = Path("/content/drive/MyDrive/DravidianLangTech_2026")

candidates = []

for p in base.rglob("*.csv"):
    name = p.name.lower()

    if "test" in name:
        candidates.append(p)

print("Possible test files:")

for p in candidates:
    print(" -", p)

print("\n" + "=" * 70)

SEARCHING FOR OFFICIAL TEST DATA
Possible test files:
 - /content/drive/MyDrive/DravidianLangTech_2026/data/TestV2 - testV2.csv



In [ ]:
import pandas as pd
import os

TEST_PATH = "/content/drive/MyDrive/DravidianLangTech_2026/data/TestV2 - testV2.csv"

print("=" * 70)
print("OFFICIAL TEST DATASET AUDIT")
print("=" * 70)

test_df = pd.read_csv(TEST_PATH)

print("\nPath:")
print(TEST_PATH)

print("\nShape:")
print(test_df.shape)

print("\nColumns:")
print(test_df.columns.tolist())

print("\nData types:")
print(test_df.dtypes)

print("\nMissing values:")
print(test_df.isnull().sum())

print("\nFirst 10 rows:")
display(test_df.head(10))

print("\n" + "=" * 70)
print("TEST LABEL CHECK")
print("=" * 70)

if "Class" in test_df.columns:
    print("✓ TEST LABELS ARE AVAILABLE")
    print("\nClass distribution:")
    print(test_df["Class"].value_counts())

    print("\nClass proportions:")
    print(test_df["Class"].value_counts(normalize=True))

else:
    print("⚠ TEST LABEL COLUMN NOT FOUND")
    print("Available columns:", test_df.columns.tolist())

print("\n" + "=" * 70)

OFFICIAL TEST DATASET AUDIT

Path:
/content/drive/MyDrive/DravidianLangTech_2026/data/TestV2 - testV2.csv

Shape:
(913, 1)

Columns:
['Text']

Data types:
Text    object
dtype: object

Missing values:
Text    0
dtype: int64

First 10 rows:


,Text
0,லக்ஷ்மி அம்மா நீங்க புலம்புங்க அவளுக அவளுகபாட்...
1,"இன்னும் கைது பண்ணல... அனைத்து பெற்றோர்களும், க..."
2,"அப்பா,அம்மா, அந்த இன்டர்வியூ பண்ற வக்கிரம்புடி..."
3,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க கார...
4,எல்லாமே script thaan. ஷகீலா உங்க scriptum அரும...
5,எங்க ரூம்லயும் ஒரு கார்த்தி இருக்கான்..... அவன...
6,பச்ச தேவடியாலுக்கும் ....விசத்தேவடியாலுக்கும் ...
7,ஆளானப்பட்ட லட்சுமி மேடத்தயே தனியா பேச வச்சிட்ட...
8,து து து நீ எல்லாம் ஒரு பொண்ணு. சனியனே எப்படி ...
9,நாட்டுக்கு ரொம்ப முக்கியம் இது.... மக்கள கெடு...



TEST LABEL CHECK
⚠ TEST LABEL COLUMN NOT FOUND
Available columns: ['Text']



In [ ]:
# ==============================================================
# STEP 1B-0 — LOAD OFFICIAL TRAINING DATA
# ==============================================================

import pandas as pd
import numpy as np

TRAIN_PATH = "/content/drive/MyDrive/DravidianLangTech_2026/data/train2.csv"

train_df = pd.read_csv(TRAIN_PATH)

print("=" * 70)
print("OFFICIAL TRAINING DATA LOADED")
print("=" * 70)

print("Path:", TRAIN_PATH)
print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

print("\nClass distribution:")
print(train_df["Class"].value_counts())

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nFirst 5 rows:")
display(train_df.head())

OFFICIAL TRAINING DATA LOADED
Path: /content/drive/MyDrive/DravidianLangTech_2026/data/train2.csv
Shape: (25946, 2)
Columns: ['Text', 'Class']

Class distribution:
Class
0    13361
1    12585
Name: count, dtype: int64

Missing values:
Text     0
Class    0
dtype: int64

First 5 rows:


,Text,Class
0,இல்லை புரோ மலேசியாவில் டிரெண்ட் ஆயிடுச்சு அதில...,0
1,ரொம்ப சந்தோஷம் இதை பாக்க ஏன்டா தமிழ் பைத்தியங்...,0
2,அவளை மாறானவராக ஆகிறாள் மருத்துவ மனையில் சேர்த்...,1
3,நீ கட்டின புடவையோட வா நான் உன்னை பாத்துக்கறேன்...,0
4,இனி அடுத்தது சூர்யா தேவி நாலு பேர் கூட்டிட்டு ...,1


In [ ]:
# ==============================================================
# STEP 1B — CLEAN DUPLICATES AND CONFLICTING LABELS
# ==============================================================

print("=" * 70)
print("DUPLICATE / CONFLICT CLEANING")
print("=" * 70)

df = train_df.copy()

# --------------------------------------------------------------
# 1. Find texts with conflicting labels
# --------------------------------------------------------------

label_counts = (
    df.groupby("Text")["Class"]
      .nunique()
)

conflicting_texts = label_counts[
    label_counts > 1
].index.tolist()

print("\nConflicting texts:", len(conflicting_texts))

if conflicting_texts:
    print("\nConflicting rows:")
    display(
        df[df["Text"].isin(conflicting_texts)]
        [["Text", "Class"]]
    )

# --------------------------------------------------------------
# 2. Remove conflicting texts
# --------------------------------------------------------------

df_clean = df[
    ~df["Text"].isin(conflicting_texts)
].copy()

print("\nAfter removing conflicting texts:")
print("Shape:", df_clean.shape)

# --------------------------------------------------------------
# 3. Remove exact duplicate texts
# --------------------------------------------------------------

before_dedup = len(df_clean)

df_clean = (
    df_clean
    .drop_duplicates(
        subset=["Text"],
        keep="first"
    )
    .reset_index(drop=True)
)

duplicates_removed = before_dedup - len(df_clean)

print("\nDuplicate rows removed:", duplicates_removed)

# --------------------------------------------------------------
# 4. Verify uniqueness
# --------------------------------------------------------------

print("\nFinal clean dataset:")
print("Rows:", len(df_clean))
print("Unique texts:", df_clean["Text"].nunique())
print(
    "Duplicate texts remaining:",
    df_clean["Text"].duplicated().sum()
)

# --------------------------------------------------------------
# 5. Verify conflicting labels are gone
# --------------------------------------------------------------

remaining_conflicts = (
    df_clean.groupby("Text")["Class"]
            .nunique()
            .gt(1)
            .sum()
)

print(
    "Conflicting texts remaining:",
    remaining_conflicts
)

# --------------------------------------------------------------
# 6. Class distribution
# --------------------------------------------------------------

print("\nFinal class distribution:")
print(df_clean["Class"].value_counts())

print("\nFinal class proportions:")
print(
    df_clean["Class"]
    .value_counts(normalize=True)
)

print("\n" + "=" * 70)
print("CLEANING COMPLETED")
print("=" * 70)

DUPLICATE / CONFLICT CLEANING

Conflicting texts: 1

Conflicting rows:


,Text,Class
2580,ரெண்டு பேரையும் செருப்புலே அடிச்சு தொரத்தரதை த...,0
16025,ரெண்டு பேரையும் செருப்புலே அடிச்சு தொரத்தரதை த...,1



After removing conflicting texts:
Shape: (25944, 2)

Duplicate rows removed: 57

Final clean dataset:
Rows: 25887
Unique texts: 25887
Duplicate texts remaining: 0
Conflicting texts remaining: 0

Final class distribution:
Class
0    13330
1    12557
Name: count, dtype: int64

Final class proportions:
Class
0    0.51493
1    0.48507
Name: proportion, dtype: float64

CLEANING COMPLETED


In [ ]:
# ==============================================================
# STEP 1C — FIXED STRATIFIED TRAIN / VALIDATION SPLIT
# ==============================================================

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

SEED = 42

print("=" * 70)
print("CREATING FIXED STRATIFIED SPLIT")
print("=" * 70)

# --------------------------------------------------------------
# Split
# --------------------------------------------------------------

train_split, val_split = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=SEED,
    stratify=df_clean["Class"]
)

# Reset indices
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

# --------------------------------------------------------------
# Display sizes
# --------------------------------------------------------------

print("\nDataset:")
print("Total:", len(df_clean))

print("\nTraining:")
print("Samples:", len(train_split))

print("\nValidation:")
print("Samples:", len(val_split))

# --------------------------------------------------------------
# Class distributions
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

print("\nTraining:")
print(train_split["Class"].value_counts())
print(train_split["Class"].value_counts(normalize=True))

print("\nValidation:")
print(val_split["Class"].value_counts())
print(val_split["Class"].value_counts(normalize=True))

# --------------------------------------------------------------
# Check for text leakage
# --------------------------------------------------------------

train_texts = set(train_split["Text"])
val_texts = set(val_split["Text"])

overlap = train_texts.intersection(val_texts)

print("\n" + "=" * 70)
print("LEAKAGE CHECK")
print("=" * 70)

print("Train unique texts:", len(train_texts))
print("Validation unique texts:", len(val_texts))
print("Text overlap:", len(overlap))

assert len(overlap) == 0, "ERROR: Text leakage detected!"

print("\n✓ No text overlap between train and validation.")

# --------------------------------------------------------------
# Save fixed splits
# --------------------------------------------------------------

TRAIN_SPLIT_PATH = (
    "/content/drive/MyDrive/"
    "DravidianLangTech_2026/results/"
    "fixed_train_split_seed42.csv"
)

VAL_SPLIT_PATH = (
    "/content/drive/MyDrive/"
    "DravidianLangTech_2026/results/"
    "fixed_validation_split_seed42.csv"
)

train_split.to_csv(
    TRAIN_SPLIT_PATH,
    index=False
)

val_split.to_csv(
    VAL_SPLIT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("SPLITS SAVED")
print("=" * 70)

print("Train:", TRAIN_SPLIT_PATH)
print("Validation:", VAL_SPLIT_PATH)

print("\n✓ Fixed split created and saved.")
print("✓ Seed:", SEED)

CREATING FIXED STRATIFIED SPLIT

Dataset:
Total: 25887

Training:
Samples: 20709

Validation:
Samples: 5178

CLASS DISTRIBUTION

Training:
Class
0    10664
1    10045
Name: count, dtype: int64
Class
0    0.514945
1    0.485055
Name: proportion, dtype: float64

Validation:
Class
0    2666
1    2512
Name: count, dtype: int64
Class
0    0.514871
1    0.485129
Name: proportion, dtype: float64

LEAKAGE CHECK
Train unique texts: 20709
Validation unique texts: 5178
Text overlap: 0

✓ No text overlap between train and validation.

SPLITS SAVED
Train: /content/drive/MyDrive/DravidianLangTech_2026/results/fixed_train_split_seed42.csv
Validation: /content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42.csv

✓ Fixed split created and saved.
✓ Seed: 42


In [ ]:
import pandas as pd
import numpy as np
import os

TRAIN_PATH = "/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_train_split_seed42.csv"
VAL_PATH   = "/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("=" * 70)
print("FIXED DATA SPLIT")
print("=" * 70)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTrain columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())

print("\nTrain class distribution:")
print(train_df["Class"].value_counts())

print("\nValidation class distribution:")
print(val_df["Class"].value_counts())

overlap = set(train_df["Text"]) & set(val_df["Text"])

print("\nText overlap:", len(overlap))

assert len(overlap) == 0

print("\n✓ Fixed split verified.")

FIXED DATA SPLIT
Train shape: (20709, 2)
Validation shape: (5178, 2)

Train columns: ['Text', 'Class']
Validation columns: ['Text', 'Class']

Train class distribution:
Class
0    10664
1    10045
Name: count, dtype: int64

Validation class distribution:
Class
0    2666
1    2512
Name: count, dtype: int64

Text overlap: 0

✓ Fixed split verified.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("scikit-learn imported successfully.")

scikit-learn imported successfully.


In [ ]:
def evaluate_model(name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )[2]

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print(f"Accuracy        : {accuracy:.4f}")
    print(f"Macro Precision : {precision:.4f}")
    print(f"Macro Recall    : {recall:.4f}")
    print(f"Macro F1        : {f1:.4f}")
    print(f"Weighted F1     : {weighted_f1:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["Non-Abusive", "Abusive"],
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(y_true, y_pred)

    print("Confusion Matrix:")
    print(cm)

    return {
        "Model": name,
        "Accuracy": accuracy,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1,
        "Weighted F1": weighted_f1,
        "Confusion Matrix": cm
    }

In [ ]:
print("=" * 70)
print("WORD TF-IDF")
print("=" * 70)

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

X_train_word = word_vectorizer.fit_transform(train_df["Text"].astype(str))
X_val_word = word_vectorizer.transform(val_df["Text"].astype(str))

y_train = train_df["Class"].values
y_val = val_df["Class"].values

print("Training matrix :", X_train_word.shape)
print("Validation matrix:", X_val_word.shape)
print("Vocabulary size :", len(word_vectorizer.vocabulary_))

WORD TF-IDF
Training matrix : (20709, 17844)
Validation matrix: (5178, 17844)
Vocabulary size : 17844


In [ ]:
print("=" * 70)
print("WORD TF-IDF + LOGISTIC REGRESSION")
print("=" * 70)

lr_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    class_weight=None,
    random_state=42
)

lr_model.fit(X_train_word, y_train)

lr_pred = lr_model.predict(X_val_word)

lr_result = evaluate_model(
    "Word TF-IDF + Logistic Regression",
    y_val,
    lr_pred
)

WORD TF-IDF + LOGISTIC REGRESSION

Word TF-IDF + Logistic Regression
Accuracy        : 0.7518
Macro Precision : 0.7521
Macro Recall    : 0.7510
Macro F1        : 0.7512
Weighted F1     : 0.7516

Classification Report:
              precision    recall  f1-score   support

 Non-Abusive     0.7488    0.7794    0.7638      2666
     Abusive     0.7553    0.7225    0.7386      2512

    accuracy                         0.7518      5178
   macro avg     0.7521    0.7510    0.7512      5178
weighted avg     0.7520    0.7518    0.7516      5178

Confusion Matrix:
[[2078  588]
 [ 697 1815]]


In [ ]:
classical_results = []

classical_results.append(lr_result)

print("\nCurrent results:")
pd.DataFrame([
    {
        k: v for k, v in r.items()
        if k != "Confusion Matrix"
    }
    for r in classical_results
])


Current results:


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Word TF-IDF + Logistic Regression,0.751835,0.752067,0.750988,0.751193,0.751569


In [ ]:
print("=" * 70)
print("WORD TF-IDF + LINEAR SVM")
print("=" * 70)

svm_word = LinearSVC(
    C=1.0,
    class_weight=None,
    random_state=42
)

svm_word.fit(X_train_word, y_train)

svm_word_pred = svm_word.predict(X_val_word)

svm_word_result = evaluate_model(
    "Word TF-IDF + Linear SVM",
    y_val,
    svm_word_pred
)

classical_results.append(svm_word_result)

WORD TF-IDF + LINEAR SVM

Word TF-IDF + Linear SVM
Accuracy        : 0.7513
Macro Precision : 0.7513
Macro Recall    : 0.7506
Macro F1        : 0.7507
Weighted F1     : 0.7511

Classification Report:
              precision    recall  f1-score   support

 Non-Abusive     0.7505    0.7742    0.7622      2666
     Abusive     0.7521    0.7269    0.7393      2512

    accuracy                         0.7513      5178
   macro avg     0.7513    0.7506    0.7507      5178
weighted avg     0.7513    0.7513    0.7511      5178

Confusion Matrix:
[[2064  602]
 [ 686 1826]]


In [ ]:
results_df = pd.DataFrame([
    {
        k: v for k, v in r.items()
        if k != "Confusion Matrix"
    }
    for r in classical_results
])

print(results_df.to_string(index=False))

                            Model  Accuracy  Macro Precision  Macro Recall  Macro F1  Weighted F1
Word TF-IDF + Logistic Regression  0.751835         0.752067      0.750988  0.751193     0.751569
         Word TF-IDF + Linear SVM  0.751255         0.751302      0.750552  0.750729     0.751069


In [ ]:
print("=" * 70)
print("CHARACTER TF-IDF")
print("=" * 70)

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=150000,
    sublinear_tf=True
)

X_train_char = char_vectorizer.fit_transform(
    train_df["Text"].astype(str)
)

X_val_char = char_vectorizer.transform(
    val_df["Text"].astype(str)
)

print("Training matrix  :", X_train_char.shape)
print("Validation matrix:", X_val_char.shape)
print("Vocabulary size  :", len(char_vectorizer.vocabulary_))

CHARACTER TF-IDF
Training matrix  : (20709, 103945)
Validation matrix: (5178, 103945)
Vocabulary size  : 103945


In [ ]:
print("=" * 70)
print("CHARACTER TF-IDF + LINEAR SVM")
print("=" * 70)

svm_char = LinearSVC(
    C=1.0,
    class_weight=None,
    random_state=42
)

svm_char.fit(
    X_train_char,
    y_train
)

svm_char_pred = svm_char.predict(
    X_val_char
)

svm_char_result = evaluate_model(
    "Character TF-IDF + Linear SVM",
    y_val,
    svm_char_pred
)

classical_results.append(svm_char_result)

CHARACTER TF-IDF + LINEAR SVM

Character TF-IDF + Linear SVM
Accuracy        : 0.8463
Macro Precision : 0.8463
Macro Recall    : 0.8459
Macro F1        : 0.8461
Weighted F1     : 0.8462

Classification Report:
              precision    recall  f1-score   support

 Non-Abusive     0.8455    0.8582    0.8518      2666
     Abusive     0.8471    0.8336    0.8403      2512

    accuracy                         0.8463      5178
   macro avg     0.8463    0.8459    0.8461      5178
weighted avg     0.8463    0.8463    0.8462      5178

Confusion Matrix:
[[2288  378]
 [ 418 2094]]


In [ ]:
results_df = pd.DataFrame([
    {
        k: v for k, v in r.items()
        if k != "Confusion Matrix"
    }
    for r in classical_results
])

print("\n" + "=" * 70)
print("CLASSICAL MODEL RESULTS SO FAR")
print("=" * 70)

print(results_df.to_string(index=False))


CLASSICAL MODEL RESULTS SO FAR
                            Model  Accuracy  Macro Precision  Macro Recall  Macro F1  Weighted F1
Word TF-IDF + Logistic Regression  0.751835         0.752067      0.750988  0.751193     0.751569
         Word TF-IDF + Linear SVM  0.751255         0.751302      0.750552  0.750729     0.751069
    Character TF-IDF + Linear SVM  0.846273         0.846308      0.845907  0.846057     0.846228


In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

results_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "classical_baseline_results.csv"
    ),
    index=False
)

print(
    "Saved:",
    os.path.join(
        RESULTS_DIR,
        "classical_baseline_results.csv"
    )
)

Saved: /content/drive/MyDrive/DravidianLangTech_2026/results/classical_baseline_results.csv


In [ ]:
from scipy.sparse import hstack

print("=" * 70)
print("WORD + CHARACTER TF-IDF")
print("=" * 70)

X_train_word_char = hstack([
    X_train_word,
    X_train_char
]).tocsr()

X_val_word_char = hstack([
    X_val_word,
    X_val_char
]).tocsr()

print("Combined training matrix  :", X_train_word_char.shape)
print("Combined validation matrix:", X_val_word_char.shape)

WORD + CHARACTER TF-IDF
Combined training matrix  : (20709, 121789)
Combined validation matrix: (5178, 121789)


In [ ]:
print("=" * 70)
print("WORD + CHARACTER TF-IDF + LINEAR SVM")
print("=" * 70)

svm_word_char = LinearSVC(
    C=1.0,
    class_weight=None,
    random_state=42
)

svm_word_char.fit(
    X_train_word_char,
    y_train
)

svm_word_char_pred = svm_word_char.predict(
    X_val_word_char
)

svm_word_char_result = evaluate_model(
    "Word + Character TF-IDF + Linear SVM",
    y_val,
    svm_word_char_pred
)

classical_results.append(svm_word_char_result)

WORD + CHARACTER TF-IDF + LINEAR SVM

Word + Character TF-IDF + Linear SVM
Accuracy        : 0.8409
Macro Precision : 0.8408
Macro Recall    : 0.8406
Macro F1        : 0.8407
Weighted F1     : 0.8409

Classification Report:
              precision    recall  f1-score   support

 Non-Abusive     0.8437    0.8481    0.8459      2666
     Abusive     0.8379    0.8332    0.8355      2512

    accuracy                         0.8409      5178
   macro avg     0.8408    0.8406    0.8407      5178
weighted avg     0.8408    0.8409    0.8409      5178

Confusion Matrix:
[[2261  405]
 [ 419 2093]]


In [ ]:
results_df = pd.DataFrame([
    {
        k: v for k, v in r.items()
        if k != "Confusion Matrix"
    }
    for r in classical_results
])

print("=" * 70)
print("ALL CLASSICAL BASELINES")
print("=" * 70)

print(results_df.to_string(index=False))

ALL CLASSICAL BASELINES
                               Model  Accuracy  Macro Precision  Macro Recall  Macro F1  Weighted F1
   Word TF-IDF + Logistic Regression  0.751835         0.752067      0.750988  0.751193     0.751569
            Word TF-IDF + Linear SVM  0.751255         0.751302      0.750552  0.750729     0.751069
       Character TF-IDF + Linear SVM  0.846273         0.846308      0.845907  0.846057     0.846228
Word + Character TF-IDF + Linear SVM  0.840865         0.840764      0.840644  0.840698     0.840851


In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

results_df.to_csv(
    f"{RESULTS_DIR}/classical_baseline_results.csv",
    index=False
)

print("Saved classical results.")

Saved classical results.


In [ ]:
print("=" * 70)
print("MURIL ENVIRONMENT CHECK")
print("=" * 70)

import torch
import transformers
import sklearn
import pandas as pd
import numpy as np

print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA         :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

print("Pandas       :", pd.__version__)
print("NumPy        :", np.__version__)
print("Sklearn      :", sklearn.__version__)

MURIL ENVIRONMENT CHECK
PyTorch      : 2.11.0+cu128
Transformers : 5.13.1
CUDA         : True
GPU          : Tesla T4
Pandas       : 2.2.2
NumPy        : 2.0.2
Sklearn      : 1.6.1


In [ ]:
import transformers

print("Transformers version:", transformers.__version__)

from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")
print("MuRIL loaded successfully.")
print("Hidden size:", encoder.config.hidden_size)

Transformers version: 5.13.1


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer loaded successfully.
MuRIL loaded successfully.
Hidden size: 768


In [ ]:
# ============================================================
# LOAD THE FIXED TRAIN / VALIDATION SPLITS
# ============================================================

import pandas as pd
import os

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

TRAIN_SPLIT_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_train_split_seed42.csv"
)

VAL_SPLIT_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_validation_split_seed42.csv"
)

train_df = pd.read_csv(TRAIN_SPLIT_PATH)
validation_df = pd.read_csv(VAL_SPLIT_PATH)

print("=" * 70)
print("FIXED SPLITS LOADED")
print("=" * 70)

print("Train shape      :", train_df.shape)
print("Validation shape :", validation_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nTrain class distribution:")
print(train_df["Class"].value_counts())

print("\nValidation class distribution:")
print(validation_df["Class"].value_counts())

# ------------------------------------------------------------
# VERIFY NO OVERLAP
# ------------------------------------------------------------

train_texts = set(train_df["Text"].astype(str))
val_texts = set(validation_df["Text"].astype(str))

overlap = train_texts.intersection(val_texts)

print("\nText overlap:", len(overlap))

assert len(overlap) == 0

print("✓ Fixed seed-42 split verified.")

FIXED SPLITS LOADED
Train shape      : (20709, 2)
Validation shape : (5178, 2)

Train columns:
['Text', 'Class']

Validation columns:
['Text', 'Class']

Train class distribution:
Class
0    10664
1    10045
Name: count, dtype: int64

Validation class distribution:
Class
0    2666
1    2512
Name: count, dtype: int64

Text overlap: 0
✓ Fixed seed-42 split verified.


In [ ]:
# ============================================================
# MuRIL TEXT-ONLY DATASET
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 128
BATCH_SIZE = 16

# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("=" * 70)
print("MuRIL TOKENIZER")
print("=" * 70)

print("Tokenizer:", MODEL_NAME)
print("Vocab size:", tokenizer.vocab_size)


# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------

class TamilMuRILDataset(Dataset):

    def __init__(
        self,
        df,
        tokenizer,
        max_length=128
    ):

        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = str(
            self.df.iloc[idx]["Text"]
        )

        label = int(
            self.df.iloc[idx]["Class"]
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    label,
                    dtype=torch.long
                )
        }

        if "token_type_ids" in encoding:

            item["token_type_ids"] = (
                encoding["token_type_ids"]
                .squeeze(0)
            )

        return item


# ------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------

train_mu_dataset = TamilMuRILDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)

val_mu_dataset = TamilMuRILDataset(
    validation_df,
    tokenizer,
    MAX_LENGTH
)

print("\nDataset sizes:")
print("Train      :", len(train_mu_dataset))
print("Validation :", len(val_mu_dataset))


# ------------------------------------------------------------
# DATALOADERS
# ------------------------------------------------------------

train_mu_loader = DataLoader(
    train_mu_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_mu_loader = DataLoader(
    val_mu_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("\n✓ DataLoaders created.")

MuRIL TOKENIZER
Tokenizer: google/muril-base-cased
Vocab size: 197258

Dataset sizes:
Train      : 20709
Validation : 5178

✓ DataLoaders created.


In [ ]:
# ============================================================
# MuRIL DATA PIPELINE CHECK
# ============================================================

batch = next(iter(train_mu_loader))

print("=" * 70)
print("MuRIL DATA PIPELINE CHECK")
print("=" * 70)

print("\nBatch keys:")
print(batch.keys())

print("\nInput IDs:")
print(batch["input_ids"].shape)

print("\nAttention mask:")
print(batch["attention_mask"].shape)

print("\nLabels:")
print(batch["labels"].shape)

print("\nFirst label:")
print(batch["labels"][0])

if "token_type_ids" in batch:
    print("\nToken type IDs:")
    print(batch["token_type_ids"].shape)

print("\n✓ Data pipeline working.")

MuRIL DATA PIPELINE CHECK

Batch keys:
dict_keys(['input_ids', 'attention_mask', 'labels', 'token_type_ids'])

Input IDs:
torch.Size([16, 128])

Attention mask:
torch.Size([16, 128])

Labels:
torch.Size([16])

First label:
tensor(0)

Token type IDs:
torch.Size([16, 128])

✓ Data pipeline working.


In [ ]:
# ============================================================
# STEP 4 — MuRIL TEXT-ONLY MODEL
# ============================================================

import torch
import torch.nn as nn
from transformers import BertModel

class MuRILTextClassifier(nn.Module):

    def __init__(
        self,
        model_name="google/muril-base-cased",
        num_labels=2
    ):
        super().__init__()

        self.encoder = BertModel.from_pretrained(
            model_name
        )

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
        labels=None
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # [CLS] representation
        cls_embedding = outputs.last_hidden_state[:, 0, :]

        cls_embedding = self.dropout(
            cls_embedding
        )

        logits = self.classifier(
            cls_embedding
        )

        if labels is not None:

            loss_fn = nn.CrossEntropyLoss()

            loss = loss_fn(
                logits,
                labels
            )

            return {
                "loss": loss,
                "logits": logits
            }

        return {
            "logits": logits
        }


model_text = MuRILTextClassifier(
    model_name=MODEL_NAME
).to(DEVICE)

print("=" * 70)
print("MuRIL TEXT-ONLY MODEL")
print("=" * 70)

print("Device:", DEVICE)
print("Hidden size:", model_text.encoder.config.hidden_size)

total_params = sum(
    p.numel()
    for p in model_text.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model_text.parameters()
    if p.requires_grad
)

print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MuRIL TEXT-ONLY MODEL
Device: cuda
Hidden size: 768
Total parameters     : 237,557,762
Trainable parameters: 237,557,762


In [ ]:
# ============================================================
# STEP 5 — FORWARD PASS SANITY CHECK
# ============================================================

model_text.eval()

batch = next(iter(train_mu_loader))

input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

token_type_ids = None

if "token_type_ids" in batch:
    token_type_ids = batch["token_type_ids"].to(DEVICE)

with torch.no_grad():

    outputs = model_text(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids,
        labels=labels
    )

print("=" * 70)
print("MuRIL TEXT-ONLY FORWARD TEST")
print("=" * 70)

print("Input shape :", input_ids.shape)
print("Logits shape:", outputs["logits"].shape)
print("Loss        :", outputs["loss"].item())

assert outputs["logits"].shape == (
    input_ids.shape[0],
    2
)

print("\n✓ Forward pass successful.")

MuRIL TEXT-ONLY FORWARD TEST
Input shape : torch.Size([16, 128])
Logits shape: torch.Size([16, 2])
Loss        : 0.6913981437683105

✓ Forward pass successful.


In [ ]:
# ============================================================
# STEP 6 — TRAIN MuRIL TEXT-ONLY
# ============================================================

import os
import random
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------

EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "DravidianLangTech_2026/results"
)

MODEL_DIR = os.path.join(
    RESULTS_DIR,
    "muril_text_only_best"
)

os.makedirs(MODEL_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(
    model_text.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_muril(model, dataloader, device):

    model.eval()

    all_predictions = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            token_type_ids = None

            if "token_type_ids" in batch:
                token_type_ids = batch[
                    "token_type_ids"
                ].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )

            loss = outputs["loss"]
            logits = outputs["logits"]

            batch_size = labels.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size

            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_loss = (
        total_loss / total_samples
    )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    weighted_f1 = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="weighted",
            zero_division=0
        )[2]
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "predictions": np.array(all_predictions),
        "labels": np.array(all_labels)
    }


# ------------------------------------------------------------
# TRAINING LOOP
# ------------------------------------------------------------

best_macro_f1 = -1.0
history = []

print("=" * 70)
print("MuRIL TEXT-ONLY TRAINING")
print("=" * 70)

print("Epochs        :", EPOCHS)
print("Learning rate :", LEARNING_RATE)
print("Weight decay  :", WEIGHT_DECAY)
print("Batch size    :", BATCH_SIZE)
print("Max length    :", MAX_LENGTH)
print("Seed          :", SEED)
print("=" * 70)


for epoch in range(EPOCHS):

    model_text.train()

    total_train_loss = 0.0
    total_train_samples = 0

    for batch_idx, batch in enumerate(
        train_mu_loader
    ):

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        labels = batch[
            "labels"
        ].to(DEVICE)

        token_type_ids = None

        if "token_type_ids" in batch:

            token_type_ids = batch[
                "token_type_ids"
            ].to(DEVICE)

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model_text(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )

        loss = outputs["loss"]

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_text.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        batch_size = labels.size(0)

        total_train_loss += (
            loss.item() * batch_size
        )

        total_train_samples += batch_size

        # Progress
        if (
            batch_idx == 0
            or (batch_idx + 1) % 200 == 0
            or (batch_idx + 1) == len(train_mu_loader)
        ):

            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Step {batch_idx + 1}/"
                f"{len(train_mu_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    train_loss = (
        total_train_loss /
        total_train_samples
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_results = evaluate_muril(
        model_text,
        val_mu_loader,
        DEVICE
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_results["loss"],
        "accuracy": val_results["accuracy"],
        "macro_precision": val_results["macro_precision"],
        "macro_recall": val_results["macro_recall"],
        "macro_f1": val_results["macro_f1"],
        "weighted_f1": val_results["weighted_f1"]
    })

    print("\n" + "-" * 70)

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
    )

    print(
        f"Training Loss     : "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation Loss   : "
        f"{val_results['loss']:.4f}"
    )

    print(
        f"Accuracy          : "
        f"{val_results['accuracy']:.4f}"
    )

    print(
        f"Macro Precision   : "
        f"{val_results['macro_precision']:.4f}"
    )

    print(
        f"Macro Recall      : "
        f"{val_results['macro_recall']:.4f}"
    )

    print(
        f"Macro F1          : "
        f"{val_results['macro_f1']:.4f}"
    )

    print(
        f"Weighted F1       : "
        f"{val_results['weighted_f1']:.4f}"
    )

    print("-" * 70)

    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if val_results["macro_f1"] > best_macro_f1:

        best_macro_f1 = (
            val_results["macro_f1"]
        )

        torch.save(
            model_text.state_dict(),
            os.path.join(
                MODEL_DIR,
                "pytorch_model.bin"
            )
        )

        print(
            f"✓ Best model saved "
            f"(Macro-F1 = "
            f"{best_macro_f1:.4f})"
        )

    print()


# ------------------------------------------------------------
# SAVE TRAINING HISTORY
# ------------------------------------------------------------

history_df = pd.DataFrame(history)

history_path = os.path.join(
    RESULTS_DIR,
    "muril_text_only_training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

print("=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(
    "Best validation Macro-F1:",
    f"{best_macro_f1:.4f}"
)

print(
    "History saved:",
    history_path
)

print(
    "Best model saved:",
    MODEL_DIR
)

MuRIL TEXT-ONLY TRAINING
Epochs        : 3
Learning rate : 2e-05
Weight decay  : 0.01
Batch size    : 16
Max length    : 128
Seed          : 42
Epoch 1/3 | Step 1/1295 | Loss: 0.6890
Epoch 1/3 | Step 200/1295 | Loss: 0.5750
Epoch 1/3 | Step 400/1295 | Loss: 0.5391
Epoch 1/3 | Step 600/1295 | Loss: 0.5261
Epoch 1/3 | Step 800/1295 | Loss: 0.5221
Epoch 1/3 | Step 1000/1295 | Loss: 0.4828
Epoch 1/3 | Step 1200/1295 | Loss: 0.3805
Epoch 1/3 | Step 1295/1295 | Loss: 0.5283

----------------------------------------------------------------------
Epoch 1/3
Training Loss     : 0.5576
Validation Loss   : 0.4754
Accuracy          : 0.8146
Macro Precision   : 0.8172
Macro Recall      : 0.8159
Macro F1          : 0.8145
Weighted F1       : 0.8144
----------------------------------------------------------------------
✓ Best model saved (Macro-F1 = 0.8145)

Epoch 2/3 | Step 1/1295 | Loss: 0.4518
Epoch 2/3 | Step 200/1295 | Loss: 0.5434
Epoch 2/3 | Step 400/1295 | Loss: 0.2270
Epoch 2/3 | Step 600/129

In [ ]:
# ============================================================
# STEP 7 — FINAL MuRIL TEXT-ONLY EVALUATION
# ============================================================

import os
import pandas as pd
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# LOAD BEST CHECKPOINT
# ------------------------------------------------------------

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "pytorch_model.bin"
)

model_text.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE
    )
)

model_text.to(DEVICE)
model_text.eval()

print("=" * 70)
print("FINAL MuRIL TEXT-ONLY EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in val_mu_loader:

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        labels = batch[
            "labels"
        ].to(DEVICE)

        token_type_ids = None

        if "token_type_ids" in batch:

            token_type_ids = batch[
                "token_type_ids"
            ].to(DEVICE)

        outputs = model_text(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        logits = outputs["logits"]

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


all_predictions = np.array(
    all_predictions
)

all_labels = np.array(
    all_labels
)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )
)

weighted_f1 = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)[2]

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\nLogits / predictions:")
print("Number of predictions:", len(all_predictions))

print("\nAccuracy        :", f"{accuracy:.4f}")
print("Macro Precision :", f"{macro_precision:.4f}")
print("Macro Recall    :", f"{macro_recall:.4f}")
print("Macro F1        :", f"{macro_f1:.4f}")
print("Weighted F1     :", f"{weighted_f1:.4f}")

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Non-Abusive",
            "Abusive"
        ],
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)

print("\nTrue Negatives :", cm[0, 0])
print("False Positives:", cm[0, 1])
print("False Negatives:", cm[1, 0])
print("True Positives :", cm[1, 1])

# ------------------------------------------------------------
# SAVE PREDICTIONS
# ------------------------------------------------------------

muril_text_predictions = validation_df[
    ["Text", "Class"]
].copy()

muril_text_predictions[
    "Predicted_Class"
] = all_predictions

muril_text_predictions[
    "Correct"
] = (
    muril_text_predictions["Class"].values
    ==
    all_predictions
)

prediction_path = os.path.join(
    RESULTS_DIR,
    "muril_text_only_validation_predictions.csv"
)

muril_text_predictions.to_csv(
    prediction_path,
    index=False
)

# ------------------------------------------------------------
# SAVE METRICS
# ------------------------------------------------------------

muril_text_results = pd.DataFrame([{
    "Model": "MuRIL Text-only",
    "Accuracy": accuracy,
    "Macro Precision": macro_precision,
    "Macro Recall": macro_recall,
    "Macro F1": macro_f1,
    "Weighted F1": weighted_f1
}])

metrics_path = os.path.join(
    RESULTS_DIR,
    "muril_text_only_results.csv"
)

muril_text_results.to_csv(
    metrics_path,
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("Predictions:")
print(prediction_path)

print("\nMetrics:")
print(metrics_path)

FINAL MuRIL TEXT-ONLY EVALUATION

Logits / predictions:
Number of predictions: 5178

Accuracy        : 0.8467
Macro Precision : 0.8512
Macro Recall    : 0.8449
Macro F1        : 0.8456
Weighted F1     : 0.8460

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 Non-Abusive     0.8182    0.9029    0.8584      2666
     Abusive     0.8842    0.7870    0.8328      2512

    accuracy                         0.8467      5178
   macro avg     0.8512    0.8449    0.8456      5178
weighted avg     0.8502    0.8467    0.8460      5178

CONFUSION MATRIX
[[2407  259]
 [ 535 1977]]

True Negatives : 2407
False Positives: 259
False Negatives: 535
True Positives : 1977

FILES SAVED
Predictions:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_text_only_validation_predictions.csv

Metrics:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_text_only_results.csv


In [ ]:
import pandas as pd
import os

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

master_results = pd.DataFrame([
    {
        "Model": "Word TF-IDF + Logistic Regression",
        "Accuracy": 0.751835,
        "Macro Precision": 0.752067,
        "Macro Recall": 0.750988,
        "Macro F1": 0.751193,
        "Weighted F1": 0.751569
    },
    {
        "Model": "Word TF-IDF + Linear SVM",
        "Accuracy": 0.751255,
        "Macro Precision": 0.751302,
        "Macro Recall": 0.750552,
        "Macro F1": 0.750729,
        "Weighted F1": 0.751069
    },
    {
        "Model": "Character TF-IDF + Linear SVM",
        "Accuracy": 0.846273,
        "Macro Precision": 0.846308,
        "Macro Recall": 0.845907,
        "Macro F1": 0.846057,
        "Weighted F1": 0.846228
    },
    {
        "Model": "Word + Character TF-IDF + Linear SVM",
        "Accuracy": 0.840865,
        "Macro Precision": 0.840764,
        "Macro Recall": 0.840644,
        "Macro F1": 0.840698,
        "Weighted F1": 0.840851
    },
    {
        "Model": "MuRIL Text-only",
        "Accuracy": 0.8467,
        "Macro Precision": 0.8512,
        "Macro Recall": 0.8449,
        "Macro F1": 0.8456,
        "Weighted F1": 0.8460
    }
])

master_results.to_csv(
    f"{RESULTS_DIR}/master_model_results.csv",
    index=False
)

print("=" * 70)
print("MASTER MODEL RESULTS")
print("=" * 70)
display(master_results)

print("\nSaved:")
print(f"{RESULTS_DIR}/master_model_results.csv")

MASTER MODEL RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Word TF-IDF + Logistic Regression,0.751835,0.752067,0.750988,0.751193,0.751569
1,Word TF-IDF + Linear SVM,0.751255,0.751302,0.750552,0.750729,0.751069
2,Character TF-IDF + Linear SVM,0.846273,0.846308,0.845907,0.846057,0.846228
3,Word + Character TF-IDF + Linear SVM,0.840865,0.840764,0.840644,0.840698,0.840851
4,MuRIL Text-only,0.846700,0.851200,0.844900,0.845600,0.846000



Saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/master_model_results.csv


In [ ]:
FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

print("=" * 70)
print("FEATURE CHECK")
print("=" * 70)

print("Number of features:", len(FEATURE_COLUMNS))

for i, col in enumerate(FEATURE_COLUMNS):
    print(f"{i:2d}. {col}")

FEATURE CHECK
Number of features: 11
 0. text_length
 1. word_count
 2. tamil_chars
 3. latin_chars
 4. digits
 5. tamil_ratio
 6. latin_ratio
 7. emoji_count
 8. punctuation_count
 9. repeated_char_patterns
10. english_word_count


In [ ]:
train_path = "/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_train_split_seed42.csv"
val_path = "/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42.csv"

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(val_path)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)

missing_train = [c for c in FEATURE_COLUMNS if c not in train_df.columns]
missing_val = [c for c in FEATURE_COLUMNS if c not in validation_df.columns]

print("\nMissing from train:", missing_train)
print("Missing from validation:", missing_val)

Train: (20709, 2)
Validation: (5178, 2)

Missing from train: ['text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']
Missing from validation: ['text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']


In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

print("=" * 70)
print("GENERATING FEATURES FROM FIXED SPLITS")
print("=" * 70)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)

GENERATING FEATURES FROM FIXED SPLITS
Train: (20709, 2)
Validation: (5178, 2)


In [ ]:
def count_tamil_chars(text):
    """
    Count Tamil Unicode characters.
    Tamil Unicode block: U+0B80 - U+0BFF
    """
    return sum(
        1 for ch in text
        if '\u0B80' <= ch <= '\u0BFF'
    )


def count_latin_chars(text):
    """
    Count Latin alphabet characters A-Z / a-z.
    """
    return sum(
        1 for ch in text
        if ('A' <= ch <= 'Z') or ('a' <= ch <= 'z')
    )


def count_digits(text):
    return sum(ch.isdigit() for ch in text)


def count_emoji(text):
    """
    Approximate emoji count using Unicode ranges.
    """
    count = 0

    for ch in text:
        cp = ord(ch)

        if (
            0x1F300 <= cp <= 0x1FAFF or
            0x2600 <= cp <= 0x26FF or
            0x2700 <= cp <= 0x27BF
        ):
            count += 1

    return count


def count_punctuation(text):
    return sum(
        1 for ch in text
        if unicodedata.category(ch).startswith("P")
    )


def count_repeated_char_patterns(text):
    """
    Counts repeated-character patterns such as:
    aaa
    !!!!
    ????
    Tamil repeated characters
    """
    patterns = re.findall(r'(.)\1{2,}', text)
    return len(patterns)


def count_english_words(text):
    """
    Approximate English word count.
    """
    words = re.findall(r'\b[a-zA-Z]+\b', text)

    return len(words)


def extract_features(text):

    if pd.isna(text):
        text = ""

    text = str(text)

    # Basic counts
    text_length = len(text)

    # whitespace-separated tokens
    words = text.split()
    word_count = len(words)

    tamil_chars = count_tamil_chars(text)
    latin_chars = count_latin_chars(text)
    digits = count_digits(text)

    emoji_count = count_emoji(text)

    punctuation_count = count_punctuation(text)

    repeated_char_patterns = count_repeated_char_patterns(text)

    english_word_count = count_english_words(text)

    # Avoid division by zero
    if text_length > 0:
        tamil_ratio = tamil_chars / text_length
        latin_ratio = latin_chars / text_length
    else:
        tamil_ratio = 0.0
        latin_ratio = 0.0

    return {
        "text_length": text_length,
        "word_count": word_count,
        "tamil_chars": tamil_chars,
        "latin_chars": latin_chars,
        "digits": digits,
        "tamil_ratio": tamil_ratio,
        "latin_ratio": latin_ratio,
        "emoji_count": emoji_count,
        "punctuation_count": punctuation_count,
        "repeated_char_patterns": repeated_char_patterns,
        "english_word_count": english_word_count
    }

In [ ]:
print("=" * 70)
print("EXTRACTING TRAIN FEATURES")
print("=" * 70)

train_features = train_df["Text"].apply(extract_features)

train_features_df = pd.DataFrame(
    train_features.tolist(),
    index=train_df.index
)

train_with_features = pd.concat(
    [
        train_df.reset_index(drop=True),
        train_features_df.reset_index(drop=True)
    ],
    axis=1
)

print("Train with features:")
print(train_with_features.shape)

print("\nColumns:")
print(train_with_features.columns.tolist())

EXTRACTING TRAIN FEATURES
Train with features:
(20709, 13)

Columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']


In [ ]:
print("=" * 70)
print("EXTRACTING VALIDATION FEATURES")
print("=" * 70)

val_features = validation_df["Text"].apply(extract_features)

val_features_df = pd.DataFrame(
    val_features.tolist(),
    index=validation_df.index
)

validation_with_features = pd.concat(
    [
        validation_df.reset_index(drop=True),
        val_features_df.reset_index(drop=True)
    ],
    axis=1
)

print("Validation with features:")
print(validation_with_features.shape)

EXTRACTING VALIDATION FEATURES
Validation with features:
(5178, 13)


In [ ]:
print("=" * 70)
print("FEATURE VERIFICATION")
print("=" * 70)

print("\nTrain shape:", train_with_features.shape)
print("Validation shape:", validation_with_features.shape)

print("\nFeature columns:")

for i, col in enumerate(FEATURE_COLUMNS):
    print(
        f"{i:2d}. {col:30s} "
        f"Train dtype={train_with_features[col].dtype}"
    )

print("\nMissing values:")
print(
    train_with_features[FEATURE_COLUMNS]
    .isna()
    .sum()
)

print("\nValidation missing values:")
print(
    validation_with_features[FEATURE_COLUMNS]
    .isna()
    .sum()
)

FEATURE VERIFICATION

Train shape: (20709, 13)
Validation shape: (5178, 13)

Feature columns:
 0. text_length                    Train dtype=int64
 1. word_count                     Train dtype=int64
 2. tamil_chars                    Train dtype=int64
 3. latin_chars                    Train dtype=int64
 4. digits                         Train dtype=int64
 5. tamil_ratio                    Train dtype=float64
 6. latin_ratio                    Train dtype=float64
 7. emoji_count                    Train dtype=int64
 8. punctuation_count              Train dtype=int64
 9. repeated_char_patterns         Train dtype=int64
10. english_word_count             Train dtype=int64

Missing values:
text_length               0
word_count                0
tamil_chars               0
latin_chars               0
digits                    0
tamil_ratio               0
latin_ratio               0
emoji_count               0
punctuation_count         0
repeated_char_patterns    0
english_word_count    

In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

train_feature_path = (
    f"{RESULTS_DIR}/fixed_train_split_seed42_with_features.csv"
)

val_feature_path = (
    f"{RESULTS_DIR}/fixed_validation_split_seed42_with_features.csv"
)

train_with_features.to_csv(
    train_feature_path,
    index=False
)

validation_with_features.to_csv(
    val_feature_path,
    index=False
)

print("=" * 70)
print("FEATURE DATASETS SAVED")
print("=" * 70)

print("Train:")
print(train_feature_path)

print("\nValidation:")
print(val_feature_path)

FEATURE DATASETS SAVED
Train:
/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_train_split_seed42_with_features.csv

Validation:
/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42_with_features.csv


In [ ]:
print("=" * 70)
print("FEATURE STATISTICS")
print("=" * 70)

feature_stats = train_with_features[FEATURE_COLUMNS].describe().T

display(feature_stats)

FEATURE STATISTICS


,count,mean,std,min,25%,50%,75%,max
text_length,20709.0,78.689797,71.339972,1.0,46.000000,61.000000,89.000000,4386.0
word_count,20709.0,10.429668,9.402255,1.0,6.000000,8.000000,12.000000,637.0
tamil_chars,20709.0,69.204452,62.095445,0.0,40.000000,54.000000,79.000000,3744.0
latin_chars,20709.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
digits,20709.0,0.000048,0.006949,0.0,0.000000,0.000000,0.000000,1.0
tamil_ratio,20709.0,0.880604,0.030903,0.0,0.867925,0.882353,0.895522,1.0
latin_ratio,20709.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
emoji_count,20709.0,0.001400,0.049607,0.0,0.000000,0.000000,0.000000,3.0
punctuation_count,20709.0,0.011058,0.136258,0.0,0.000000,0.000000,0.000000,4.0
repeated_char_patterns,20709.0,0.002511,0.051942,0.0,0.000000,0.000000,0.000000,2.0


In [ ]:
def detect_script(text):

    if pd.isna(text):
        return "Other"

    text = str(text)

    tamil = count_tamil_chars(text)
    latin = count_latin_chars(text)

    if tamil > 0 and latin > 0:
        return "Mixed"

    elif tamil > 0:
        return "Tamil"

    elif latin > 0:
        return "Latin/Tanglish"

    else:
        return "Other"


train_with_features["script_type"] = (
    train_with_features["Text"].apply(detect_script)
)

validation_with_features["script_type"] = (
    validation_with_features["Text"].apply(detect_script)
)

print("=" * 70)
print("SCRIPT DISTRIBUTION")
print("=" * 70)

print("\nTRAIN")
print(train_with_features["script_type"].value_counts())

print("\nVALIDATION")
print(validation_with_features["script_type"].value_counts())

SCRIPT DISTRIBUTION

TRAIN
script_type
Tamil    20699
Other       10
Name: count, dtype: int64

VALIDATION
script_type
Tamil    5176
Other       2
Name: count, dtype: int64


In [ ]:
print("=" * 70)
print("CHECKING ACTUAL TEXT CONTENT")
print("=" * 70)

# Show examples containing Latin characters
latin_examples = train_with_features[
    train_with_features["Text"].astype(str).str.contains(
        r"[A-Za-z]",
        regex=True,
        na=False
    )
]

print("\nNumber of train texts containing A-Z/a-z:")
print(len(latin_examples))

if len(latin_examples) > 0:
    display(
        latin_examples[
            ["Text", "latin_chars", "english_word_count", "script_type"]
        ].head(20)
    )

# Show examples containing punctuation
punct_examples = train_with_features[
    train_with_features["Text"].astype(str).str.contains(
        r"[!?.,:;]",
        regex=True,
        na=False
    )
]

print("\nNumber of train texts containing ASCII punctuation:")
print(len(punct_examples))

if len(punct_examples) > 0:
    display(
        punct_examples[
            ["Text", "punctuation_count"]
        ].head(20)
    )

CHECKING ACTUAL TEXT CONTENT

Number of train texts containing A-Z/a-z:
0

Number of train texts containing ASCII punctuation:
0


In [ ]:
print("=" * 70)
print("RAW TEXT INSPECTION")
print("=" * 70)

for i in range(min(20, len(train_df))):
    text = str(train_df.iloc[i]["Text"])

    print(f"\n{i}:")
    print(text)

    latin = re.findall(r"[A-Za-z]", text)
    english = re.findall(r"\b[A-Za-z]+\b", text)

    print("Latin chars:", len(latin))
    print("English words:", english[:10])

RAW TEXT INSPECTION

0:
என்னைப் பிடிக்கப் போகிறாயா நான் எளிதில் வசப்படமாட்டேன்
Latin chars: 0
English words: []

1:
டி போடி எல்லோருக்கும் வலிக்கிறது டி பண்டா
Latin chars: 0
English words: []

2:
நீங்கள் வீடியோவை நிறுத்துங்கள் பின்னர் நீங்கள் அழைப்பைப் பார்க்கலாம்
Latin chars: 0
English words: []

3:
அளுத்தே
Latin chars: 0
English words: []

4:
அருமை அம்மா நீங்க சரியாகவே சொன்னீங்க இந்த பெண் நம்பவே முடியாதவள்
Latin chars: 0
English words: []

5:
தினம் தினம் இப்படி ஒரு சம்பவத்தை சொல்லலாம்
Latin chars: 0
English words: []

6:
இயேசுவே எப்போது நிறுத்தப்படும் தகப்பன் நேராக மலம் செய்கிறார் என்றால் மக்களுக்கு ஏன் பணம் தேவை நான் சிந்திக்க விரும்புகிறேன்
Latin chars: 0
English words: []

7:
இனி மறைக்காதே இனி உன்னை யாரும் நம்ப மாட்டார்கள்
Latin chars: 0
English words: []

8:
எவ முஞ்சிக்கு மேக்கப் ஹேர் ஸ்டைல் வேற கேளட்டு நாயே பெரிய ஹீரோயின் னு நினைப்பு முஞ்ச பாக்க முடியல டா சமி டேய் உங்க மாதிரி ஆளுங்க தான் டா எவள பெரிய ஆளா ஆகுறிங்க எவ்வளவு வெக்கமே இல்லாம பேசுற நிர்வாணமா இன்டர்வ்யூ குடுக்குறங்களாம் 

In [ ]:
print("=" * 70)
print("CHECKING OLD FEATURE DATASET")
print("=" * 70)

old_path = "/content/drive/MyDrive/DravidianLangTech_2026/results/train_with_analysis_features.csv"

old_df = pd.read_csv(old_path)

print("Old dataset shape:", old_df.shape)
print("Old columns:")
print(old_df.columns.tolist())

print("\nOld Latin statistics:")

for col in [
    "latin_chars",
    "latin_ratio",
    "english_word_count"
]:
    if col in old_df.columns:
        print(
            col,
            "non-zero =",
            (old_df[col] > 0).sum(),
            "max =",
            old_df[col].max()
        )

CHECKING OLD FEATURE DATASET
Old dataset shape: (25946, 14)
Old columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'script_type', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']

Old Latin statistics:
latin_chars non-zero = 0 max = 0
latin_ratio non-zero = 0 max = 0.0
english_word_count non-zero = 0 max = 0


In [ ]:
# ============================================================
# STEP 1 — CORRECT FEATURE EXTRACTION
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter

print("=" * 70)
print("CORRECT FEATURE EXTRACTION")
print("=" * 70)


# ------------------------------------------------------------
# Unicode ranges
# ------------------------------------------------------------

# Tamil Unicode block:
# U+0B80 – U+0BFF

TAMIL_RE = re.compile(r'[\u0B80-\u0BFF]')

# Basic Latin alphabet
LATIN_RE = re.compile(r'[A-Za-z]')

# English word
ENGLISH_WORD_RE = re.compile(r'\b[A-Za-z]+\b')

# Digits
DIGIT_RE = re.compile(r'\d')

# Tamil + Latin word tokens
WORD_RE = re.compile(r'\S+')


# ------------------------------------------------------------
# Emoji detection
# ------------------------------------------------------------

def is_emoji(char):
    """
    Approximate Unicode emoji detection.
    """
    code = ord(char)

    return (
        0x1F300 <= code <= 0x1FAFF or
        0x2600 <= code <= 0x26FF or
        0x2700 <= code <= 0x27BF
    )


# ------------------------------------------------------------
# Feature extraction
# ------------------------------------------------------------

def extract_features(text):

    if pd.isna(text):
        text = ""

    text = str(text)

    # --------------------------------------------------------
    # Character counts
    # --------------------------------------------------------

    tamil_chars = len(TAMIL_RE.findall(text))
    latin_chars = len(LATIN_RE.findall(text))
    digits = len(DIGIT_RE.findall(text))

    # --------------------------------------------------------
    # Length
    # --------------------------------------------------------

    text_length = len(text)

    # --------------------------------------------------------
    # Words
    # --------------------------------------------------------

    words = WORD_RE.findall(text)
    word_count = len(words)

    # --------------------------------------------------------
    # English words
    # --------------------------------------------------------

    english_words = ENGLISH_WORD_RE.findall(text)

    english_word_count = len(english_words)

    # --------------------------------------------------------
    # Ratios
    # --------------------------------------------------------

    if text_length > 0:
        tamil_ratio = tamil_chars / text_length
        latin_ratio = latin_chars / text_length
    else:
        tamil_ratio = 0.0
        latin_ratio = 0.0

    # --------------------------------------------------------
    # Emoji
    # --------------------------------------------------------

    emoji_count = sum(is_emoji(ch) for ch in text)

    # --------------------------------------------------------
    # Punctuation
    # --------------------------------------------------------

    punctuation_count = sum(
        1 for ch in text
        if unicodedata.category(ch).startswith("P")
    )

    # --------------------------------------------------------
    # Repeated character patterns
    # --------------------------------------------------------

    # Examples:
    # aaa
    # !!!!
    # .....
    # ட்ட்ட

    repeated_char_patterns = len(
        re.findall(r'(.)\1{2,}', text, flags=re.UNICODE)
    )

    # --------------------------------------------------------
    # Script type
    # --------------------------------------------------------

    has_tamil = tamil_chars > 0
    has_latin = latin_chars > 0

    if has_tamil and has_latin:
        script_type = "Tamil+Latin"

    elif has_tamil:
        script_type = "Tamil"

    elif has_latin:
        script_type = "Latin"

    else:
        script_type = "Other"

    return {
        "text_length": text_length,
        "word_count": word_count,
        "tamil_chars": tamil_chars,
        "latin_chars": latin_chars,
        "digits": digits,
        "tamil_ratio": tamil_ratio,
        "latin_ratio": latin_ratio,
        "script_type": script_type,
        "emoji_count": emoji_count,
        "punctuation_count": punctuation_count,
        "repeated_char_patterns": repeated_char_patterns,
        "english_word_count": english_word_count
    }


print("Feature extractor defined successfully.")

CORRECT FEATURE EXTRACTION
Feature extractor defined successfully.


In [ ]:
# ============================================================
# TEST FEATURE EXTRACTION
# ============================================================

test_texts = [
    "வணக்கம் தமிழ்",
    "hello world",
    "இது ஒரு test comment",
    "Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க",
    "எல்லாமே script thaan",
    "தமிழ்123",
]

test_df = pd.DataFrame({"Text": test_texts})

test_features = pd.DataFrame(
    [extract_features(x) for x in test_df["Text"]]
)

test_output = pd.concat(
    [test_df, test_features],
    axis=1
)

print(test_output.to_string(index=False))

                                      Text  text_length  word_count  tamil_chars  latin_chars  digits  tamil_ratio  latin_ratio script_type  emoji_count  punctuation_count  repeated_char_patterns  english_word_count
                             வணக்கம் தமிழ்           13           2           12            0       0     0.923077     0.000000       Tamil            0                  0                       0                   0
                               hello world           11           2            0           10       0     0.000000     0.909091       Latin            0                  0                       0                   2
                      இது ஒரு test comment           20           4            6           11       0     0.300000     0.550000 Tamil+Latin            0                  0                       0                   2
Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க           42           5           30            8       0     0.714286     0.190476 Tamil+La

In [ ]:
# ============================================================
# REGENERATE FEATURES FOR FIXED SPLITS
# ============================================================

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

TRAIN_PATH = f"{RESULTS_DIR}/fixed_train_split_seed42.csv"
VAL_PATH   = f"{RESULTS_DIR}/fixed_validation_split_seed42.csv"

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VAL_PATH)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)


# Extract features
train_features = pd.DataFrame(
    [extract_features(x) for x in train_df["Text"]]
)

val_features = pd.DataFrame(
    [extract_features(x) for x in validation_df["Text"]]
)


# Add to datasets
train_with_features = pd.concat(
    [train_df.reset_index(drop=True),
     train_features.reset_index(drop=True)],
    axis=1
)

validation_with_features = pd.concat(
    [validation_df.reset_index(drop=True),
     val_features.reset_index(drop=True)],
    axis=1
)


# Save
TRAIN_FEATURE_PATH = (
    f"{RESULTS_DIR}/fixed_train_split_seed42_with_features_v2.csv"
)

VAL_FEATURE_PATH = (
    f"{RESULTS_DIR}/fixed_validation_split_seed42_with_features_v2.csv"
)

train_with_features.to_csv(
    TRAIN_FEATURE_PATH,
    index=False
)

validation_with_features.to_csv(
    VAL_FEATURE_PATH,
    index=False
)


print("=" * 70)
print("FEATURE DATASETS SAVED")
print("=" * 70)

print("Train:")
print(TRAIN_FEATURE_PATH)

print("\nValidation:")
print(VAL_FEATURE_PATH)

Train: (20709, 2)
Validation: (5178, 2)
FEATURE DATASETS SAVED
Train:
/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_train_split_seed42_with_features_v2.csv

Validation:
/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42_with_features_v2.csv


In [ ]:
# ============================================================
# CRITICAL SCRIPT / TANGlish AUDIT
# ============================================================

print("=" * 70)
print("SCRIPT / TANGlish AUDIT")
print("=" * 70)

print("\nTRAIN SCRIPT DISTRIBUTION")
print(train_with_features["script_type"].value_counts())

print("\nVALIDATION SCRIPT DISTRIBUTION")
print(validation_with_features["script_type"].value_counts())


print("\nTRAIN LATIN STATISTICS")

train_latin = train_with_features["latin_chars"]

print(
    "Latin-containing samples:",
    (train_latin > 0).sum(),
    "/",
    len(train_with_features)
)

print(
    "Latin-only samples:",
    (
        train_with_features["script_type"] == "Latin"
    ).sum()
)

print(
    "Tamil+Latin samples:",
    (
        train_with_features["script_type"] == "Tamil+Latin"
    ).sum()
)

print(
    "Tamil-only samples:",
    (
        train_with_features["script_type"] == "Tamil"
    ).sum()
)


print("\nVALIDATION LATIN STATISTICS")

val_latin = validation_with_features["latin_chars"]

print(
    "Latin-containing samples:",
    (val_latin > 0).sum(),
    "/",
    len(validation_with_features)
)

print(
    "Latin-only samples:",
    (
        validation_with_features["script_type"] == "Latin"
    ).sum()
)

print(
    "Tamil+Latin samples:",
    (
        validation_with_features["script_type"] == "Tamil+Latin"
    ).sum()
)

print(
    "Tamil-only samples:",
    (
        validation_with_features["script_type"] == "Tamil"
    ).sum()
)

SCRIPT / TANGlish AUDIT

TRAIN SCRIPT DISTRIBUTION
script_type
Tamil    20699
Other       10
Name: count, dtype: int64

VALIDATION SCRIPT DISTRIBUTION
script_type
Tamil    5176
Other       2
Name: count, dtype: int64

TRAIN LATIN STATISTICS
Latin-containing samples: 0 / 20709
Latin-only samples: 0
Tamil+Latin samples: 0
Tamil-only samples: 20699

VALIDATION LATIN STATISTICS
Latin-containing samples: 0 / 5178
Latin-only samples: 0
Tamil+Latin samples: 0
Tamil-only samples: 5176


In [ ]:
print("=" * 70)
print("FINAL FEATURE SANITY CHECK")
print("=" * 70)

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count",
]

print("\nNumber of features:", len(FEATURE_COLUMNS))

for i, col in enumerate(FEATURE_COLUMNS):
    print(f"{i:2d}. {col}")

print("\nTrain:", train_features_df.shape)
print("Validation:", val_features_df.shape)

print("\nLatin-containing train samples:")
print((train_features_df["latin_chars"] > 0).sum())

print("\nLatin-containing validation samples:")
print((val_features_df["latin_chars"] > 0).sum())

print("\nEnglish-word-containing train samples:")
print((train_features_df["english_word_count"] > 0).sum())

print("\nEnglish-word-containing validation samples:")
print((val_features_df["english_word_count"] > 0).sum())

FINAL FEATURE SANITY CHECK

Number of features: 11
 0. text_length
 1. word_count
 2. tamil_chars
 3. latin_chars
 4. digits
 5. tamil_ratio
 6. latin_ratio
 7. emoji_count
 8. punctuation_count
 9. repeated_char_patterns
10. english_word_count

Train: (20709, 11)
Validation: (5178, 11)

Latin-containing train samples:
0

Latin-containing validation samples:
0

English-word-containing train samples:
0

English-word-containing validation samples:
0


In [ ]:
# ============================================================
# OFFICIAL TEST SCRIPT / TANGLISH AUDIT — FIXED
# ============================================================

import pandas as pd

print("=" * 70)
print("OFFICIAL TEST SCRIPT / TANGLISH AUDIT")
print("=" * 70)

TEST_PATH = "/content/drive/MyDrive/DravidianLangTech_2026/data/TestV2 - testV2.csv"

test_raw = pd.read_csv(TEST_PATH)

print("Test shape:", test_raw.shape)
print("Columns:", test_raw.columns.tolist())

# ------------------------------------------------------------
# Apply the existing feature extractor to EACH text
# ------------------------------------------------------------

test_feature_rows = []

for text in test_raw["Text"].astype(str):
    test_feature_rows.append(extract_features(text))

test_features = pd.DataFrame(test_feature_rows)

# Make sure index matches test_raw
test_features.index = test_raw.index

print("\nFeature extraction completed.")
print("Feature shape:", test_features.shape)

# ------------------------------------------------------------
# SCRIPT DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SCRIPT DISTRIBUTION")
print("=" * 70)

print(test_features["script_type"].value_counts(dropna=False))

# ------------------------------------------------------------
# LATIN / TANGLISH STATISTICS
# ------------------------------------------------------------

latin_count = (test_features["latin_chars"] > 0).sum()

tamil_latin_count = (
    test_features["script_type"] == "Tamil+Latin"
).sum()

latin_only_count = (
    test_features["script_type"] == "Latin"
).sum()

english_word_count = (
    test_features["english_word_count"] > 0
).sum()

print("\nLatin-containing samples :", latin_count)
print("Tamil + Latin samples    :", tamil_latin_count)
print("Latin-only samples       :", latin_only_count)
print("English-word samples     :", english_word_count)

# ------------------------------------------------------------
# PERCENTAGES
# ------------------------------------------------------------

total = len(test_features)

print("\nPercentages:")
print(
    f"Latin-containing : {latin_count}/{total} "
    f"({latin_count / total * 100:.2f}%)"
)

print(
    f"Tamil+Latin      : {tamil_latin_count}/{total} "
    f"({tamil_latin_count / total * 100:.2f}%)"
)

print(
    f"Latin-only       : {latin_only_count}/{total} "
    f"({latin_only_count / total * 100:.2f}%)"
)

print(
    f"English words    : {english_word_count}/{total} "
    f"({english_word_count / total * 100:.2f}%)"
)

# ------------------------------------------------------------
# EXAMPLES OF TAMIL + LATIN / TANGLISH
# ------------------------------------------------------------

mask = test_features["latin_chars"] > 0

examples = pd.DataFrame({
    "Text": test_raw.loc[mask, "Text"].values,
    "Script": test_features.loc[mask, "script_type"].values,
    "Latin_chars": test_features.loc[mask, "latin_chars"].values,
    "English_words": test_features.loc[mask, "english_word_count"].values
})

print("\n" + "=" * 70)
print("LATIN-CONTAINING TEST EXAMPLES")
print("=" * 70)

display(examples.head(20))

OFFICIAL TEST SCRIPT / TANGLISH AUDIT
Test shape: (913, 1)
Columns: ['Text']

Feature extraction completed.
Feature shape: (913, 12)

SCRIPT DISTRIBUTION
script_type
Tamil          705
Tamil+Latin    208
Name: count, dtype: int64

Latin-containing samples : 208
Tamil + Latin samples    : 208
Latin-only samples       : 0
English-word samples     : 200

Percentages:
Latin-containing : 208/913 (22.78%)
Tamil+Latin      : 208/913 (22.78%)
Latin-only       : 0/913 (0.00%)
English words    : 200/913 (21.91%)

LATIN-CONTAINING TEST EXAMPLES


,Text,Script,Latin_chars,English_words
0,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க கார...,Tamil+Latin,14,2
1,எல்லாமே script thaan. ஷகீலா உங்க scriptum அரும...,Tamil+Latin,37,6
2,நாட்டுக்கு ரொம்ப முக்கியம் இது.... மக்கள கெடு...,Tamil+Latin,17,2
3,மூஞ்சி pundai ய பார்த்தா 1000 சுன்னிய பார்த்து...,Tamil+Latin,6,1
4,Coming April ல இருந்து Salary எல்லாம் குறைய போ...,Tamil+Latin,32,5
5,அவ அழரத விட நீங்க போடுர BGMஅ கேட்டா தான்டா பயம...,Tamil+Latin,3,0
6,இரண்டு வல்கர் ponugala சாக்கடை. கட்டி வச்சி அட...,Tamil+Latin,8,1
7,தயவு செய்து இதை பார்த்தாவது பெண்கள் திருந்தவும...,Tamil+Latin,6,1
8,இரண்டு ponnugalum தெரு நாய்ங்க தூ தூ,Tamil+Latin,10,1
9,Tik tok nallathu than but இந்தியா மக்கள் தான் ...,Tamil+Latin,38,6


In [ ]:
# ============================================================
# SAVE OFFICIAL TEST SCRIPT AUDIT
# ============================================================

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

test_audit_df = pd.DataFrame({
    "Metric": [
        "Total test samples",
        "Tamil-only samples",
        "Tamil+Latin samples",
        "Latin-only samples",
        "Latin-containing samples",
        "English-word-containing samples"
    ],
    "Count": [
        len(test_features),
        (test_features["script_type"] == "Tamil").sum(),
        (test_features["script_type"] == "Tamil+Latin").sum(),
        (test_features["script_type"] == "Latin").sum(),
        (test_features["latin_chars"] > 0).sum(),
        (test_features["english_word_count"] > 0).sum()
    ]
})

test_audit_df["Percentage"] = (
    test_audit_df["Count"] / len(test_features) * 100
)

display(test_audit_df)

test_audit_df.to_csv(
    f"{RESULTS_DIR}/official_test_script_audit.csv",
    index=False
)

print("\nSaved:")
print(f"{RESULTS_DIR}/official_test_script_audit.csv")

,Metric,Count,Percentage
0,Total test samples,913,100.000000
1,Tamil-only samples,705,77.217963
2,Tamil+Latin samples,208,22.782037
3,Latin-only samples,0,0.000000
4,Latin-containing samples,208,22.782037
5,English-word-containing samples,200,21.905805



Saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/official_test_script_audit.csv


In [ ]:
# ============================================================
# FINAL FEATURE-FUSION DATA CHECK
# ============================================================

TRAIN_FEATURE_PATH = (
    "/content/drive/MyDrive/DravidianLangTech_2026/results/"
    "fixed_train_split_seed42_with_features.csv"
)

VAL_FEATURE_PATH = (
    "/content/drive/MyDrive/DravidianLangTech_2026/results/"
    "fixed_validation_split_seed42_with_features.csv"
)

fusion_train_df = pd.read_csv(TRAIN_FEATURE_PATH)
fusion_val_df = pd.read_csv(VAL_FEATURE_PATH)

print("=" * 70)
print("FINAL FEATURE-FUSION DATA CHECK")
print("=" * 70)

print("Train:", fusion_train_df.shape)
print("Validation:", fusion_val_df.shape)

print("\nColumns:")
print(fusion_train_df.columns.tolist())

print("\nClass distribution - Train:")
print(fusion_train_df["Class"].value_counts())

print("\nClass distribution - Validation:")
print(fusion_val_df["Class"].value_counts())

print("\nFeature missing values - Train:")
print(
    fusion_train_df[FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

print("\nFeature missing values - Validation:")
print(
    fusion_val_df[FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

print("\nLatin-containing samples - Train:")
print((fusion_train_df["latin_chars"] > 0).sum())

print("\nLatin-containing samples - Validation:")
print((fusion_val_df["latin_chars"] > 0).sum())

print("\n✓ Feature-fusion data ready.")

FINAL FEATURE-FUSION DATA CHECK
Train: (20709, 13)
Validation: (5178, 13)

Columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']

Class distribution - Train:
Class
0    10664
1    10045
Name: count, dtype: int64

Class distribution - Validation:
Class
0    2666
1    2512
Name: count, dtype: int64

Feature missing values - Train:
0

Feature missing values - Validation:
0

Latin-containing samples - Train:
0

Latin-containing samples - Validation:
0

✓ Feature-fusion data ready.


In [ ]:
# ============================================================
# MuRIL + 11 FEATURE FUSION
# STEP 1: FEATURE CONFIGURATION
# ============================================================

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

NUM_FEATURES = len(FEATURE_COLUMNS)

print("=" * 70)
print("FEATURE-FUSION CONFIGURATION")
print("=" * 70)

print("Number of features:", NUM_FEATURES)

for i, col in enumerate(FEATURE_COLUMNS):
    print(f"{i:2d}. {col}")

assert NUM_FEATURES == 11

print("\n✓ Exactly 11 numerical features.")

FEATURE-FUSION CONFIGURATION
Number of features: 11
 0. text_length
 1. word_count
 2. tamil_chars
 3. latin_chars
 4. digits
 5. tamil_ratio
 6. latin_ratio
 7. emoji_count
 8. punctuation_count
 9. repeated_char_patterns
10. english_word_count

✓ Exactly 11 numerical features.


In [ ]:
# ============================================================
# STEP 2: FEATURE STANDARDIZATION
# ============================================================

from sklearn.preprocessing import StandardScaler
import numpy as np
import joblib

scaler = StandardScaler()

X_train_features = scaler.fit_transform(
    fusion_train_df[FEATURE_COLUMNS].values
)

X_val_features = scaler.transform(
    fusion_val_df[FEATURE_COLUMNS].values
)

print("=" * 70)
print("FEATURE STANDARDIZATION")
print("=" * 70)

print("Train feature matrix:", X_train_features.shape)
print("Validation feature matrix:", X_val_features.shape)

print("\nTrain mean:")
print(np.round(X_train_features.mean(axis=0), 4))

print("\nTrain std:")
print(np.round(X_train_features.std(axis=0), 4))

print("\n✓ Scaler fitted ONLY on training data.")

FEATURE STANDARDIZATION
Train feature matrix: (20709, 11)
Validation feature matrix: (5178, 11)

Train mean:
[ 0.  0.  0.  0.  0.  0.  0. -0.  0.  0.  0.]

Train std:
[1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 0.]

✓ Scaler fitted ONLY on training data.


In [ ]:
SCALER_PATH = (
    "/content/drive/MyDrive/DravidianLangTech_2026/results/"
    "muril_feature_scaler.pkl"
)

joblib.dump(scaler, SCALER_PATH)

print("\nScaler saved:")
print(SCALER_PATH)


Scaler saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_scaler.pkl


In [ ]:
# ============================================================
# STEP 3: MuRIL FEATURE-FUSION DATASET
# ============================================================

import torch
from torch.utils.data import Dataset

class TamilMuRILFeatureDataset(Dataset):

    def __init__(
        self,
        df,
        feature_matrix,
        tokenizer,
        max_length=128
    ):
        self.df = df.reset_index(drop=True)
        self.features = torch.tensor(
            feature_matrix,
            dtype=torch.float32
        )
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = str(self.df.loc[idx, "Text"])
        label = int(self.df.loc[idx, "Class"])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "features": self.features[idx],
            "labels": torch.tensor(
                label,
                dtype=torch.long
            )
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = (
                encoding["token_type_ids"].squeeze(0)
            )

        return item


fusion_train_dataset = TamilMuRILFeatureDataset(
    fusion_train_df,
    X_train_features,
    tokenizer,
    max_length=128
)

fusion_val_dataset = TamilMuRILFeatureDataset(
    fusion_val_df,
    X_val_features,
    tokenizer,
    max_length=128
)

print("=" * 70)
print("FEATURE-FUSION DATASETS")
print("=" * 70)

print("Train dataset:", len(fusion_train_dataset))
print("Validation dataset:", len(fusion_val_dataset))

FEATURE-FUSION DATASETS
Train dataset: 20709
Validation dataset: 5178


In [ ]:
# ============================================================
# STEP 4: DATALOADERS
# ============================================================

from torch.utils.data import DataLoader

BATCH_SIZE = 16

fusion_train_loader = DataLoader(
    fusion_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

fusion_val_loader = DataLoader(
    fusion_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("=" * 70)
print("DATALOADERS")
print("=" * 70)

print("Train batches:", len(fusion_train_loader))
print("Validation batches:", len(fusion_val_loader))

DATALOADERS
Train batches: 1295
Validation batches: 324


In [ ]:
# ============================================================
# STEP 5: BATCH CHECK
# ============================================================

batch = next(iter(fusion_train_loader))

print("=" * 70)
print("FEATURE-FUSION BATCH CHECK")
print("=" * 70)

print("Keys:")
print(batch.keys())

print("\nInput IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Features:", batch["features"].shape)
print("Labels:", batch["labels"].shape)

if "token_type_ids" in batch:
    print("Token type IDs:", batch["token_type_ids"].shape)

print("\nFeature dtype:", batch["features"].dtype)
print("Feature example:")
print(batch["features"][0])

print("\nLabel example:")
print(batch["labels"][0])

assert batch["features"].shape[1] == 11

print("\n✓ Batch structure correct.")

FEATURE-FUSION BATCH CHECK
Keys:
dict_keys(['input_ids', 'attention_mask', 'features', 'labels', 'token_type_ids'])

Input IDs: torch.Size([16, 128])
Attention mask: torch.Size([16, 128])
Features: torch.Size([16, 11])
Labels: torch.Size([16])
Token type IDs: torch.Size([16, 128])

Feature dtype: torch.float32
Feature example:
tensor([-0.8507, -0.7902, -0.8568,  0.0000, -0.0069,  0.2681,  0.0000, -0.0282,
        -0.0812, -0.0483,  0.0000])

Label example:
tensor(0)

✓ Batch structure correct.


In [ ]:
# ============================================================
# STEP 6: MuRIL + 11 FEATURE FUSION MODEL
# ============================================================

import torch
import torch.nn as nn
from transformers import AutoModel

MODEL_NAME = "google/muril-base-cased"

class MuRILFeatureFusion(nn.Module):

    def __init__(
        self,
        model_name=MODEL_NAME,
        num_features=11,
        feature_dim=64,
        fusion_dim=256,
        num_labels=2
    ):
        super().__init__()

        # ----------------------------------------------------
        # MuRIL encoder
        # ----------------------------------------------------
        self.encoder = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = self.encoder.config.hidden_size

        # ----------------------------------------------------
        # Numerical feature projection
        # 11 -> 64
        # ----------------------------------------------------
        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        # ----------------------------------------------------
        # Fusion
        # MuRIL 768 + features 64 = 832
        # ----------------------------------------------------
        self.fusion = nn.Sequential(
            nn.Linear(
                hidden_size + feature_dim,
                fusion_dim
            ),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        # ----------------------------------------------------
        # Classification head
        # ----------------------------------------------------
        self.classifier = nn.Linear(
            fusion_dim,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        features,
        token_type_ids=None,
        labels=None
    ):

        # ----------------------------------------------------
        # MuRIL
        # ----------------------------------------------------
        encoder_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }

        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids

        outputs = self.encoder(**encoder_kwargs)

        # CLS representation
        cls_embedding = outputs.last_hidden_state[:, 0, :]

        # ----------------------------------------------------
        # Numerical features
        # ----------------------------------------------------
        feature_embedding = self.feature_projection(
            features
        )

        # ----------------------------------------------------
        # Concatenate
        # ----------------------------------------------------
        combined = torch.cat(
            [cls_embedding, feature_embedding],
            dim=1
        )

        # ----------------------------------------------------
        # Fusion layer
        # ----------------------------------------------------
        fused = self.fusion(combined)

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------
        logits = self.classifier(fused)

        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------
        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }


print("=" * 70)
print("MuRIL FEATURE-FUSION MODEL")
print("=" * 70)

fusion_model = MuRILFeatureFusion(
    model_name=MODEL_NAME,
    num_features=11,
    feature_dim=64,
    fusion_dim=256,
    num_labels=2
)

print(fusion_model)

MuRIL FEATURE-FUSION MODEL


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MuRILFeatureFusion(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elemen

In [ ]:
# ============================================================
# STEP 7: MOVE MODEL TO GPU
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

fusion_model = fusion_model.to(device)

print("=" * 70)
print("DEVICE")
print("=" * 70)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE
Device: cuda
GPU: Tesla T4


In [ ]:
# ============================================================
# STEP 8: PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in fusion_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in fusion_model.parameters()
    if p.requires_grad
)

print("=" * 70)
print("MODEL PARAMETERS")
print("=" * 70)

print(
    f"Total parameters     : {total_params:,}"
)

print(
    f"Trainable parameters : {trainable_params:,}"
)

MODEL PARAMETERS
Total parameters     : 237,771,394
Trainable parameters : 237,771,394


In [ ]:
# ============================================================
# STEP 9: FEATURE-FUSION FORWARD TEST
# ============================================================

fusion_model.eval()

batch = next(iter(fusion_train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
features = batch["features"].to(device)
labels = batch["labels"].to(device)

token_type_ids = None

if "token_type_ids" in batch:
    token_type_ids = batch["token_type_ids"].to(device)

with torch.no_grad():

    outputs = fusion_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        features=features,
        token_type_ids=token_type_ids,
        labels=labels
    )

print("=" * 70)
print("FEATURE-FUSION FORWARD TEST")
print("=" * 70)

print("Input IDs shape :", input_ids.shape)
print("Features shape  :", features.shape)
print("Labels shape    :", labels.shape)

print("\nLogits shape    :", outputs["logits"].shape)
print("Loss            :", outputs["loss"].item())

print("\nLogits:")
print(outputs["logits"][:5])

assert outputs["logits"].shape == (16, 2)

print("\n✓ Forward pass successful.")

FEATURE-FUSION FORWARD TEST
Input IDs shape : torch.Size([16, 128])
Features shape  : torch.Size([16, 11])
Labels shape    : torch.Size([16])

Logits shape    : torch.Size([16, 2])
Loss            : 0.76906818151474

Logits:
tensor([[-0.1223,  0.1632],
        [-0.4045,  0.1749],
        [ 0.0453, -0.0330],
        [ 0.0107, -0.1106],
        [-0.2998,  0.1931]], device='cuda:0')

✓ Forward pass successful.


In [ ]:
# ============================================================
# STEP 10: TRAIN MuRIL + 11 FEATURE FUSION
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
from torch.optim import AdamW
from tqdm.auto import tqdm

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# TRAINING CONFIGURATION
# ------------------------------------------------------------

EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

RESULTS_DIR = (
    "/content/drive/MyDrive/DravidianLangTech_2026/results"
)

CHECKPOINT_DIR = (
    f"{RESULTS_DIR}/muril_feature_fusion_best"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ------------------------------------------------------------
# OPTIMIZER
# ------------------------------------------------------------

optimizer = AdamW(
    fusion_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_fusion_model(model, dataloader, device):

    model.eval()

    all_predictions = []
    all_labels = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            features = batch["features"].to(device)
            labels = batch["labels"].to(device)

            token_type_ids = None

            if "token_type_ids" in batch:
                token_type_ids = batch["token_type_ids"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                features=features,
                token_type_ids=token_type_ids,
                labels=labels
            )

            loss = outputs["loss"]
            logits = outputs["logits"]

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_loss = total_loss / total_samples

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    weighted_f1 = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="weighted",
            zero_division=0
        )[2]
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "predictions": np.array(all_predictions),
        "labels": np.array(all_labels)
    }


# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

history = []

best_macro_f1 = -1.0

print("=" * 70)
print("MuRIL + 11 FEATURE FUSION TRAINING")
print("=" * 70)

print("Model              :", MODEL_NAME)
print("Epochs             :", EPOCHS)
print("Learning rate      :", LEARNING_RATE)
print("Weight decay       :", WEIGHT_DECAY)
print("Batch size         :", BATCH_SIZE)
print("Max length         :", 128)
print("Number of features :", NUM_FEATURES)
print("Seed               :", SEED)
print("Device             :", device)

print("=" * 70)


for epoch in range(EPOCHS):

    fusion_model.train()

    running_loss = 0.0
    total_train_samples = 0

    progress_bar = tqdm(
        fusion_train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for step, batch in enumerate(progress_bar):

        optimizer.zero_grad(set_to_none=True)

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        features = batch["features"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        token_type_ids = None

        if "token_type_ids" in batch:
            token_type_ids = batch["token_type_ids"].to(
                device,
                non_blocking=True
            )

        outputs = fusion_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            features=features,
            token_type_ids=token_type_ids,
            labels=labels
        )

        loss = outputs["loss"]

        loss.backward()

        optimizer.step()

        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        total_train_samples += batch_size

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        running_loss / total_train_samples
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_results = evaluate_fusion_model(
        fusion_model,
        fusion_val_loader,
        device
    )

    history_row = {
        "epoch": epoch + 1,
        "training_loss": train_loss,
        "validation_loss": val_results["loss"],
        "accuracy": val_results["accuracy"],
        "macro_precision": val_results["macro_precision"],
        "macro_recall": val_results["macro_recall"],
        "macro_f1": val_results["macro_f1"],
        "weighted_f1": val_results["weighted_f1"]
    }

    history.append(history_row)

    print("\n" + "-" * 70)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("-" * 70)

    print(
        f"Training Loss     : {train_loss:.4f}"
    )

    print(
        f"Validation Loss   : {val_results['loss']:.4f}"
    )

    print(
        f"Accuracy          : {val_results['accuracy']:.4f}"
    )

    print(
        f"Macro Precision   : "
        f"{val_results['macro_precision']:.4f}"
    )

    print(
        f"Macro Recall      : "
        f"{val_results['macro_recall']:.4f}"
    )

    print(
        f"Macro F1          : "
        f"{val_results['macro_f1']:.4f}"
    )

    print(
        f"Weighted F1       : "
        f"{val_results['weighted_f1']:.4f}"
    )

    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    current_f1 = val_results["macro_f1"]

    if current_f1 > best_macro_f1:

        best_macro_f1 = current_f1

        torch.save(
            {
                "model_state_dict": fusion_model.state_dict(),
                "epoch": epoch + 1,
                "macro_f1": current_f1,
                "config": {
                    "model_name": MODEL_NAME,
                    "num_features": NUM_FEATURES,
                    "feature_columns": FEATURE_COLUMNS,
                    "feature_dim": 64,
                    "fusion_dim": 256,
                    "num_labels": 2,
                    "max_length": 128,
                    "seed": SEED
                }
            },
            f"{CHECKPOINT_DIR}/pytorch_model.bin"
        )

        print(
            f"✓ Best model saved "
            f"(Macro-F1 = {best_macro_f1:.4f})"
        )

    print("-" * 70)


# ------------------------------------------------------------
# SAVE TRAINING HISTORY
# ------------------------------------------------------------

history_df = pd.DataFrame(history)

HISTORY_PATH = (
    f"{RESULTS_DIR}/"
    "muril_feature_fusion_training_history.csv"
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

print("\n" + "=" * 70)
print("FEATURE-FUSION TRAINING COMPLETED")
print("=" * 70)

print(
    f"Best validation Macro-F1: "
    f"{best_macro_f1:.4f}"
)

print("\nHistory saved:")
print(HISTORY_PATH)

print("\nBest model saved:")
print(CHECKPOINT_DIR)

MuRIL + 11 FEATURE FUSION TRAINING
Model              : google/muril-base-cased
Epochs             : 3
Learning rate      : 2e-05
Weight decay       : 0.01
Batch size         : 16
Max length         : 128
Number of features : 11
Seed               : 42
Device             : cuda


Epoch 1/3:   0%|          | 0/1295 [00:00<?, ?it/s]


----------------------------------------------------------------------
Epoch 1/3
----------------------------------------------------------------------
Training Loss     : 0.4933
Validation Loss   : 0.4243
Accuracy          : 0.8366
Macro Precision   : 0.8365
Macro Recall      : 0.8367
Macro F1          : 0.8365
Weighted F1       : 0.8366
✓ Best model saved (Macro-F1 = 0.8365)
----------------------------------------------------------------------


Epoch 2/3:   0%|          | 0/1295 [00:00<?, ?it/s]


----------------------------------------------------------------------
Epoch 2/3
----------------------------------------------------------------------
Training Loss     : 0.3457
Validation Loss   : 0.3590
Accuracy          : 0.8426
Macro Precision   : 0.8442
Macro Recall      : 0.8436
Macro F1          : 0.8426
Weighted F1       : 0.8425
✓ Best model saved (Macro-F1 = 0.8426)
----------------------------------------------------------------------


Epoch 3/3:   0%|          | 0/1295 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ae77b940860>
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ae77b940860>Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
self._shutdown_workers()    
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()
    if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

     if w.is_alive(): 
         ^ ^ ^ ^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/lib/python


----------------------------------------------------------------------
Epoch 3/3
----------------------------------------------------------------------
Training Loss     : 0.2500
Validation Loss   : 0.3922
Accuracy          : 0.8579
Macro Precision   : 0.8585
Macro Recall      : 0.8586
Macro F1          : 0.8579
Weighted F1       : 0.8579
✓ Best model saved (Macro-F1 = 0.8579)
----------------------------------------------------------------------

FEATURE-FUSION TRAINING COMPLETED
Best validation Macro-F1: 0.8579

History saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_training_history.csv

Best model saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best


In [ ]:
# ============================================================
# STEP 1 — ENVIRONMENT + PATHS
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from google.colab import drive

drive.mount("/content/drive")

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"
DATA_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/data"

BEST_MODEL_PATH = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_best"
)

TRAIN_FEATURE_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_train_split_seed42_with_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_validation_split_seed42_with_features.csv"
)

TEST_PATH = os.path.join(
    DATA_DIR,
    "TestV2 - testV2.csv"
)

print("=" * 70)
print("PATH CHECK")
print("=" * 70)

print("Results:", RESULTS_DIR)
print("Best model:", BEST_MODEL_PATH)
print("Train features:", TRAIN_FEATURE_PATH)
print("Validation features:", VAL_FEATURE_PATH)
print("Official test:", TEST_PATH)

print()
print("Best model exists:", os.path.exists(BEST_MODEL_PATH))
print("Train feature file exists:", os.path.exists(TRAIN_FEATURE_PATH))
print("Validation feature file exists:", os.path.exists(VAL_FEATURE_PATH))
print("Test file exists:", os.path.exists(TEST_PATH))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print()
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PATH CHECK
Results: /content/drive/MyDrive/DravidianLangTech_2026/results
Best model: /content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best
Train features: /content/drive/MyDrive/DravidianLangTech_2026/results/fixed_train_split_seed42_with_features.csv
Validation features: /content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42_with_features.csv
Official test: /content/drive/MyDrive/DravidianLangTech_2026/data/TestV2 - testV2.csv

Best model exists: True
Train feature file exists: True
Validation feature file exists: True
Test file exists: True

Device: cuda
GPU: Tesla T4


In [ ]:
# ============================================================
# STEP 2 — IMPORTS + REPRODUCIBILITY
# ============================================================

import re
import unicodedata

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import AutoTokenizer, BertModel

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Seed: 42
PyTorch: 2.11.0+cu128
CUDA: True


In [ ]:
# ============================================================
# STEP 3 — LOAD TRAIN / VALIDATION FEATURE DATA
# ============================================================

train_feature_df = pd.read_csv(TRAIN_FEATURE_PATH)
val_feature_df = pd.read_csv(VAL_FEATURE_PATH)

print("=" * 70)
print("FEATURE DATA")
print("=" * 70)

print("Train shape:", train_feature_df.shape)
print("Validation shape:", val_feature_df.shape)

print()
print("Train columns:")
print(train_feature_df.columns.tolist())

print()
print("Validation columns:")
print(val_feature_df.columns.tolist())

FEATURE DATA
Train shape: (20709, 13)
Validation shape: (5178, 13)

Train columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']

Validation columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']


In [ ]:
# ============================================================
# STEP 4 — VERIFY 11 FEATURES
# ============================================================

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

print("=" * 70)
print("FEATURE VERIFICATION")
print("=" * 70)

missing_train = [
    c for c in FEATURE_COLUMNS
    if c not in train_feature_df.columns
]

missing_val = [
    c for c in FEATURE_COLUMNS
    if c not in val_feature_df.columns
]

print("Missing from train:", missing_train)
print("Missing from validation:", missing_val)

assert len(missing_train) == 0
assert len(missing_val) == 0

print()
print("Number of features:", len(FEATURE_COLUMNS))

print()
print("Train missing values:")
print(train_feature_df[FEATURE_COLUMNS].isnull().sum().sum())

print()
print("Validation missing values:")
print(val_feature_df[FEATURE_COLUMNS].isnull().sum().sum())

assert train_feature_df[FEATURE_COLUMNS].isnull().sum().sum() == 0
assert val_feature_df[FEATURE_COLUMNS].isnull().sum().sum() == 0

print()
print("✓ All 11 features verified")

FEATURE VERIFICATION
Missing from train: []
Missing from validation: []

Number of features: 11

Train missing values:
0

Validation missing values:
0

✓ All 11 features verified


In [ ]:
# ============================================================
# STEP 5 — FEATURE SANITY CHECK
# ============================================================

print("=" * 70)
print("FEATURE SANITY CHECK")
print("=" * 70)

print("\nTrain Latin-containing:",
      (train_feature_df["latin_chars"] > 0).sum())

print("Validation Latin-containing:",
      (val_feature_df["latin_chars"] > 0).sum())

print("\nTrain English-word samples:",
      (train_feature_df["english_word_count"] > 0).sum())

print("Validation English-word samples:",
      (val_feature_df["english_word_count"] > 0).sum())

print("\nTrain feature means:")
print(train_feature_df[FEATURE_COLUMNS].mean())

print("\nValidation feature means:")
print(val_feature_df[FEATURE_COLUMNS].mean())

FEATURE SANITY CHECK

Train Latin-containing: 0
Validation Latin-containing: 0

Train English-word samples: 0
Validation English-word samples: 0

Train feature means:
text_length               78.689797
word_count                10.429668
tamil_chars               69.204452
latin_chars                0.000000
digits                     0.000048
tamil_ratio                0.880604
latin_ratio                0.000000
emoji_count                0.001400
punctuation_count          0.011058
repeated_char_patterns     0.002511
english_word_count         0.000000
dtype: float64

Validation feature means:
text_length               79.196215
word_count                10.499421
tamil_chars               69.663770
latin_chars                0.000000
digits                     0.000193
tamil_ratio                0.880750
latin_ratio                0.000000
emoji_count                0.002897
punctuation_count          0.008884
repeated_char_patterns     0.003283
english_word_count         0.000000

In [ ]:
# ============================================================
# STEP 6 — LOAD MuRIL TOKENIZER
# ============================================================

MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded:", MODEL_NAME)
print("Max length:", MAX_LENGTH)

Tokenizer loaded: google/muril-base-cased
Max length: 128


In [ ]:
# ============================================================
# STEP 7 — FEATURE-FUSION DATASET
# ============================================================

class TamilAbuseFusionDataset(Dataset):

    def __init__(
        self,
        df,
        tokenizer,
        feature_columns,
        max_length=128
    ):

        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.feature_columns = feature_columns
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        text = str(row["Text"])
        label = int(row["Class"])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        features = row[
            self.feature_columns
        ].values.astype(np.float32)

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "features": torch.tensor(
                features,
                dtype=torch.float32
            ),
            "labels": torch.tensor(
                label,
                dtype=torch.long
            )
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = (
                encoding["token_type_ids"].squeeze(0)
            )

        return item


val_fusion_dataset = TamilAbuseFusionDataset(
    val_feature_df,
    tokenizer,
    FEATURE_COLUMNS,
    MAX_LENGTH
)

val_fusion_dataloader = DataLoader(
    val_fusion_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Validation dataset:", len(val_fusion_dataset))
print("Validation batches:", len(val_fusion_dataloader))

Validation dataset: 5178
Validation batches: 324


In [ ]:
# ============================================================
# STEP 8 — MuRIL FEATURE-FUSION MODEL
# ============================================================

import torch.nn as nn


class MuRILFeatureFusion(nn.Module):

    def __init__(
        self,
        model_name="google/muril-base-cased",
        num_features=11,
        num_labels=2
    ):

        super().__init__()

        self.encoder = BertModel.from_pretrained(
            model_name
        )

        hidden_size = self.encoder.config.hidden_size

        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.fusion = nn.Sequential(
            nn.Linear(hidden_size + 64, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Linear(
            256,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        features,
        token_type_ids=None
    ):

        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        pooled_output = encoder_outputs.pooler_output

        feature_output = self.feature_projection(
            features
        )

        combined = torch.cat(
            [pooled_output, feature_output],
            dim=1
        )

        fused = self.fusion(combined)

        logits = self.classifier(fused)

        return logits

In [ ]:
# ============================================================
# STEP 1 — INSPECT THE SAVED MuRIL FEATURE-FUSION CHECKPOINT
# ============================================================

import os

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"
BEST_MODEL_DIR = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_best"
)

print("=" * 70)
print("CHECKING SAVED FEATURE-FUSION MODEL")
print("=" * 70)

print("Directory:", BEST_MODEL_DIR)
print("Exists:", os.path.exists(BEST_MODEL_DIR))
print("Is directory:", os.path.isdir(BEST_MODEL_DIR))

if os.path.isdir(BEST_MODEL_DIR):
    print("\nFiles inside checkpoint directory:")

    for root, dirs, files in os.walk(BEST_MODEL_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            print(" -", full_path)

print("\n✓ Check completed")

CHECKING SAVED FEATURE-FUSION MODEL
Directory: /content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best
Exists: True
Is directory: True

Files inside checkpoint directory:
 - /content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best/pytorch_model.bin

✓ Check completed


In [ ]:
# ============================================================
# STEP 2 — LOAD BEST MuRIL + FEATURE-FUSION MODEL
#           AND EVALUATE ON VALIDATION SET
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_best",
    "pytorch_model.bin"
)

VAL_FEATURE_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_validation_split_seed42_with_features.csv"
)

print("=" * 70)
print("MuRIL + 11 FEATURE FUSION — FINAL VALIDATION")
print("=" * 70)

print("Checkpoint:")
print(CHECKPOINT_PATH)

print("\nValidation data:")
print(VAL_FEATURE_PATH)

print("\nCheckpoint exists:", os.path.exists(CHECKPOINT_PATH))
print("Validation file exists:", os.path.exists(VAL_FEATURE_PATH))


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)


# ------------------------------------------------------------
# LOAD VALIDATION DATA
# ------------------------------------------------------------

validation_df = pd.read_csv(
    VAL_FEATURE_PATH
)

print("\nValidation shape:", validation_df.shape)

print("\nValidation columns:")
print(validation_df.columns.tolist())


# ------------------------------------------------------------
# FEATURE COLUMNS
# ------------------------------------------------------------

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

print("\nNumber of features:", len(FEATURE_COLUMNS))

missing_features = [
    c for c in FEATURE_COLUMNS
    if c not in validation_df.columns
]

if missing_features:
    raise ValueError(
        f"Missing feature columns: {missing_features}"
    )

print("✓ All 11 features found")


# ------------------------------------------------------------
# CHECK LABEL
# ------------------------------------------------------------

if "Class" not in validation_df.columns:
    raise ValueError(
        "Validation dataset does not contain 'Class' column."
    )

print("\nClass distribution:")
print(validation_df["Class"].value_counts())


# ------------------------------------------------------------
# CHECK FEATURE VALUES
# ------------------------------------------------------------

print("\nFeature missing values:")

print(
    validation_df[FEATURE_COLUMNS]
    .isna()
    .sum()
)

if validation_df[FEATURE_COLUMNS].isna().sum().sum() > 0:
    raise ValueError(
        "Feature dataset contains missing values."
    )

print("✓ Feature values valid")


# ============================================================
# CREATE DATASET
# ============================================================

# IMPORTANT:
# This assumes your TamilAbuseFeatureDataset class already exists.
#
# If your class has a different name, tell me the class name
# and I will adjust this cell.

val_fusion_dataset = TamilAbuseFeatureDataset(
    validation_df,
    tokenizer,
    max_length=128,
    feature_columns=FEATURE_COLUMNS
)

print("\n✓ Validation feature dataset created")

print("Dataset size:", len(val_fusion_dataset))


# ============================================================
# DATALOADER
# ============================================================

val_fusion_dataloader = DataLoader(
    val_fusion_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("✓ Validation DataLoader created")

# ------------------------------------------------------------
# BATCH CHECK
# ------------------------------------------------------------

sample_batch = next(iter(val_fusion_dataloader))

print("\nBatch keys:")
print(sample_batch.keys())

print("\nInput IDs:")
print(sample_batch["input_ids"].shape)

print("\nAttention mask:")
print(sample_batch["attention_mask"].shape)

print("\nFeatures:")
print(sample_batch["features"].shape)

print("\nLabels:")
print(sample_batch["labels"].shape)


# ============================================================
# CREATE MODEL ARCHITECTURE
# ============================================================

print("\n" + "=" * 70)
print("CREATING MODEL")
print("=" * 70)

model = MuRILFeatureFusion(
    model_name="google/muril-base-cased",
    num_features=11,
    num_labels=2
)

print("✓ Model architecture created")


# ============================================================
# LOAD CHECKPOINT
# ============================================================

print("\nLoading checkpoint...")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("Checkpoint type:", type(checkpoint))

# ------------------------------------------------------------
# Determine checkpoint format
# ------------------------------------------------------------

if isinstance(checkpoint, dict):

    print("\nCheckpoint keys:")
    print(list(checkpoint.keys())[:20])

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint["state_dict"]

    else:

        # Assume the dictionary itself is state_dict
        state_dict = checkpoint

else:

    state_dict = checkpoint


# ------------------------------------------------------------
# Load weights
# ------------------------------------------------------------

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False
)

print("\nMissing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

print("\n✓ Checkpoint loaded")


# ============================================================
# MOVE MODEL TO GPU
# ============================================================

model.to(device)
model.eval()

print("✓ Model moved to:", device)


# ============================================================
# FINAL VALIDATION PREDICTION
# ============================================================

all_predictions = []
all_labels = []
all_logits = []

print("\n" + "=" * 70)
print("RUNNING VALIDATION PREDICTION")
print("=" * 70)

with torch.no_grad():

    for batch_idx, batch in enumerate(
        val_fusion_dataloader
    ):

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        features = batch["features"].to(device)

        labels = batch["labels"].to(device)

        token_type_ids = batch.get(
            "token_type_ids"
        )

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            features=features,
            token_type_ids=token_type_ids
        )

        # ----------------------------------------------------
        # Handle different output formats
        # ----------------------------------------------------

        if isinstance(outputs, dict):

            if "logits" not in outputs:
                raise ValueError(
                    f"Model dictionary output has keys: "
                    f"{outputs.keys()}"
                )

            logits = outputs["logits"]

        elif hasattr(outputs, "logits"):

            logits = outputs.logits

        else:

            logits = outputs

        if not torch.is_tensor(logits):

            raise TypeError(
                f"Expected logits Tensor, got "
                f"{type(logits)}"
            )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_logits.extend(
            logits.cpu().numpy()
        )

        if (batch_idx + 1) % 50 == 0:

            print(
                f"Processed "
                f"{batch_idx + 1}/"
                f"{len(val_fusion_dataloader)} batches"
            )


# ============================================================
# NUMPY
# ============================================================

all_predictions = np.asarray(
    all_predictions
)

all_labels = np.asarray(
    all_labels
)

all_logits = np.asarray(
    all_logits
)

print("\nPrediction count:", len(all_predictions))

print("Label count:", len(all_labels))

print("Logit shape:", all_logits.shape)


# ============================================================
# METRICS
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_precision, macro_recall, macro_f1, _ = \
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

weighted_f1 = \
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )[2]


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL MuRIL + 11 FEATURE FUSION RESULTS")
print("=" * 70)

print(
    f"Accuracy        : {accuracy:.4f}"
)

print(
    f"Macro Precision : {macro_precision:.4f}"
)

print(
    f"Macro Recall    : {macro_recall:.4f}"
)

print(
    f"Macro F1        : {macro_f1:.4f}"
)

print(
    f"Weighted F1     : {weighted_f1:.4f}"
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

report_text = classification_report(
    all_labels,
    all_predictions,
    target_names=[
        "Non-Abusive",
        "Abusive"
    ],
    digits=4,
    zero_division=0
)

print(report_text)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)


# ============================================================
# SAVE PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "True_Label":
        all_labels,

    "Predicted_Label":
        all_predictions,

    "Logit_NonAbusive":
        all_logits[:, 0],

    "Logit_Abusive":
        all_logits[:, 1]
})

prediction_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_validation_predictions.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)


# ============================================================
# SAVE METRICS
# ============================================================

metrics_df = pd.DataFrame([{

    "Model":
        "MuRIL + 11 Feature Fusion",

    "Accuracy":
        accuracy,

    "Macro Precision":
        macro_precision,

    "Macro Recall":
        macro_recall,

    "Macro F1":
        macro_f1,

    "Weighted F1":
        weighted_f1,

    "TN":
        tn,

    "FP":
        fp,

    "FN":
        fn,

    "TP":
        tp
}])

metrics_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_results.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)


# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(
    "Predictions:",
    prediction_path
)

print(
    "Metrics:",
    metrics_path
)

print("\n✓ FEATURE-FUSION VALIDATION EVALUATION COMPLETED")

MuRIL + 11 FEATURE FUSION — FINAL VALIDATION
Checkpoint:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best/pytorch_model.bin

Validation data:
/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42_with_features.csv

Checkpoint exists: True
Validation file exists: True

Device: cuda

Validation shape: (5178, 13)

Validation columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']

Number of features: 11
✓ All 11 features found

Class distribution:
Class
0    2666
1    2512
Name: count, dtype: int64

Feature missing values:
text_length               0
word_count                0
tamil_chars               0
latin_chars               0
digits                    0
tamil_ratio               0
latin_ratio               0
emoji_count               0
punctuation_count         0
repeate

NameError: name 'TamilAbuseFeatureDataset' is not defined

In [ ]:
# ============================================================
# RESTORE MuRIL TOKENIZER + DATASET + FEATURE-FUSION MODEL
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

print("=" * 70)
print("RESTORING MuRIL FEATURE-FUSION COMPONENTS")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 128
NUM_FEATURES = 11
NUM_LABELS = 2

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Features:", NUM_FEATURES)

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("✓ Tokenizer loaded")

RESTORING MuRIL FEATURE-FUSION COMPONENTS
Device: cuda
Model: google/muril-base-cased
Max length: 128
Features: 11
✓ Tokenizer loaded


In [ ]:
# ============================================================
# DATASET CLASS
# ============================================================

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]


class TamilAbuseFeatureDataset(Dataset):

    def __init__(
        self,
        df,
        tokenizer,
        max_length=128,
        feature_columns=None
    ):

        self.df = df.reset_index(drop=True).copy()

        self.tokenizer = tokenizer

        self.max_length = max_length

        self.feature_columns = (
            feature_columns
            if feature_columns is not None
            else FEATURE_COLUMNS
        )

        # ----------------------------------------------------
        # Text
        # ----------------------------------------------------

        self.texts = (
            self.df["Text"]
            .fillna("")
            .astype(str)
            .tolist()
        )

        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        if "Class" in self.df.columns:

            self.labels = (
                self.df["Class"]
                .astype(int)
                .tolist()
            )

        else:

            self.labels = None

        # ----------------------------------------------------
        # Features
        # ----------------------------------------------------

        self.features = (
            self.df[self.feature_columns]
            .astype(np.float32)
            .values
        )

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "features":
                torch.tensor(
                    self.features[idx],
                    dtype=torch.float32
                )
        }

        if "token_type_ids" in encoding:

            item["token_type_ids"] = (
                encoding["token_type_ids"]
                .squeeze(0)
            )

        if self.labels is not None:

            item["labels"] = torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )

        return item


print("✓ TamilAbuseFeatureDataset restored")

✓ TamilAbuseFeatureDataset restored


In [ ]:
# ============================================================
# MuRIL + FEATURE FUSION MODEL
# ============================================================

class MuRILFeatureFusion(nn.Module):

    def __init__(
        self,
        model_name="google/muril-base-cased",
        num_features=11,
        num_labels=2
    ):

        super().__init__()

        # ----------------------------------------------------
        # MuRIL encoder
        # ----------------------------------------------------

        self.muril = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = (
            self.muril.config.hidden_size
        )

        # ----------------------------------------------------
        # Feature projection
        # ----------------------------------------------------

        self.feature_projection = nn.Sequential(

            nn.Linear(
                num_features,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.1)
        )

        # ----------------------------------------------------
        # Fusion classifier
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                hidden_size + 64,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                256,
                num_labels
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        features,
        token_type_ids=None
    ):

        # ----------------------------------------------------
        # MuRIL
        # ----------------------------------------------------

        if token_type_ids is not None:

            outputs = self.muril(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )

        else:

            outputs = self.muril(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # ----------------------------------------------------
        # CLS representation
        # ----------------------------------------------------

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        # ----------------------------------------------------
        # Feature representation
        # ----------------------------------------------------

        feature_embedding = (
            self.feature_projection(features)
        )

        # ----------------------------------------------------
        # Fusion
        # ----------------------------------------------------

        fused = torch.cat(
            [
                cls_embedding,
                feature_embedding
            ],
            dim=1
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        logits = self.classifier(fused)

        return logits


print("✓ MuRILFeatureFusion class restored")

✓ MuRILFeatureFusion class restored


In [ ]:
# ============================================================
# ARCHITECTURE CHECK
# ============================================================

test_model = MuRILFeatureFusion(
    model_name=MODEL_NAME,
    num_features=11,
    num_labels=2
)

test_model.to(device)

sample_dataset = TamilAbuseFeatureDataset(
    validation_df,
    tokenizer,
    max_length=128,
    feature_columns=FEATURE_COLUMNS
)

sample_loader = DataLoader(
    sample_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

batch = next(iter(sample_loader))

print("=" * 70)
print("ARCHITECTURE CHECK")
print("=" * 70)

print("Input IDs:", batch["input_ids"].shape)
print("Features:", batch["features"].shape)
print("Labels:", batch["labels"].shape)

with torch.no_grad():

    test_logits = test_model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        features=batch["features"].to(device),
        token_type_ids=(
            batch["token_type_ids"].to(device)
            if "token_type_ids" in batch
            else None
        )
    )

print("Logits:", test_logits.shape)

assert test_logits.shape == (2, 2)

print("\n✓ Architecture works")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ARCHITECTURE CHECK
Input IDs: torch.Size([2, 128])
Features: torch.Size([2, 11])
Labels: torch.Size([2])
Logits: torch.Size([2, 2])

✓ Architecture works


In [ ]:
# ============================================================
# LOAD THE TRAINED CHECKPOINT
# ============================================================

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/DravidianLangTech_2026/"
    "results/muril_feature_fusion_best/pytorch_model.bin"
)

print("=" * 70)
print("LOADING TRAINED CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):

    print("Checkpoint keys:")
    print(list(checkpoint.keys())[:20])

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint["state_dict"]

    else:

        state_dict = checkpoint

else:

    state_dict = checkpoint


missing_keys, unexpected_keys = test_model.load_state_dict(
    state_dict,
    strict=False
)

print("\nMissing keys:")
print(missing_keys)

print("\nUnexpected keys:")
print(unexpected_keys)

if len(missing_keys) > 0:
    print("\n⚠ WARNING: Missing model parameters.")
else:
    print("\n✓ All model parameters loaded")

test_model.to(device)
test_model.eval()

model = test_model

print("\n✓ TRAINED MuRIL FEATURE-FUSION MODEL LOADED")

LOADING TRAINED CHECKPOINT
Checkpoint type: <class 'dict'>
Checkpoint keys:
['model_state_dict', 'epoch', 'macro_f1', 'config']

Missing keys:
['muril.embeddings.word_embeddings.weight', 'muril.embeddings.position_embeddings.weight', 'muril.embeddings.token_type_embeddings.weight', 'muril.embeddings.LayerNorm.weight', 'muril.embeddings.LayerNorm.bias', 'muril.encoder.layer.0.attention.self.query.weight', 'muril.encoder.layer.0.attention.self.query.bias', 'muril.encoder.layer.0.attention.self.key.weight', 'muril.encoder.layer.0.attention.self.key.bias', 'muril.encoder.layer.0.attention.self.value.weight', 'muril.encoder.layer.0.attention.self.value.bias', 'muril.encoder.layer.0.attention.output.dense.weight', 'muril.encoder.layer.0.attention.output.dense.bias', 'muril.encoder.layer.0.attention.output.LayerNorm.weight', 'muril.encoder.layer.0.attention.output.LayerNorm.bias', 'muril.encoder.layer.0.intermediate.dense.weight', 'muril.encoder.layer.0.intermediate.dense.bias', 'muril.encode

In [ ]:
# ============================================================
# FINAL VALIDATION EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

val_fusion_dataset = TamilAbuseFeatureDataset(
    validation_df,
    tokenizer,
    max_length=128,
    feature_columns=FEATURE_COLUMNS
)

val_fusion_dataloader = DataLoader(
    val_fusion_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

all_predictions = []
all_labels = []
all_logits = []

model.eval()

with torch.no_grad():

    for batch in val_fusion_dataloader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        features = batch["features"].to(device)

        labels = batch["labels"].to(device)

        token_type_ids = batch.get("token_type_ids")

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            features=features,
            token_type_ids=token_type_ids
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_logits.extend(
            logits.cpu().numpy()
        )


all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)
all_logits = np.array(all_logits)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_precision, macro_recall, macro_f1, _ = \
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

weighted_f1 = \
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )[2]

# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print("=" * 70)
print("FINAL MuRIL + 11 FEATURE FUSION VALIDATION RESULTS")
print("=" * 70)

print(f"Samples         : {len(all_labels)}")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Macro Precision : {macro_precision:.4f}")
print(f"Macro Recall    : {macro_recall:.4f}")
print(f"Macro F1        : {macro_f1:.4f}")
print(f"Weighted F1     : {weighted_f1:.4f}")

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Non-Abusive",
            "Abusive"
        ],
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

prediction_df = pd.DataFrame({

    "True_Label": all_labels,

    "Predicted_Label": all_predictions,

    "Logit_NonAbusive": all_logits[:, 0],

    "Logit_Abusive": all_logits[:, 1]
})

prediction_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_validation_predictions.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)

metrics_df = pd.DataFrame([{

    "Model": "MuRIL + 11 Feature Fusion",

    "Accuracy": accuracy,

    "Macro Precision": macro_precision,

    "Macro Recall": macro_recall,

    "Macro F1": macro_f1,

    "Weighted F1": weighted_f1,

    "TN": tn,

    "FP": fp,

    "FN": fn,

    "TP": tp
}])

metrics_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_results.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

print("\n✓ Predictions saved:")
print(prediction_path)

print("\n✓ Metrics saved:")
print(metrics_path)

FINAL MuRIL + 11 FEATURE FUSION VALIDATION RESULTS
Samples         : 5178
Accuracy        : 0.5104
Macro Precision : 0.3890
Macro Recall    : 0.4958
Macro F1        : 0.3425
Weighted F1     : 0.3524

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 Non-Abusive     0.5128    0.9865    0.6748      2666
     Abusive     0.2653    0.0052    0.0102      2512

    accuracy                         0.5104      5178
   macro avg     0.3890    0.4958    0.3425      5178
weighted avg     0.3927    0.5104    0.3524      5178

CONFUSION MATRIX
[[2630   36]
 [2499   13]]

TN: 2630
FP: 36
FN: 2499
TP: 13

✓ Predictions saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_validation_predictions.csv

✓ Metrics saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_results.csv


In [ ]:
# ============================================================
# FINAL CORRECT RELOADED EVALUATION
# MuRIL + 11 Feature Fusion
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch import nn

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# 2. PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/DravidianLangTech_2026"

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results"
)

TRAIN_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_train_split_seed42_with_features.csv"
)

VAL_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_validation_split_seed42_with_features.csv"
)

CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_best",
    "pytorch_model.bin"
)

print("=" * 70)
print("MuRIL + 11 FEATURE FUSION — RELOADED EVALUATION")
print("=" * 70)

print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

assert os.path.exists(TRAIN_PATH), "Training feature file not found"
assert os.path.exists(VAL_PATH), "Validation feature file not found"
assert os.path.exists(CHECKPOINT_PATH), "Checkpoint not found"

# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print()
print("Device:", device)

# ============================================================
# 4. LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VAL_PATH)

print()
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

# ============================================================
# 5. EXACT 11 FEATURE ORDER
# ============================================================

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

print()
print("=" * 70)
print("FEATURE VERIFICATION")
print("=" * 70)

print("Number of features:", len(FEATURE_COLUMNS))

for col in FEATURE_COLUMNS:
    assert col in train_df.columns, f"Missing train feature: {col}"
    assert col in validation_df.columns, f"Missing validation feature: {col}"

print("✓ All 11 features found")

# ============================================================
# 6. TRAINING STATISTICS
# IMPORTANT:
# Calculate ONLY from training data.
# ============================================================

train_features = train_df[FEATURE_COLUMNS].astype(np.float32)

feature_means = train_features.mean()
feature_stds = train_features.std()

# Prevent division by zero
feature_stds = feature_stds.replace(0, 1.0)

print()
print("Training feature means:")
print(feature_means)

print()
print("Training feature std:")
print(feature_stds)

# ============================================================
# 7. STANDARDIZE FEATURES
# ============================================================

val_features_raw = validation_df[
    FEATURE_COLUMNS
].astype(np.float32)

val_features_scaled = (
    val_features_raw - feature_means
) / feature_stds

val_features_scaled = val_features_scaled.astype(
    np.float32
)

print()
print("=" * 70)
print("SCALED FEATURE CHECK")
print("=" * 70)

print(
    pd.DataFrame(
        val_features_scaled,
        columns=FEATURE_COLUMNS
    ).head()
)

print()
print("Validation scaled means:")
print(val_features_scaled.mean())

print()
print("Validation scaled std:")
print(val_features_scaled.std())

# ============================================================
# 8. TOKENIZER
# ============================================================

MODEL_NAME = "google/muril-base-cased"

MAX_LENGTH = 128
BATCH_SIZE = 16

print()
print("=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("✓ Tokenizer loaded")

# ============================================================
# 9. DATASET
# ============================================================

class TamilFeatureFusionDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        features,
        max_length=128
    ):

        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.features = np.asarray(
            features,
            dtype=np.float32
        )
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = str(
            self.df.iloc[idx]["Text"]
        )

        label = int(
            self.df.iloc[idx]["Class"]
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "features": torch.tensor(
                self.features[idx],
                dtype=torch.float32
            ),
            "labels": torch.tensor(
                label,
                dtype=torch.long
            )
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = (
                encoding["token_type_ids"].squeeze(0)
            )

        return item


val_dataset = TamilFeatureFusionDataset(
    validation_df,
    tokenizer,
    val_features_scaled,
    MAX_LENGTH
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print()
print("✓ Validation dataset created")
print("Validation samples:", len(val_dataset))

# ============================================================
# 10. MODEL ARCHITECTURE
# IMPORTANT:
# This must match the architecture used during training.
# ============================================================

class MuRILFeatureFusion(nn.Module):

    def __init__(
        self,
        model_name,
        num_features=11,
        num_labels=2
    ):

        super().__init__()

        self.muril = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = self.muril.config.hidden_size

        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        features,
        token_type_ids=None
    ):

        muril_outputs = self.muril(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # CLS representation
        text_embedding = muril_outputs.last_hidden_state[:, 0, :]

        feature_embedding = self.feature_projection(
            features
        )

        fused = torch.cat(
            [
                text_embedding,
                feature_embedding
            ],
            dim=1
        )

        logits = self.classifier(fused)

        return logits


print()
print("=" * 70)
print("CREATING MODEL")
print("=" * 70)

model = MuRILFeatureFusion(
    model_name=MODEL_NAME,
    num_features=11,
    num_labels=2
)

print("✓ Model architecture created")

# ============================================================
# 11. LOAD CHECKPOINT
# ============================================================

print()
print("=" * 70)
print("LOADING CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("Checkpoint type:", type(checkpoint))

# Handle different saving formats
if isinstance(checkpoint, dict):

    print("Checkpoint keys:")
    print(list(checkpoint.keys())[:20])

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint[
            "state_dict"
        ]

    else:

        # HuggingFace-style directory checkpoint
        # saved directly as state_dict
        state_dict = checkpoint

else:

    state_dict = checkpoint

# Remove possible "module." prefix
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        key = key[len("module."):]

    clean_state_dict[key] = value

# ============================================================
# LOAD WITH STRICT CHECK
# ============================================================

missing_keys, unexpected_keys = model.load_state_dict(
    clean_state_dict,
    strict=False
)

print()
print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

if len(missing_keys) > 0:
    print()
    print("⚠ WARNING: Missing model parameters!")

if len(unexpected_keys) > 0:
    print()
    print("⚠ WARNING: Unexpected checkpoint parameters!")

if len(missing_keys) == 0:
    print()
    print("✓ Checkpoint parameters loaded")

model.to(device)
model.eval()

# ============================================================
# 12. CHECK ONE BATCH
# ============================================================

print()
print("=" * 70)
print("BATCH CHECK")
print("=" * 70)

batch = next(iter(val_loader))

print("Input IDs:", batch["input_ids"].shape)
print("Attention:", batch["attention_mask"].shape)
print("Features:", batch["features"].shape)
print("Labels:", batch["labels"].shape)

print()
print("First feature vector:")
print(batch["features"][0])

# ============================================================
# 13. FORWARD CHECK
# ============================================================

with torch.no_grad():

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    features = batch["features"].to(device)
    labels = batch["labels"].to(device)

    token_type_ids = batch.get(
        "token_type_ids"
    )

    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(device)

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        features=features,
        token_type_ids=token_type_ids
    )

print()
print("Logits shape:", logits.shape)
print("First logits:")
print(logits[:5])

# ============================================================
# 14. FINAL PREDICTION
# ============================================================

all_predictions = []
all_labels = []
all_logits = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        features = batch["features"].to(device)
        labels = batch["labels"].to(device)

        token_type_ids = batch.get(
            "token_type_ids"
        )

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            features=features,
            token_type_ids=token_type_ids
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_logits.extend(
            logits.cpu().numpy()
        )

all_predictions = np.asarray(
    all_predictions
)

all_labels = np.asarray(
    all_labels
)

all_logits = np.asarray(
    all_logits
)

# ============================================================
# 15. METRICS
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )
)

weighted_f1 = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )[2]
)

# ============================================================
# 16. RESULTS
# ============================================================

print()
print("=" * 70)
print("FINAL RELOADED FEATURE-FUSION RESULTS")
print("=" * 70)

print("Samples         :", len(all_predictions))
print(f"Accuracy        : {accuracy:.4f}")
print(f"Macro Precision : {macro_precision:.4f}")
print(f"Macro Recall    : {macro_recall:.4f}")
print(f"Macro F1        : {macro_f1:.4f}")
print(f"Weighted F1     : {weighted_f1:.4f}")

# ============================================================
# 17. CLASSIFICATION REPORT
# ============================================================

print()
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Non-Abusive",
            "Abusive"
        ],
        digits=4,
        zero_division=0
    )
)

# ============================================================
# 18. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print()
print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)

if cm.shape == (2, 2):

    tn, fp, fn, tp = cm.ravel()

    print()
    print("TN:", tn)
    print("FP:", fp)
    print("FN:", fn)
    print("TP:", tp)

# ============================================================
# 19. SAVE PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "True_Label":
        all_labels,

    "Predicted_Label":
        all_predictions,

    "Logit_NonAbusive":
        all_logits[:, 0],

    "Logit_Abusive":
        all_logits[:, 1]
})

prediction_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_validation_predictions_correct.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)

# ============================================================
# 20. SAVE METRICS
# ============================================================

metrics_df = pd.DataFrame([{

    "Model":
        "MuRIL + 11 Feature Fusion",

    "Accuracy":
        accuracy,

    "Macro Precision":
        macro_precision,

    "Macro Recall":
        macro_recall,

    "Macro F1":
        macro_f1,

    "Weighted F1":
        weighted_f1,

    "TN":
        tn if cm.shape == (2, 2) else np.nan,

    "FP":
        fp if cm.shape == (2, 2) else np.nan,

    "FN":
        fn if cm.shape == (2, 2) else np.nan,

    "TP":
        tp if cm.shape == (2, 2) else np.nan

}])

metrics_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_results_correct.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

# ============================================================
# 21. DONE
# ============================================================

print()
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print(prediction_path)
print(metrics_path)

print()
print("✓ RELOADED EVALUATION COMPLETED")

MuRIL + 11 FEATURE FUSION — RELOADED EVALUATION
Train: /content/drive/MyDrive/DravidianLangTech_2026/results/fixed_train_split_seed42_with_features.csv
Validation: /content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42_with_features.csv
Checkpoint: /content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best/pytorch_model.bin

Device: cuda

Train shape: (20709, 13)
Validation shape: (5178, 13)

FEATURE VERIFICATION
Number of features: 11
✓ All 11 features found

Training feature means:
text_length               78.689796
word_count                10.429668
tamil_chars               69.204453
latin_chars                0.000000
digits                     0.000048
tamil_ratio                0.880604
latin_ratio                0.000000
emoji_count                0.001400
punctuation_count          0.011058
repeated_char_patterns     0.002511
english_word_count         0.000000
dtype: float32

Training feature std:
text_length               7

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model architecture created

LOADING CHECKPOINT
Checkpoint type: <class 'dict'>
Checkpoint keys:
['model_state_dict', 'epoch', 'macro_f1', 'config']

Missing keys: ['muril.embeddings.word_embeddings.weight', 'muril.embeddings.position_embeddings.weight', 'muril.embeddings.token_type_embeddings.weight', 'muril.embeddings.LayerNorm.weight', 'muril.embeddings.LayerNorm.bias', 'muril.encoder.layer.0.attention.self.query.weight', 'muril.encoder.layer.0.attention.self.query.bias', 'muril.encoder.layer.0.attention.self.key.weight', 'muril.encoder.layer.0.attention.self.key.bias', 'muril.encoder.layer.0.attention.self.value.weight', 'muril.encoder.layer.0.attention.self.value.bias', 'muril.encoder.layer.0.attention.output.dense.weight', 'muril.encoder.layer.0.attention.output.dense.bias', 'muril.encoder.layer.0.attention.output.LayerNorm.weight', 'muril.encoder.layer.0.attention.output.LayerNorm.bias', 'muril.encoder.layer.0.intermediate.dense.weight', 'muril.encoder.layer.0.intermediate.dens

In [ ]:
# ============================================================
# INSPECT SAVED MuRIL FEATURE-FUSION CHECKPOINT
# ============================================================

import os
import torch

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/DravidianLangTech_2026/"
    "results/muril_feature_fusion_best/pytorch_model.bin"
)

print("=" * 70)
print("CHECKPOINT ARCHITECTURE INSPECTION")
print("=" * 70)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for k in checkpoint.keys():
        print(" ", k)

    state_dict = checkpoint["model_state_dict"]

else:
    state_dict = checkpoint

print("\nNumber of parameters:", len(state_dict))

print("\nFirst 30 parameter names:")
for i, name in enumerate(state_dict.keys()):
    if i >= 30:
        break
    print(f"{i:3d}: {name}")

print("\n" + "=" * 70)
print("ARCHITECTURE SIGNATURE")
print("=" * 70)

prefixes = set()

for name in state_dict.keys():
    prefixes.add(name.split(".")[0])

print("Top-level parameter prefixes:")
for p in sorted(prefixes):
    print(" ", p)

print("\n✓ Inspection completed")

CHECKPOINT ARCHITECTURE INSPECTION
Checkpoint type: <class 'dict'>

Checkpoint keys:
  model_state_dict
  epoch
  macro_f1
  config

Number of parameters: 209

First 30 parameter names:
  0: encoder.embeddings.word_embeddings.weight
  1: encoder.embeddings.position_embeddings.weight
  2: encoder.embeddings.token_type_embeddings.weight
  3: encoder.embeddings.LayerNorm.weight
  4: encoder.embeddings.LayerNorm.bias
  5: encoder.encoder.layer.0.attention.self.query.weight
  6: encoder.encoder.layer.0.attention.self.query.bias
  7: encoder.encoder.layer.0.attention.self.key.weight
  8: encoder.encoder.layer.0.attention.self.key.bias
  9: encoder.encoder.layer.0.attention.self.value.weight
 10: encoder.encoder.layer.0.attention.self.value.bias
 11: encoder.encoder.layer.0.attention.output.dense.weight
 12: encoder.encoder.layer.0.attention.output.dense.bias
 13: encoder.encoder.layer.0.attention.output.LayerNorm.weight
 14: encoder.encoder.layer.0.attention.output.LayerNorm.bias
 15: encode

In [ ]:
# ============================================================
# EXACT CHECKPOINT ARCHITECTURE DETAILS
# ============================================================

import torch

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/DravidianLangTech_2026/"
    "results/muril_feature_fusion_best/pytorch_model.bin"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

state_dict = checkpoint["model_state_dict"]

print("=" * 70)
print("EXACT SAVED MODEL ARCHITECTURE")
print("=" * 70)

print("Epoch    :", checkpoint.get("epoch"))
print("Macro F1 :", checkpoint.get("macro_f1"))
print("Config   :", checkpoint.get("config"))

print()
print("=" * 70)
print("NON-MuRIL / CUSTOM LAYERS")
print("=" * 70)

for name, tensor in state_dict.items():

    if (
        name.startswith("fusion.")
        or name.startswith("feature_projection.")
        or name.startswith("classifier.")
    ):
        print(f"{name:50s} {tuple(tensor.shape)}")

print()
print("=" * 70)
print("ENCODER SUMMARY")
print("=" * 70)

encoder_params = [
    (name, tensor)
    for name, tensor in state_dict.items()
    if name.startswith("encoder.")
]

print("Encoder parameters:", len(encoder_params))

for name, tensor in encoder_params[:10]:
    print(f"{name:50s} {tuple(tensor.shape)}")

print()
print("Encoder parameter count:")

encoder_count = sum(
    tensor.numel()
    for name, tensor in encoder_params
)

print(f"{encoder_count:,}")

print()
print("=" * 70)
print("TOTAL PARAMETER COUNT")
print("=" * 70)

total_count = sum(
    tensor.numel()
    for tensor in state_dict.values()
)

print(f"{total_count:,}")

print()
print("✓ Architecture inspection completed")

EXACT SAVED MODEL ARCHITECTURE
Epoch    : 3
Macro F1 : 0.8578599868236373
Config   : {'model_name': 'google/muril-base-cased', 'num_features': 11, 'feature_columns': ['text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count'], 'feature_dim': 64, 'fusion_dim': 256, 'num_labels': 2, 'max_length': 128, 'seed': 42}

NON-MuRIL / CUSTOM LAYERS
feature_projection.0.weight                        (64, 11)
feature_projection.0.bias                          (64,)
feature_projection.1.weight                        (64,)
feature_projection.1.bias                          (64,)
fusion.0.weight                                    (256, 832)
fusion.0.bias                                      (256,)
fusion.1.weight                                    (256,)
fusion.1.bias                                      (256,)
classifier.weight                                  (2, 256)
classifie

In [ ]:
# ============================================================
# FINAL CORRECT MuRIL + 11 FEATURE FUSION VALIDATION
# EXACT CHECKPOINT ARCHITECTURE
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from transformers import AutoModel, AutoTokenizer

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "google/muril-base-cased"

RESULTS_DIR = (
    "/content/drive/MyDrive/DravidianLangTech_2026/results"
)

CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_best",
    "pytorch_model.bin"
)

VALIDATION_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_validation_split_seed42_with_features.csv"
)

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

NUM_FEATURES = 11
FEATURE_DIM = 64
FUSION_DIM = 256
NUM_LABELS = 2
MAX_LENGTH = 128
BATCH_SIZE = 16

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# PRINT CONFIG
# ============================================================

print("=" * 70)
print("FINAL MuRIL + 11 FEATURE FUSION VALIDATION")
print("=" * 70)

print("Model       :", MODEL_NAME)
print("Checkpoint  :", CHECKPOINT_PATH)
print("Validation  :", VALIDATION_PATH)
print("Device      :", device)

# ============================================================
# CHECK FILES
# ============================================================

assert os.path.isfile(
    CHECKPOINT_PATH
), f"Checkpoint not found: {CHECKPOINT_PATH}"

assert os.path.isfile(
    VALIDATION_PATH
), f"Validation file not found: {VALIDATION_PATH}"

print()
print("✓ Checkpoint exists")
print("✓ Validation file exists")

# ============================================================
# LOAD VALIDATION DATA
# ============================================================

validation_df = pd.read_csv(
    VALIDATION_PATH
)

print()
print("=" * 70)
print("VALIDATION DATA")
print("=" * 70)

print("Shape:", validation_df.shape)

print("\nColumns:")
print(validation_df.columns.tolist())

# Check required columns
required_columns = ["Text", "Class"] + FEATURE_COLUMNS

missing_columns = [
    col
    for col in required_columns
    if col not in validation_df.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

print("\n✓ All required columns present")

# ============================================================
# FEATURE CHECK
# ============================================================

feature_df = validation_df[FEATURE_COLUMNS]

print()
print("=" * 70)
print("FEATURE CHECK")
print("=" * 70)

print(
    "Missing feature values:",
    int(feature_df.isna().sum().sum())
)

assert feature_df.isna().sum().sum() == 0, (
    "Feature values contain NaN"
)

print("✓ No missing feature values")

print("\nValidation feature means:")
print(feature_df.mean())

# ============================================================
# CLASS CHECK
# ============================================================

print()
print("=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

print(
    validation_df["Class"].value_counts().sort_index()
)

# ============================================================
# TOKENIZER
# ============================================================

print()
print("=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("✓ Tokenizer loaded")

# ============================================================
# DATASET
# ============================================================

class TamilAbuseFeatureDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        feature_columns,
        max_length=128
    ):

        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.feature_columns = feature_columns
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        text = str(row["Text"])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        features = row[
            self.feature_columns
        ].values.astype(np.float32)

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "features": torch.tensor(
                features,
                dtype=torch.float32
            ),
            "labels": torch.tensor(
                int(row["Class"]),
                dtype=torch.long
            )
        }

        if "token_type_ids" in encoding:

            item["token_type_ids"] = (
                encoding["token_type_ids"]
                .squeeze(0)
            )

        return item


val_fusion_dataset = TamilAbuseFeatureDataset(
    validation_df,
    tokenizer,
    FEATURE_COLUMNS,
    MAX_LENGTH
)

val_fusion_dataloader = torch.utils.data.DataLoader(
    val_fusion_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print()
print("✓ Validation dataset created")
print("Validation samples:", len(val_fusion_dataset))

# ============================================================
# EXACT MODEL ARCHITECTURE
# ============================================================

class ExactMuRILFeatureFusion(nn.Module):

    def __init__(
        self,
        model_name,
        num_features=11,
        feature_dim=64,
        fusion_dim=256,
        num_labels=2
    ):

        super().__init__()

        # ----------------------------------------------------
        # IMPORTANT:
        # checkpoint expects this module name:
        #
        # encoder.*
        # ----------------------------------------------------

        self.encoder = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = self.encoder.config.hidden_size

        assert hidden_size == 768, (
            f"Expected MuRIL hidden size 768, "
            f"got {hidden_size}"
        )

        # ----------------------------------------------------
        # Feature projection
        #
        # checkpoint:
        # feature_projection.0
        # feature_projection.1
        # ----------------------------------------------------

        self.feature_projection = nn.Sequential(
            nn.Linear(
                num_features,
                feature_dim
            ),
            nn.LayerNorm(
                feature_dim
            )
        )

        # ----------------------------------------------------
        # Fusion
        #
        # checkpoint:
        # fusion.0 = Linear(832, 256)
        # fusion.1 = LayerNorm(256)
        #
        # 768 + 64 = 832
        # ----------------------------------------------------

        self.fusion = nn.Sequential(
            nn.Linear(
                hidden_size + feature_dim,
                fusion_dim
            ),
            nn.LayerNorm(
                fusion_dim
            )
        )

        # ----------------------------------------------------
        # Classifier
        #
        # checkpoint:
        # classifier = Linear(256, 2)
        # ----------------------------------------------------

        self.classifier = nn.Linear(
            fusion_dim,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        features,
        token_type_ids=None
    ):

        # ----------------------------------------------------
        # MuRIL encoder
        # ----------------------------------------------------

        encoder_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }

        if token_type_ids is not None:

            encoder_kwargs["token_type_ids"] = (
                token_type_ids
            )

        outputs = self.encoder(
            **encoder_kwargs
        )

        # ----------------------------------------------------
        # CLS representation
        # ----------------------------------------------------

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        # ----------------------------------------------------
        # Project numerical features
        # ----------------------------------------------------

        projected_features = (
            self.feature_projection(features)
        )

        # ----------------------------------------------------
        # Concatenate MuRIL + features
        # ----------------------------------------------------

        combined = torch.cat(
            [
                cls_embedding,
                projected_features
            ],
            dim=1
        )

        # ----------------------------------------------------
        # Fusion
        # ----------------------------------------------------

        fused = self.fusion(
            combined
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        logits = self.classifier(
            fused
        )

        return logits


# ============================================================
# CREATE MODEL
# ============================================================

print()
print("=" * 70)
print("CREATING EXACT MODEL ARCHITECTURE")
print("=" * 70)

model = ExactMuRILFeatureFusion(
    model_name=MODEL_NAME,
    num_features=NUM_FEATURES,
    feature_dim=FEATURE_DIM,
    fusion_dim=FUSION_DIM,
    num_labels=NUM_LABELS
)

print("✓ Model architecture created")

# ============================================================
# LOAD CHECKPOINT
# ============================================================

print()
print("=" * 70)
print("LOADING TRAINED CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("Checkpoint keys:")
print(list(checkpoint.keys()))

state_dict = checkpoint[
    "model_state_dict"
]

print()
print("Saved training epoch:")
print(checkpoint.get("epoch"))

print("Saved validation Macro-F1:")
print(checkpoint.get("macro_f1"))

print()
print("Loading with STRICT=True...")

# ============================================================
# CRITICAL:
# STRICT TRUE
# ============================================================

load_result = model.load_state_dict(
    state_dict,
    strict=True
)

print()
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

assert len(load_result.missing_keys) == 0, (
    "ERROR: Missing model parameters!"
)

assert len(load_result.unexpected_keys) == 0, (
    "ERROR: Unexpected checkpoint parameters!"
)

print()
print("✓ EXACT CHECKPOINT LOADED")
print("✓ Missing keys: 0")
print("✓ Unexpected keys: 0")

# ============================================================
# MOVE MODEL TO DEVICE
# ============================================================

model.to(device)
model.eval()

print("✓ Model moved to:", device)

# ============================================================
# BATCH CHECK
# ============================================================

print()
print("=" * 70)
print("BATCH CHECK")
print("=" * 70)

first_batch = next(
    iter(val_fusion_dataloader)
)

print(
    "Input IDs:",
    first_batch["input_ids"].shape
)

print(
    "Attention:",
    first_batch["attention_mask"].shape
)

print(
    "Features:",
    first_batch["features"].shape
)

print(
    "Labels:",
    first_batch["labels"].shape
)

if "token_type_ids" in first_batch:

    print(
        "Token types:",
        first_batch["token_type_ids"].shape
    )

# ============================================================
# FORWARD CHECK
# ============================================================

with torch.no_grad():

    input_ids = (
        first_batch["input_ids"]
        .to(device)
    )

    attention_mask = (
        first_batch["attention_mask"]
        .to(device)
    )

    features = (
        first_batch["features"]
        .to(device)
    )

    token_type_ids = first_batch.get(
        "token_type_ids"
    )

    if token_type_ids is not None:

        token_type_ids = (
            token_type_ids.to(device)
        )

    test_logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        features=features,
        token_type_ids=token_type_ids
    )

print()
print("Test logits shape:", test_logits.shape)

assert test_logits.shape == (
    first_batch["input_ids"].shape[0],
    2
)

print("✓ Forward pass correct")

# ============================================================
# FINAL PREDICTION
# ============================================================

print()
print("=" * 70)
print("RUNNING FINAL VALIDATION")
print("=" * 70)

all_predictions = []
all_labels = []
all_logits = []

with torch.no_grad():

    for batch_idx, batch in enumerate(
        val_fusion_dataloader
    ):

        input_ids = (
            batch["input_ids"]
            .to(device)
        )

        attention_mask = (
            batch["attention_mask"]
            .to(device)
        )

        features = (
            batch["features"]
            .to(device)
        )

        labels = (
            batch["labels"]
            .to(device)
        )

        token_type_ids = batch.get(
            "token_type_ids"
        )

        if token_type_ids is not None:

            token_type_ids = (
                token_type_ids.to(device)
            )

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            features=features,
            token_type_ids=token_type_ids
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_logits.extend(
            logits.cpu().numpy()
        )

# ============================================================
# NUMPY
# ============================================================

all_predictions = np.asarray(
    all_predictions
)

all_labels = np.asarray(
    all_labels
)

all_logits = np.asarray(
    all_logits
)

print()
print("Predictions:", len(all_predictions))
print("Labels     :", len(all_labels))
print("Logits     :", all_logits.shape)

assert len(all_predictions) == len(
    validation_df
)

# ============================================================
# METRICS
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_precision, macro_recall, macro_f1, _ = \
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

weighted_f1 = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)[2]

# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 70)
print("FINAL VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy        : {accuracy:.4f}")
print(f"Macro Precision : {macro_precision:.4f}")
print(f"Macro Recall    : {macro_recall:.4f}")
print(f"Macro F1        : {macro_f1:.4f}")
print(f"Weighted F1     : {weighted_f1:.4f}")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print()
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

report = classification_report(
    all_labels,
    all_predictions,
    target_names=[
        "Non-Abusive",
        "Abusive"
    ],
    digits=4,
    zero_division=0
)

print(report)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print()
print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)

if cm.shape == (2, 2):

    tn, fp, fn, tp = cm.ravel()

    print()
    print("TN:", tn)
    print("FP:", fp)
    print("FN:", fn)
    print("TP:", tp)

# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

print()
print("=" * 70)
print("PREDICTION DISTRIBUTION")
print("=" * 70)

prediction_distribution = pd.Series(
    all_predictions
).value_counts().sort_index()

print(prediction_distribution)

# ============================================================
# COMPARE WITH SAVED BEST SCORE
# ============================================================

saved_f1 = float(
    checkpoint["macro_f1"]
)

print()
print("=" * 70)
print("CHECKPOINT SCORE COMPARISON")
print("=" * 70)

print(
    f"Saved best Macro-F1 : {saved_f1:.6f}"
)

print(
    f"Reloaded Macro-F1   : {macro_f1:.6f}"
)

difference = abs(
    saved_f1 - macro_f1
)

print(
    f"Absolute difference  : {difference:.6f}"
)

# ============================================================
# SAVE PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "Text": validation_df["Text"].values,

    "True_Label": all_labels,

    "Predicted_Label": all_predictions,

    "Logit_NonAbusive": all_logits[:, 0],

    "Logit_Abusive": all_logits[:, 1]
})

prediction_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_validation_predictions_CORRECT.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)

# ============================================================
# SAVE METRICS
# ============================================================

metrics_df = pd.DataFrame([{

    "Model":
        "MuRIL + 11 Feature Fusion",

    "Checkpoint_Macro_F1":
        saved_f1,

    "Reloaded_Accuracy":
        accuracy,

    "Reloaded_Macro_Precision":
        macro_precision,

    "Reloaded_Macro_Recall":
        macro_recall,

    "Reloaded_Macro_F1":
        macro_f1,

    "Reloaded_Weighted_F1":
        weighted_f1,

    "TN": tn if cm.shape == (2, 2) else None,

    "FP": fp if cm.shape == (2, 2) else None,

    "FN": fn if cm.shape == (2, 2) else None,

    "TP": tp if cm.shape == (2, 2) else None
}])

metrics_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_results_CORRECT.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

# ============================================================
# FINAL
# ============================================================

print()
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print("Predictions:")
print(prediction_path)

print()
print("Metrics:")
print(metrics_path)

print()
print("=" * 70)
print("✓ CORRECT VALIDATION EVALUATION COMPLETED")
print("=" * 70)

FINAL MuRIL + 11 FEATURE FUSION VALIDATION
Model       : google/muril-base-cased
Checkpoint  : /content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best/pytorch_model.bin
Validation  : /content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42_with_features.csv
Device      : cuda

✓ Checkpoint exists
✓ Validation file exists

VALIDATION DATA
Shape: (5178, 13)

Columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']

✓ All required columns present

FEATURE CHECK
Missing feature values: 0
✓ No missing feature values

Validation feature means:
text_length               79.196215
word_count                10.499421
tamil_chars               69.663770
latin_chars                0.000000
digits                     0.000193
tamil_ratio                0.880750
latin_ratio                0.000000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model architecture created

LOADING TRAINED CHECKPOINT
Checkpoint keys:
['model_state_dict', 'epoch', 'macro_f1', 'config']

Saved training epoch:
3
Saved validation Macro-F1:
0.8578599868236373

Loading with STRICT=True...

Missing keys: []
Unexpected keys: []

✓ EXACT CHECKPOINT LOADED
✓ Missing keys: 0
✓ Unexpected keys: 0
✓ Model moved to: cuda

BATCH CHECK
Input IDs: torch.Size([16, 128])
Attention: torch.Size([16, 128])
Features: torch.Size([16, 11])
Labels: torch.Size([16])
Token types: torch.Size([16, 128])

Test logits shape: torch.Size([16, 2])
✓ Forward pass correct

RUNNING FINAL VALIDATION

Predictions: 5178
Labels     : 5178
Logits     : (5178, 2)

FINAL VALIDATION RESULTS
Accuracy        : 0.8621
Macro Precision : 0.8621
Macro Recall    : 0.8618
Macro F1        : 0.8619
Weighted F1     : 0.8621

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 Non-Abusive     0.8620    0.8717    0.8668      2666
     Abusive     0.8622    0.8519    0.8570  

In [ ]:
# ============================================================
# FINAL RESULTS SUMMARY — MuRIL + 11 FEATURE FUSION
# ============================================================

import os
import pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

METRICS_FILE = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_results_CORRECT.csv"
)

PREDICTIONS_FILE = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_validation_predictions_CORRECT.csv"
)

HISTORY_FILE = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_training_history.csv"
)

SUMMARY_FILE = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_FINAL_SUMMARY.csv"
)

print("=" * 70)
print("FINAL MuRIL + 11 FEATURE FUSION RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

print("Metrics file     :", os.path.exists(METRICS_FILE))
print("Predictions file :", os.path.exists(PREDICTIONS_FILE))
print("History file     :", os.path.exists(HISTORY_FILE))

assert os.path.exists(METRICS_FILE), "Metrics file not found"
assert os.path.exists(PREDICTIONS_FILE), "Predictions file not found"
assert os.path.exists(HISTORY_FILE), "Training history file not found"

# ------------------------------------------------------------
# Load files
# ------------------------------------------------------------

metrics_df = pd.read_csv(METRICS_FILE)
predictions_df = pd.read_csv(PREDICTIONS_FILE)
history_df = pd.read_csv(HISTORY_FILE)

# ------------------------------------------------------------
# Show actual columns
# ------------------------------------------------------------

print()
print("=" * 70)
print("METRICS COLUMNS")
print("=" * 70)

print(metrics_df.columns.tolist())

# ------------------------------------------------------------
# Extract final metrics
# ------------------------------------------------------------

row = metrics_df.iloc[0]

# IMPORTANT:
# Your corrected CSV contains:
# Reloaded_Accuracy
# Reloaded_Macro_Precision
# Reloaded_Macro_Recall
# Reloaded_Macro_F1
# Reloaded_Weighted_F1

final_accuracy = row["Reloaded_Accuracy"]
final_precision = row["Reloaded_Macro_Precision"]
final_recall = row["Reloaded_Macro_Recall"]
final_macro_f1 = row["Reloaded_Macro_F1"]
final_weighted_f1 = row["Reloaded_Weighted_F1"]

checkpoint_f1 = row["Checkpoint_Macro_F1"]

tn = int(row["TN"])
fp = int(row["FP"])
fn = int(row["FN"])
tp = int(row["TP"])

# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL VALIDATION METRICS")
print("=" * 70)

print(f"Accuracy          : {final_accuracy:.4f}")
print(f"Macro Precision   : {final_precision:.4f}")
print(f"Macro Recall      : {final_recall:.4f}")
print(f"Macro F1          : {final_macro_f1:.4f}")
print(f"Weighted F1       : {final_weighted_f1:.4f}")

print()
print("Confusion Matrix:")
print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRAINING HISTORY")
print("=" * 70)

print(history_df.to_string(index=False))

# ------------------------------------------------------------
# Best training checkpoint vs reloaded evaluation
# ------------------------------------------------------------

print()
print("=" * 70)
print("CHECKPOINT VS RELOADED EVALUATION")
print("=" * 70)

print(f"Checkpoint Macro-F1 : {checkpoint_f1:.6f}")
print(f"Reloaded Macro-F1   : {final_macro_f1:.6f}")
print(
    f"Difference          : "
    f"{final_macro_f1 - checkpoint_f1:.6f}"
)

# ------------------------------------------------------------
# Prediction distribution
# ------------------------------------------------------------

if "Predicted_Label" in predictions_df.columns:

    print()
    print("=" * 70)
    print("PREDICTION DISTRIBUTION")
    print("=" * 70)

    print(
        predictions_df["Predicted_Label"]
        .value_counts()
        .sort_index()
    )

# ------------------------------------------------------------
# Create clean final summary
# ------------------------------------------------------------

summary_df = pd.DataFrame([{
    "Model": "MuRIL + 11 Feature Fusion",
    "Base_Model": "google/muril-base-cased",
    "Epochs": 3,
    "Learning_Rate": 2e-5,
    "Batch_Size": 16,
    "Max_Length": 128,
    "Num_Features": 11,
    "Checkpoint_Macro_F1": checkpoint_f1,
    "Final_Accuracy": final_accuracy,
    "Final_Macro_Precision": final_precision,
    "Final_Macro_Recall": final_recall,
    "Final_Macro_F1": final_macro_f1,
    "Final_Weighted_F1": final_weighted_f1,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}])

# ------------------------------------------------------------
# Save final summary
# ------------------------------------------------------------

summary_df.to_csv(
    SUMMARY_FILE,
    index=False
)

print()
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(summary_df.to_string(index=False))

print()
print("=" * 70)
print("SAVED")
print("=" * 70)

print("Final summary:")
print(SUMMARY_FILE)

print()
print("✓ FINAL RESULTS SUMMARY COMPLETED")

FINAL MuRIL + 11 FEATURE FUSION RESULTS
Metrics file     : True
Predictions file : True
History file     : True

METRICS COLUMNS
['Model', 'Checkpoint_Macro_F1', 'Reloaded_Accuracy', 'Reloaded_Macro_Precision', 'Reloaded_Macro_Recall', 'Reloaded_Macro_F1', 'Reloaded_Weighted_F1', 'TN', 'FP', 'FN', 'TP']

FINAL VALIDATION METRICS
Accuracy          : 0.8621
Macro Precision   : 0.8621
Macro Recall      : 0.8618
Macro F1          : 0.8619
Weighted F1       : 0.8621

Confusion Matrix:
TN = 2324
FP = 342
FN = 372
TP = 2140

TRAINING HISTORY
 epoch  training_loss  validation_loss  accuracy  macro_precision  macro_recall  macro_f1  weighted_f1
     1       0.493281         0.424282  0.836616         0.836460      0.836656  0.836526     0.836640
     2       0.345658         0.358965  0.842603         0.844180      0.843642  0.842588     0.842542
     3       0.250000         0.392177  0.857860         0.858538      0.858585  0.857860     0.857865

CHECKPOINT VS RELOADED EVALUATION
Checkpoint M

In [ ]:
# ============================================================
# FIND TEST CSV
# ============================================================

import os
import glob

BASE_DIR = "/content/drive/MyDrive/DravidianLangTech_2026"

print("=" * 70)
print("SEARCHING FOR TEST CSV FILES")
print("=" * 70)

csv_files = glob.glob(
    BASE_DIR + "/**/*.csv",
    recursive=True
)

for path in csv_files:
    try:
        import pandas as pd
        df = pd.read_csv(path, nrows=3)

        print(
            f"{path}\n"
            f"   rows(sampled)=3 | columns={list(df.columns)}"
        )

    except Exception as e:
        print(f"{path}  [could not read]")

print()
print("✓ Search completed")

SEARCHING FOR TEST CSV FILES
/content/drive/MyDrive/DravidianLangTech_2026/results/train_with_analysis_features.csv
   rows(sampled)=3 | columns=['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'script_type', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']
/content/drive/MyDrive/DravidianLangTech_2026/results/clean_modeling_dataset.csv
   rows(sampled)=3 | columns=['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'script_type', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']
/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_split.csv
   rows(sampled)=3 | columns=['index', 'split']
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_error_analysis.csv
   rows(sampled)=3 | columns=['Text', 'True_Label', 'Predicted_Label', 'Error_Type']
/content/drive/M

In [ ]:
# ============================================================
# FINAL TEST PREDICTION
# MuRIL + 11 FEATURE FUSION
# ============================================================

import os
import re
import unicodedata
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/DravidianLangTech_2026"
RESULTS_DIR = os.path.join(BASE_DIR, "results")

TEST_PATH = os.path.join(
    BASE_DIR,
    "data",
    "TestV2 - testV2.csv"
)

TRAIN_FEATURE_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_train_split_seed42_with_features.csv"
)

CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_best",
    "pytorch_model.bin"
)

OUTPUT_PATH = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_TEST_predictions.csv"
)

MODEL_NAME = "google/muril-base-cased"

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_FEATURES = 11
NUM_LABELS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# HEADER
# ============================================================

print("=" * 70)
print("FINAL TEST PREDICTION")
print("MuRIL + 11 FEATURE FUSION")
print("=" * 70)

print("Test file   :", TEST_PATH)
print("Checkpoint  :", CHECKPOINT_PATH)
print("Device      :", DEVICE)

assert os.path.exists(TEST_PATH), \
    f"Test file not found: {TEST_PATH}"

assert os.path.exists(TRAIN_FEATURE_PATH), \
    f"Training feature file not found: {TRAIN_FEATURE_PATH}"

assert os.path.exists(CHECKPOINT_PATH), \
    f"Checkpoint not found: {CHECKPOINT_PATH}"

print("✓ Required files exist")


# ============================================================
# LOAD TRAINING FEATURES
# ============================================================

print()
print("=" * 70)
print("LOADING TRAINING FEATURE STATISTICS")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FEATURE_PATH)

assert all(
    col in train_df.columns
    for col in FEATURE_COLUMNS
), "Training feature columns are missing"

# IMPORTANT:
# These are the exact statistics used to standardize
# the feature-fusion inputs during training.

feature_means = train_df[FEATURE_COLUMNS].mean()
feature_stds = train_df[FEATURE_COLUMNS].std()

# The original training pipeline effectively used 1.0
# for zero-variance features.
feature_stds = feature_stds.replace(0, 1.0)

print("Training feature means:")
print(feature_means)

print()
print("Training feature std:")
print(feature_stds)

print("✓ Training feature statistics loaded")


# ============================================================
# FEATURE EXTRACTION
# ============================================================

def is_tamil_char(ch):
    """
    Tamil Unicode block:
    U+0B80 - U+0BFF
    """
    return "\u0B80" <= ch <= "\u0BFF"


def is_latin_char(ch):
    """
    Detect Latin alphabet characters.
    """
    return (
        ("A" <= ch <= "Z") or
        ("a" <= ch <= "z")
    )


def count_emoji(text):
    """
    Approximate emoji counter consistent with Unicode ranges.
    """
    count = 0

    for ch in text:
        code = ord(ch)

        if (
            0x1F300 <= code <= 0x1FAFF or
            0x2600 <= code <= 0x27BF
        ):
            count += 1

    return count


def count_punctuation(text):
    """
    Count Unicode punctuation characters.
    """
    return sum(
        1
        for ch in text
        if unicodedata.category(ch).startswith("P")
    )


def count_repeated_char_patterns(text):
    """
    Count simple repeated-character patterns such as:
    aaa, xxxx, !!!! etc.
    """
    return len(
        re.findall(
            r"(.)\1{2,}",
            text,
            flags=re.UNICODE
        )
    )


def extract_features(text):

    if pd.isna(text):
        text = ""

    text = str(text)

    # Character counts
    text_length = len(text)

    tamil_chars = sum(
        is_tamil_char(ch)
        for ch in text
    )

    latin_chars = sum(
        is_latin_char(ch)
        for ch in text
    )

    digits = sum(
        ch.isdigit()
        for ch in text
    )

    # Words
    words = text.split()
    word_count = len(words)

    # English-word detection
    #
    # IMPORTANT:
    # We preserve the same broad English-word concept used
    # in the existing feature pipeline.
    english_word_count = sum(
        1
        for word in words
        if re.fullmatch(
            r"[A-Za-z]+",
            word
        )
    )

    # Ratios
    if text_length > 0:
        tamil_ratio = tamil_chars / text_length
        latin_ratio = latin_chars / text_length
    else:
        tamil_ratio = 0.0
        latin_ratio = 0.0

    emoji_count = count_emoji(text)

    punctuation_count = count_punctuation(text)

    repeated_char_patterns = count_repeated_char_patterns(text)

    return {
        "text_length": text_length,
        "word_count": word_count,
        "tamil_chars": tamil_chars,
        "latin_chars": latin_chars,
        "digits": digits,
        "tamil_ratio": tamil_ratio,
        "latin_ratio": latin_ratio,
        "emoji_count": emoji_count,
        "punctuation_count": punctuation_count,
        "repeated_char_patterns": repeated_char_patterns,
        "english_word_count": english_word_count
    }


# ============================================================
# LOAD TEST DATA
# ============================================================

print()
print("=" * 70)
print("LOADING TEST DATA")
print("=" * 70)

test_df = pd.read_csv(TEST_PATH)

assert "Text" in test_df.columns, \
    "Test file does not contain 'Text' column"

print("Test shape:", test_df.shape)
print("Test columns:", test_df.columns.tolist())

print()
print("First 5 test rows:")
print(test_df.head())

print()
print("✓ Test data loaded")


# ============================================================
# EXTRACT 11 FEATURES
# ============================================================

print()
print("=" * 70)
print("EXTRACTING 11 TEST FEATURES")
print("=" * 70)

feature_records = []

for text in test_df["Text"]:
    feature_records.append(
        extract_features(text)
    )

test_feature_df = pd.DataFrame(
    feature_records,
    columns=FEATURE_COLUMNS
)

print("Feature shape:", test_feature_df.shape)

print()
print("Raw test feature means:")
print(test_feature_df.mean())

print()
print("Raw test feature std:")
print(test_feature_df.std())

print("✓ 11 features extracted")


# ============================================================
# SCALE FEATURES
# ============================================================

print()
print("=" * 70)
print("SCALING TEST FEATURES")
print("=" * 70)

scaled_features = (
    test_feature_df - feature_means
) / feature_stds

# Make sure there are no NaNs/Infs
scaled_features = scaled_features.replace(
    [np.inf, -np.inf],
    0
).fillna(0)

print("Scaled feature means:")
print(scaled_features.mean())

print()
print("Scaled feature std:")
print(scaled_features.std())

print()
print("First 5 scaled feature vectors:")
print(scaled_features.head())

assert scaled_features.shape[1] == 11

print("✓ Features scaled using TRAINING statistics")


# ============================================================
# TOKENIZER
# ============================================================

print()
print("=" * 70)
print("LOADING MuRIL TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("✓ Tokenizer loaded")


# ============================================================
# DATASET
# ============================================================

class TestFeatureDataset(Dataset):

    def __init__(
        self,
        texts,
        features,
        tokenizer,
        max_length=128
    ):

        self.texts = list(texts)

        self.features = torch.tensor(
            features.values,
            dtype=torch.float32
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "features": self.features[idx]
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = (
                encoding["token_type_ids"]
                .squeeze(0)
            )

        return item


test_dataset = TestFeatureDataset(
    test_df["Text"],
    scaled_features,
    tokenizer,
    MAX_LENGTH
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print()
print("✓ Test dataset created")
print("Test samples:", len(test_dataset))


# ============================================================
# EXACT SAVED MODEL ARCHITECTURE
# ============================================================

print()
print("=" * 70)
print("CREATING EXACT SAVED MODEL ARCHITECTURE")
print("=" * 70)


class MuRILFeatureFusion(nn.Module):

    def __init__(
        self,
        model_name,
        num_features=11,
        num_labels=2,
        feature_dim=64,
        fusion_dim=256
    ):

        super().__init__()

        # IMPORTANT:
        # The checkpoint inspection showed that the saved model
        # uses the top-level name "encoder".

        self.encoder = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = (
            self.encoder.config.hidden_size
        )

        self.feature_projection = nn.Sequential(
            nn.Linear(
                num_features,
                feature_dim
            ),
            nn.LayerNorm(feature_dim)
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                hidden_size + feature_dim,
                fusion_dim
            ),
            nn.LayerNorm(fusion_dim)
        )

        self.classifier = nn.Linear(
            fusion_dim,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        features,
        token_type_ids=None
    ):

        encoder_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }

        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids

        outputs = self.encoder(
            **encoder_kwargs
        )

        # CLS representation
        cls_output = outputs.last_hidden_state[:, 0, :]

        feature_output = self.feature_projection(
            features
        )

        combined = torch.cat(
            [
                cls_output,
                feature_output
            ],
            dim=1
        )

        fused = self.fusion(
            combined
        )

        logits = self.classifier(
            fused
        )

        return logits


model = MuRILFeatureFusion(
    model_name=MODEL_NAME,
    num_features=NUM_FEATURES,
    num_labels=NUM_LABELS,
    feature_dim=64,
    fusion_dim=256
)

print("✓ Model architecture created")


# ============================================================
# LOAD CHECKPOINT
# ============================================================

print()
print("=" * 70)
print("LOADING TRAINED CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("Checkpoint keys:")
print(list(checkpoint.keys()))

state_dict = checkpoint["model_state_dict"]

# ------------------------------------------------------------
# STRICT CHECKPOINT LOAD
# ------------------------------------------------------------

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False
)

print()
print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

assert len(missing_keys) == 0, \
    f"Missing checkpoint parameters: {missing_keys}"

assert len(unexpected_keys) == 0, \
    f"Unexpected checkpoint parameters: {unexpected_keys}"

print()
print("✓ EXACT CHECKPOINT LOADED")
print("✓ Missing keys: 0")
print("✓ Unexpected keys: 0")

model.to(DEVICE)
model.eval()

print("✓ Model moved to:", DEVICE)


# ============================================================
# BATCH CHECK
# ============================================================

print()
print("=" * 70)
print("TEST BATCH CHECK")
print("=" * 70)

first_batch = next(iter(test_loader))

input_ids = first_batch["input_ids"].to(DEVICE)
attention_mask = first_batch["attention_mask"].to(DEVICE)
features = first_batch["features"].to(DEVICE)

token_type_ids = first_batch.get(
    "token_type_ids"
)

if token_type_ids is not None:
    token_type_ids = token_type_ids.to(DEVICE)

with torch.no_grad():

    test_logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        features=features,
        token_type_ids=token_type_ids
    )

print("Input IDs :", input_ids.shape)
print("Attention :", attention_mask.shape)
print("Features  :", features.shape)

if token_type_ids is not None:
    print("Token types:", token_type_ids.shape)

print("Logits    :", test_logits.shape)

assert test_logits.shape[1] == 2

print("✓ Test forward pass correct")


# ============================================================
# RUN TEST PREDICTION
# ============================================================

print()
print("=" * 70)
print("RUNNING TEST PREDICTION")
print("=" * 70)

all_predictions = []
all_logits = []

with torch.no_grad():

    for batch_idx, batch in enumerate(test_loader):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        features = batch["features"].to(DEVICE)

        token_type_ids = batch.get(
            "token_type_ids"
        )

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            features=features,
            token_type_ids=token_type_ids
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_logits.extend(
            logits.cpu().numpy()
        )

        if (batch_idx + 1) % 50 == 0:
            print(
                f"Processed batches: "
                f"{batch_idx + 1}/{len(test_loader)}"
            )


# ============================================================
# CONVERT RESULTS
# ============================================================

all_predictions = np.array(
    all_predictions
)

all_logits = np.array(
    all_logits
)

print()
print("=" * 70)
print("TEST PREDICTION SUMMARY")
print("=" * 70)

print("Test samples :", len(test_df))
print("Predictions   :", len(all_predictions))
print("Logits shape  :", all_logits.shape)

assert len(all_predictions) == len(test_df)
assert all_logits.shape == (
    len(test_df),
    2
)

print("✓ Prediction count matches test set")


# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

print()
print("=" * 70)
print("TEST PREDICTION DISTRIBUTION")
print("=" * 70)

prediction_distribution = pd.Series(
    all_predictions
).value_counts().sort_index()

print(prediction_distribution)


# ============================================================
# CREATE OUTPUT
# ============================================================

submission_df = pd.DataFrame({
    "Text": test_df["Text"],
    "Predicted_Label": all_predictions,
    "Logit_NonAbusive": all_logits[:, 0],
    "Logit_Abusive": all_logits[:, 1]
})

# Preserve any original ID/index column if present
for possible_id in [
    "id",
    "ID",
    "Id",
    "index"
]:

    if possible_id in test_df.columns:

        submission_df.insert(
            0,
            possible_id,
            test_df[possible_id].values
        )

        break


# ============================================================
# SAVE TEST PREDICTIONS
# ============================================================

submission_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print()
print("=" * 70)
print("TEST PREDICTIONS SAVED")
print("=" * 70)

print(OUTPUT_PATH)

print()
print("Output shape:", submission_df.shape)

print()
print("First 10 predictions:")
print(submission_df.head(10))

print()
print("=" * 70)
print("✓ FINAL TEST PREDICTION COMPLETED")
print("=" * 70)

FINAL TEST PREDICTION
MuRIL + 11 FEATURE FUSION
Test file   : /content/drive/MyDrive/DravidianLangTech_2026/data/TestV2 - testV2.csv
Checkpoint  : /content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best/pytorch_model.bin
Device      : cuda
✓ Required files exist

LOADING TRAINING FEATURE STATISTICS
Training feature means:
text_length               78.689797
word_count                10.429668
tamil_chars               69.204452
latin_chars                0.000000
digits                     0.000048
tamil_ratio                0.880604
latin_ratio                0.000000
emoji_count                0.001400
punctuation_count          0.011058
repeated_char_patterns     0.002511
english_word_count         0.000000
dtype: float64

Training feature std:
text_length               71.339972
word_count                 9.402255
tamil_chars               62.095445
latin_chars                1.000000
digits                     0.006949
tamil_ratio                0.030903
la

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model architecture created

LOADING TRAINED CHECKPOINT
Checkpoint keys:
['model_state_dict', 'epoch', 'macro_f1', 'config']

Missing keys: []
Unexpected keys: []

✓ EXACT CHECKPOINT LOADED
✓ Missing keys: 0
✓ Unexpected keys: 0
✓ Model moved to: cuda

TEST BATCH CHECK
Input IDs : torch.Size([16, 128])
Attention : torch.Size([16, 128])
Features  : torch.Size([16, 11])
Token types: torch.Size([16, 128])
Logits    : torch.Size([16, 2])
✓ Test forward pass correct

RUNNING TEST PREDICTION
Processed batches: 50/58

TEST PREDICTION SUMMARY
Test samples : 913
Predictions   : 913
Logits shape  : (913, 2)
✓ Prediction count matches test set

TEST PREDICTION DISTRIBUTION
0    574
1    339
Name: count, dtype: int64

TEST PREDICTIONS SAVED
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_TEST_predictions.csv

Output shape: (913, 4)

First 10 predictions:
                                                Text  Predicted_Label  \
0  லக்ஷ்மி அம்மா நீங்க புலம்புங்க அவளுக அவளு

In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/DravidianLangTech_2026"

print("=" * 70)
print("FINAL SUBMISSION FORMAT CHECK")
print("=" * 70)

# ---------------------------------------------------------
# 1. Current prediction file
# ---------------------------------------------------------
pred_path = os.path.join(
    BASE,
    "results",
    "muril_feature_fusion_TEST_predictions.csv"
)

print("\nPrediction file:")
print(pred_path)
print("Exists:", os.path.exists(pred_path))

pred_df = pd.read_csv(pred_path)

print("\nPrediction shape:")
print(pred_df.shape)

print("\nPrediction columns:")
print(pred_df.columns.tolist())

print("\nFirst 5 rows:")
print(pred_df.head())

print("\nPrediction distribution:")
print(pred_df["Predicted_Label"].value_counts().sort_index())


# ---------------------------------------------------------
# 2. Test file
# ---------------------------------------------------------
test_path = os.path.join(
    BASE,
    "data",
    "TestV2 - testV2.csv"
)

test_df = pd.read_csv(test_path)

print("\n" + "=" * 70)
print("TEST FILE")
print("=" * 70)

print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

print("\nFirst 5 rows:")
print(test_df.head())


# ---------------------------------------------------------
# 3. Check whether an ID column exists
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("POSSIBLE ID COLUMNS")
print("=" * 70)

possible_id_columns = [
    "id",
    "ID",
    "index",
    "Index",
    "tweet_id",
    "Tweet_ID",
    "sample_id",
    "sampleId"
]

found_ids = [
    c for c in test_df.columns
    if c in possible_id_columns
]

if found_ids:
    print("Found:", found_ids)
else:
    print("No obvious ID column found.")


# ---------------------------------------------------------
# 4. Search for submission/sample files
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("SEARCHING FOR SUBMISSION-RELATED FILES")
print("=" * 70)

search_dirs = [
    os.path.join(BASE, "data"),
    os.path.join(BASE, "results")
]

keywords = [
    "submission",
    "sample",
    "submit",
    "test"
]

matches = []

for directory in search_dirs:
    if not os.path.exists(directory):
        continue

    for root, dirs, files in os.walk(directory):
        for file in files:
            lower = file.lower()

            if any(k in lower for k in keywords):
                matches.append(os.path.join(root, file))

for path in sorted(matches):
    print(path)

print("\nFiles found:", len(matches))


# ---------------------------------------------------------
# 5. Inspect CSV candidates
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("CSV FORMAT INSPECTION")
print("=" * 70)

for path in sorted(matches):

    if not path.lower().endswith(".csv"):
        continue

    try:
        df = pd.read_csv(path)

        print("\nFILE:")
        print(os.path.basename(path))

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

        print(df.head(2))

    except Exception as e:
        print("Could not read:", path)
        print("Error:", e)


print("\n" + "=" * 70)
print("FORMAT CHECK COMPLETED")
print("=" * 70)

FINAL SUBMISSION FORMAT CHECK

Prediction file:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_TEST_predictions.csv
Exists: True

Prediction shape:
(913, 4)

Prediction columns:
['Text', 'Predicted_Label', 'Logit_NonAbusive', 'Logit_Abusive']

First 5 rows:
                                                Text  Predicted_Label  \
0  லக்ஷ்மி அம்மா நீங்க புலம்புங்க அவளுக அவளுகபாட்...                1   
1  இன்னும் கைது பண்ணல... அனைத்து பெற்றோர்களும், க...                0   
2  அப்பா,அம்மா, அந்த இன்டர்வியூ பண்ற வக்கிரம்புடி...                1   
3  Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க கார...                0   
4  எல்லாமே script thaan. ஷகீலா உங்க scriptum அரும...                0   

   Logit_NonAbusive  Logit_Abusive  
0         -1.582235       1.534207  
1          2.195306      -2.549205  
2         -0.904284       0.254149  
3          1.619103      -1.539040  
4          2.844470      -2.824374  

Prediction distribution:
Predicted_Label
0    574
1 

In [ ]:
import os
import pandas as pd

# ============================================================
# FINAL SUBMISSION GENERATION
# MuRIL + 11 Feature Fusion
# ============================================================

BASE = "/content/drive/MyDrive/DravidianLangTech_2026"

TEST_PATH = os.path.join(
    BASE,
    "data",
    "TestV2 - testV2.csv"
)

PRED_PATH = os.path.join(
    BASE,
    "results",
    "muril_feature_fusion_TEST_predictions.csv"
)

RESULTS_DIR = os.path.join(BASE, "results")

print("=" * 70)
print("CREATING FINAL SUBMISSION FILES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load test data
# ------------------------------------------------------------
test_df = pd.read_csv(TEST_PATH)

# ------------------------------------------------------------
# 2. Load model predictions
# ------------------------------------------------------------
pred_df = pd.read_csv(PRED_PATH)

print("\nTest shape       :", test_df.shape)
print("Prediction shape :", pred_df.shape)

# ------------------------------------------------------------
# 3. Basic integrity checks
# ------------------------------------------------------------
assert len(test_df) == len(pred_df), (
    f"Row mismatch: test={len(test_df)}, predictions={len(pred_df)}"
)

assert "Text" in test_df.columns
assert "Predicted_Label" in pred_df.columns

# ------------------------------------------------------------
# 4. Verify row order using Text
# ------------------------------------------------------------
same_text = test_df["Text"].astype(str).equals(
    pred_df["Text"].astype(str)
)

print("\nText/order identical:", same_text)

if not same_text:
    raise ValueError(
        "STOP: Test Text and prediction Text are not in identical order."
    )

print("✓ Row order verified")

# ------------------------------------------------------------
# 5. Verify labels
# ------------------------------------------------------------
valid_labels = {0, 1}

actual_labels = set(
    pred_df["Predicted_Label"].dropna().astype(int).unique()
)

print("Prediction labels:", sorted(actual_labels))

if not actual_labels.issubset(valid_labels):
    raise ValueError(
        f"Invalid prediction labels found: {actual_labels}"
    )

print("✓ Labels are valid 0/1")

# ------------------------------------------------------------
# 6. Verify prediction count
# ------------------------------------------------------------
print("\nNumber of test samples :", len(test_df))
print("Number of predictions  :", len(pred_df))

assert len(test_df) == 913
assert len(pred_df) == 913

print("✓ Exactly 913 predictions")

# ------------------------------------------------------------
# 7. Prediction distribution
# ------------------------------------------------------------
print("\nPrediction distribution:")
print(
    pred_df["Predicted_Label"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 8. Create MINIMAL submission
# ------------------------------------------------------------
#
# IMPORTANT:
# We use "Class" as the submission column because
# the training dataset uses "Class" for the target.
#
# This produces:
#
# Class
# 1
# 0
# 1
# ...
#
# ------------------------------------------------------------

submission = pd.DataFrame({
    "Class": pred_df["Predicted_Label"].astype(int)
})

submission_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_FINAL_SUBMISSION.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print("\n✓ Minimal submission saved:")
print(submission_path)

print("\nSubmission shape:")
print(submission.shape)

print("\nSubmission columns:")
print(submission.columns.tolist())

print("\nFirst 10 rows:")
print(submission.head(10))

# ------------------------------------------------------------
# 9. Create backup submission containing Text + Class
# ------------------------------------------------------------

submission_with_text = pd.DataFrame({
    "Text": test_df["Text"],
    "Class": pred_df["Predicted_Label"].astype(int)
})

submission_with_text_path = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_FINAL_SUBMISSION_WITH_TEXT.csv"
)

submission_with_text.to_csv(
    submission_with_text_path,
    index=False
)

print("\n✓ Backup submission saved:")
print(submission_with_text_path)

# ------------------------------------------------------------
# 10. Final verification
# ------------------------------------------------------------

check_submission = pd.read_csv(submission_path)

assert check_submission.shape == (913, 1)
assert check_submission.columns.tolist() == ["Class"]
assert check_submission["Class"].isin([0, 1]).all()

print("\n" + "=" * 70)
print("FINAL SUBMISSION VERIFICATION")
print("=" * 70)

print("Rows       :", len(check_submission))
print("Columns    :", check_submission.columns.tolist())
print("Labels     :", sorted(check_submission["Class"].unique()))
print("Class 0    :", int((check_submission["Class"] == 0).sum()))
print("Class 1    :", int((check_submission["Class"] == 1).sum()))

print("\n✓ Submission file is internally valid")
print("=" * 70)

CREATING FINAL SUBMISSION FILES

Test shape       : (913, 1)
Prediction shape : (913, 4)

Text/order identical: True
✓ Row order verified
Prediction labels: [np.int64(0), np.int64(1)]
✓ Labels are valid 0/1

Number of test samples : 913
Number of predictions  : 913
✓ Exactly 913 predictions

Prediction distribution:
Predicted_Label
0    574
1    339
Name: count, dtype: int64

✓ Minimal submission saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_FINAL_SUBMISSION.csv

Submission shape:
(913, 1)

Submission columns:
['Class']

First 10 rows:
   Class
0      1
1      0
2      1
3      0
4      0
5      0
6      1
7      0
8      1
9      0

✓ Backup submission saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_FINAL_SUBMISSION_WITH_TEXT.csv

FINAL SUBMISSION VERIFICATION
Rows       : 913
Columns    : ['Class']
Labels     : [np.int64(0), np.int64(1)]
Class 0    : 574
Class 1    : 339

✓ Submission file is internally valid


In [ ]:
# ============================================================
# STEP 1 — RECONNECT GOOGLE DRIVE + VERIFY SAVED MODEL
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

print("=" * 70)
print("CHECKING SAVED FEATURE-FUSION MODEL")
print("=" * 70)

print("Results directory:")
print(RESULTS_DIR)

print("\nFiles related to feature fusion:")

for f in os.listdir(RESULTS_DIR):
    if "feature_fusion" in f.lower():
        path = os.path.join(RESULTS_DIR, f)
        print(f" - {f}")

BEST_MODEL_PATH = os.path.join(
    RESULTS_DIR,
    "muril_feature_fusion_best"
)

print("\nCheckpoint path:")
print(BEST_MODEL_PATH)

if os.path.exists(BEST_MODEL_PATH):
    print("\n✓ FEATURE-FUSION CHECKPOINT FOUND")
else:
    print("\n✗ CHECKPOINT NOT FOUND")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CHECKING SAVED FEATURE-FUSION MODEL
Results directory:
/content/drive/MyDrive/DravidianLangTech_2026/results

Files related to feature fusion:
 - muril_feature_fusion_error_analysis.csv
 - muril_feature_fusion_confusion_matrix.png
 - muril_feature_fusion_best
 - muril_feature_fusion_training_history.csv

Checkpoint path:
/content/drive/MyDrive/DravidianLangTech_2026/results/muril_feature_fusion_best

✓ FEATURE-FUSION CHECKPOINT FOUND


In [ ]:
# ============================================================
# STEP 2 — LOAD FIXED TRAIN / VALIDATION SPLITS
# ============================================================

import pandas as pd
import os

RESULTS_DIR = "/content/drive/MyDrive/DravidianLangTech_2026/results"

TRAIN_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_train_split_seed42_with_features.csv"
)

VAL_PATH = os.path.join(
    RESULTS_DIR,
    "fixed_validation_split_seed42_with_features.csv"
)

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VAL_PATH)

print("=" * 70)
print("FIXED DATASET LOADED")
print("=" * 70)

print("Train shape      :", train_df.shape)
print("Validation shape :", validation_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nClass distribution — Train:")
print(train_df["Class"].value_counts())

print("\nClass distribution — Validation:")
print(validation_df["Class"].value_counts())

FIXED DATASET LOADED
Train shape      : (20709, 13)
Validation shape : (5178, 13)

Train columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']

Validation columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']

Class distribution — Train:
Class
0    10664
1    10045
Name: count, dtype: int64

Class distribution — Validation:
Class
0    2666
1    2512
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 3 — LOAD MuRIL TOKENIZER
# ============================================================

import torch
from transformers import AutoTokenizer

MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("=" * 70)
print("TOKENIZER LOADED")
print("=" * 70)

print("Model:", MODEL_NAME)
print("Vocabulary size:", len(tokenizer))

TOKENIZER LOADED
Model: google/muril-base-cased
Vocabulary size: 197258


In [ ]:
# ============================================================
# STEP 3B — MuRIL FEATURE-FUSION MODEL
# ============================================================

import torch
import torch.nn as nn
from transformers import AutoModel


class MuRILFeatureFusion(nn.Module):

    def __init__(
        self,
        model_name="google/muril-base-cased",
        num_features=11,
        num_labels=2
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = self.encoder.config.hidden_size

        # 11 numerical features -> 64 dimensions
        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        # MuRIL 768 + feature 64 = 832
        self.fusion = nn.Sequential(
            nn.Linear(hidden_size + 64, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Linear(
            256,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        features,
        token_type_ids=None,
        labels=None
    ):

        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # CLS representation
        text_embedding = encoder_outputs.last_hidden_state[:, 0, :]

        # Numerical features
        feature_embedding = self.feature_projection(
            features
        )

        # Fusion
        combined = torch.cat(
            [
                text_embedding,
                feature_embedding
            ],
            dim=1
        )

        fused = self.fusion(combined)

        logits = self.classifier(fused)

        if labels is not None:

            loss_fn = nn.CrossEntropyLoss()

            loss = loss_fn(
                logits,
                labels
            )

            return {
                "loss": loss,
                "logits": logits
            }

        return {
            "logits": logits
        }

In [ ]:
# ============================================================
# STEP 4 — FEATURE DATASET
# ============================================================

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]

print("=" * 70)
print("FEATURE VERIFICATION")
print("=" * 70)

print("Number of features:", len(FEATURE_COLUMNS))

for i, col in enumerate(FEATURE_COLUMNS):
    print(f"{i:2d}. {col}")

missing_train = [
    c for c in FEATURE_COLUMNS
    if c not in train_df.columns
]

missing_val = [
    c for c in FEATURE_COLUMNS
    if c not in validation_df.columns
]

print("\nMissing train features:", missing_train)
print("Missing validation features:", missing_val)

assert len(missing_train) == 0
assert len(missing_val) == 0

print("\n✓ All 11 features available")

FEATURE VERIFICATION
Number of features: 11
 0. text_length
 1. word_count
 2. tamil_chars
 3. latin_chars
 4. digits
 5. tamil_ratio
 6. latin_ratio
 7. emoji_count
 8. punctuation_count
 9. repeated_char_patterns
10. english_word_count

Missing train features: []
Missing validation features: []

✓ All 11 features available


In [ ]:
# ============================================================
# STEP 5 — MuRIL FEATURE-FUSION DATASET
# ============================================================

from torch.utils.data import Dataset, DataLoader


class TamilMuRILFeatureDataset(Dataset):

    def __init__(
        self,
        df,
        tokenizer,
        feature_columns,
        max_length=128
    ):

        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.feature_columns = feature_columns
        self.max_length = max_length

        self.texts = self.df["Text"].astype(str).tolist()

        self.labels = (
            self.df["Class"]
            .astype(int)
            .tolist()
        )

        self.features = (
            self.df[feature_columns]
            .astype("float32")
            .values
        )

        # ----------------------------------------------------
        # Feature normalization
        # ----------------------------------------------------
        # IMPORTANT:
        # Your previous training used normalized feature
        # values. We will use the saved train/validation
        # feature values as they are stored.
        # ----------------------------------------------------

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = self.texts[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "features": torch.tensor(
                self.features[idx],
                dtype=torch.float32
            ),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = (
                encoding["token_type_ids"].squeeze(0)
            )

        return item

In [ ]:
# ============================================================
# CHECK FEATURE VALUES
# ============================================================

print("=" * 70)
print("FEATURE VALUE CHECK")
print("=" * 70)

print(train_df[FEATURE_COLUMNS].head())

print("\nMeans:")
print(train_df[FEATURE_COLUMNS].mean())

print("\nStd:")
print(train_df[FEATURE_COLUMNS].std())

FEATURE VALUE CHECK
   text_length  word_count  tamil_chars  latin_chars  digits  tamil_ratio  \
0           54           6           49            0       0     0.907407   
1           41           6           36            0       0     0.878049   
2           68           7           62            0       0     0.911765   
3            7           1            7            0       0     1.000000   
4           64           9           56            0       0     0.875000   

   latin_ratio  emoji_count  punctuation_count  repeated_char_patterns  \
0          0.0            0                  0                       0   
1          0.0            0                  0                       0   
2          0.0            0                  0                       0   
3          0.0            0                  0                       0   
4          0.0            0                  0                       0   

   english_word_count  
0                   0  
1                   0  


In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT + PATHS
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = "/content/drive/MyDrive/DravidianLangTech_2026"
DATA_DIR = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train2.csv")
TEST_PATH = os.path.join(DATA_DIR, "TestV2 - testV2.csv")

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("Device:", DEVICE)
print("Seed:", SEED)
print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)

ENVIRONMENT
Device: cuda
Seed: 42
Train: /content/drive/MyDrive/DravidianLangTech_2026/data/train2.csv
Test : /content/drive/MyDrive/DravidianLangTech_2026/data/TestV2 - testV2.csv


In [ ]:
# ============================================================
# CELL 2 — LOAD ORIGINAL DATASET
# ============================================================

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("=" * 70)
print("ORIGINAL DATA")
print("=" * 70)

print("Train shape:", train_raw.shape)
print("Train columns:", train_raw.columns.tolist())

print()
print("Test shape:", test_raw.shape)
print("Test columns:", test_raw.columns.tolist())

print()
print("Train class distribution:")
print(train_raw["Class"].value_counts().sort_index())

ORIGINAL DATA
Train shape: (25946, 2)
Train columns: ['Text', 'Class']

Test shape: (913, 1)
Test columns: ['Text']

Train class distribution:
Class
0    13361
1    12585
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 3 — DUPLICATE / CONFLICT CLEANING
# ============================================================

df = train_raw.copy()

df["Text"] = df["Text"].astype(str)

# ------------------------------------------------------------
# Find texts having conflicting labels
# ------------------------------------------------------------

label_counts = df.groupby("Text")["Class"].nunique()

conflicting_texts = label_counts[
    label_counts > 1
].index.tolist()

print("=" * 70)
print("DATA CLEANING")
print("=" * 70)

print("Original rows:", len(df))
print("Conflicting texts:", len(conflicting_texts))

if len(conflicting_texts) > 0:
    print("\nConflicting examples:")
    print(
        df[df["Text"].isin(conflicting_texts)]
        [["Text", "Class"]]
        .to_string(index=False)
    )

# ------------------------------------------------------------
# Remove all conflicting texts
# ------------------------------------------------------------

df = df[
    ~df["Text"].isin(conflicting_texts)
].copy()

# ------------------------------------------------------------
# Remove exact duplicate rows
# ------------------------------------------------------------

before_duplicates = len(df)

df = df.drop_duplicates(
    subset=["Text"],
    keep="first"
).reset_index(drop=True)

duplicates_removed = before_duplicates - len(df)

print()
print("Rows after removing conflicts:", before_duplicates)
print("Duplicate rows removed:", duplicates_removed)

print()
print("FINAL CLEAN DATASET")
print("Rows:", len(df))
print("Unique texts:", df["Text"].nunique())

print()
print("Class distribution:")
print(df["Class"].value_counts().sort_index())

DATA CLEANING
Original rows: 25946
Conflicting texts: 1

Conflicting examples:
                                                            Text  Class
ரெண்டு பேரையும் செருப்புலே அடிச்சு தொரத்தரதை தவிர வேறு வழி இல்லை      0
ரெண்டு பேரையும் செருப்புலே அடிச்சு தொரத்தரதை தவிர வேறு வழி இல்லை      1

Rows after removing conflicts: 25944
Duplicate rows removed: 57

FINAL CLEAN DATASET
Rows: 25887
Unique texts: 25887

Class distribution:
Class
0    13330
1    12557
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 4 — FIXED STRATIFIED SPLIT
# ============================================================

train_df, validation_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["Class"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)

print("=" * 70)
print("FIXED STRATIFIED SPLIT")
print("=" * 70)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)

print()
print("Train distribution:")
print(train_df["Class"].value_counts().sort_index())

print()
print("Validation distribution:")
print(validation_df["Class"].value_counts().sort_index())

# ------------------------------------------------------------
# Leakage check
# ------------------------------------------------------------

train_texts = set(train_df["Text"])
val_texts = set(validation_df["Text"])

overlap = train_texts.intersection(val_texts)

print()
print("Text overlap:", len(overlap))

assert len(overlap) == 0

print("✓ No train/validation text overlap")

# ------------------------------------------------------------
# Save fixed split
# ------------------------------------------------------------

train_split_path = os.path.join(
    RESULTS_DIR,
    "fixed_train_split_seed42.csv"
)

val_split_path = os.path.join(
    RESULTS_DIR,
    "fixed_validation_split_seed42.csv"
)

train_df.to_csv(train_split_path, index=False)
validation_df.to_csv(val_split_path, index=False)

print()
print("Saved:")
print(train_split_path)
print(val_split_path)

FIXED STRATIFIED SPLIT
Train: (20709, 2)
Validation: (5178, 2)

Train distribution:
Class
0    10664
1    10045
Name: count, dtype: int64

Validation distribution:
Class
0    2666
1    2512
Name: count, dtype: int64

Text overlap: 0
✓ No train/validation text overlap

Saved:
/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_train_split_seed42.csv
/content/drive/MyDrive/DravidianLangTech_2026/results/fixed_validation_split_seed42.csv


In [ ]:
# ============================================================
# CELL 5 — CORRECT 11-FEATURE EXTRACTION
# ============================================================

import unicodedata

FEATURE_COLUMNS = [
    "text_length",
    "word_count",
    "tamil_chars",
    "latin_chars",
    "digits",
    "tamil_ratio",
    "latin_ratio",
    "emoji_count",
    "punctuation_count",
    "repeated_char_patterns",
    "english_word_count"
]


def is_tamil_char(ch):
    """
    Tamil Unicode block:
    U+0B80 - U+BFF
    """
    return "\u0B80" <= ch <= "\u0BFF"


def is_latin_char(ch):
    """
    Detect Latin alphabet characters.
    Handles both ASCII and extended Latin characters.
    """

    if ("A" <= ch <= "Z") or ("a" <= ch <= "z"):
        return True

    name = unicodedata.name(ch, "")

    return "LATIN" in name


def count_emojis(text):
    """
    Approximate emoji detection using Unicode ranges.
    """

    count = 0

    for ch in text:

        code = ord(ch)

        if (
            0x1F300 <= code <= 0x1FAFF
            or 0x2600 <= code <= 0x27BF
            or 0x2300 <= code <= 0x23FF
        ):
            count += 1

    return count


def count_repeated_patterns(text):
    """
    Counts repeated-character patterns such as:
    aaa
    !!!
    ...
    """

    pattern = r"(.)\1{2,}"

    return len(re.findall(pattern, text))


def extract_features_correct(texts):

    rows = []

    for text in texts:

        if pd.isna(text):
            text = ""

        text = str(text)

        # ----------------------------------------------------
        # Basic counts
        # ----------------------------------------------------

        text_length = len(text)

        words = text.split()

        word_count = len(words)

        tamil_chars = sum(
            is_tamil_char(ch)
            for ch in text
        )

        latin_chars = sum(
            is_latin_char(ch)
            for ch in text
        )

        digits = sum(
            ch.isdigit()
            for ch in text
        )

        # ----------------------------------------------------
        # Ratios
        # ----------------------------------------------------

        if text_length > 0:

            tamil_ratio = tamil_chars / text_length

            latin_ratio = latin_chars / text_length

        else:

            tamil_ratio = 0.0
            latin_ratio = 0.0

        # ----------------------------------------------------
        # Emoji
        # ----------------------------------------------------

        emoji_count = count_emojis(text)

        # ----------------------------------------------------
        # Punctuation
        # ----------------------------------------------------

        punctuation_count = sum(
            unicodedata.category(ch).startswith("P")
            for ch in text
        )

        # ----------------------------------------------------
        # Repeated characters
        # ----------------------------------------------------

        repeated_char_patterns = count_repeated_patterns(text)

        # ----------------------------------------------------
        # English word count
        # ----------------------------------------------------

        english_words = re.findall(
            r"\b[A-Za-z]+(?:['-][A-Za-z]+)*\b",
            text
        )

        english_word_count = len(english_words)

        rows.append({
            "text_length": text_length,
            "word_count": word_count,
            "tamil_chars": tamil_chars,
            "latin_chars": latin_chars,
            "digits": digits,
            "tamil_ratio": tamil_ratio,
            "latin_ratio": latin_ratio,
            "emoji_count": emoji_count,
            "punctuation_count": punctuation_count,
            "repeated_char_patterns": repeated_char_patterns,
            "english_word_count": english_word_count
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# CELL 6 — FEATURE EXTRACTOR SANITY TEST
# ============================================================

test_examples = pd.Series([
    "வணக்கம் தமிழ்",
    "hello world",
    "இது ஒரு test comment",
    "Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க",
    "எல்லாமே script thaan",
    "தமிழ்123"
])

example_features = extract_features_correct(
    test_examples
)

example_output = pd.DataFrame({
    "Text": test_examples,
    "Features": example_features.to_dict("records")
})

print("=" * 70)
print("FEATURE EXTRACTOR SANITY TEST")
print("=" * 70)

display(example_output)

print()
print("Expected:")
print("✓ hello world → Latin")
print("✓ test comment → Tamil + Latin")
print("✓ Suganthi ... → Tamil + Latin")

FEATURE EXTRACTOR SANITY TEST


,Text,Features
0,வணக்கம் தமிழ்,"{'text_length': 13, 'word_count': 2, 'tamil_ch..."
1,hello world,"{'text_length': 11, 'word_count': 2, 'tamil_ch..."
2,இது ஒரு test comment,"{'text_length': 20, 'word_count': 4, 'tamil_ch..."
3,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க,"{'text_length': 42, 'word_count': 5, 'tamil_ch..."
4,எல்லாமே script thaan,"{'text_length': 20, 'word_count': 3, 'tamil_ch..."
5,தமிழ்123,"{'text_length': 8, 'word_count': 1, 'tamil_cha..."



Expected:
✓ hello world → Latin
✓ test comment → Tamil + Latin
✓ Suganthi ... → Tamil + Latin


In [ ]:
# ============================================================
# CELL 6 — FEATURE EXTRACTOR SANITY TEST
# ============================================================

test_examples = pd.Series([
    "வணக்கம் தமிழ்",
    "hello world",
    "இது ஒரு test comment",
    "Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க",
    "எல்லாமே script thaan",
    "தமிழ்123"
])

example_features = extract_features_correct(
    test_examples
)

example_output = pd.DataFrame({
    "Text": test_examples,
    "Features": example_features.to_dict("records")
})

print("=" * 70)
print("FEATURE EXTRACTOR SANITY TEST")
print("=" * 70)

display(example_output)

print()
print("Expected:")
print("✓ hello world → Latin")
print("✓ test comment → Tamil + Latin")
print("✓ Suganthi ... → Tamil + Latin")

FEATURE EXTRACTOR SANITY TEST


,Text,Features
0,வணக்கம் தமிழ்,"{'text_length': 13, 'word_count': 2, 'tamil_ch..."
1,hello world,"{'text_length': 11, 'word_count': 2, 'tamil_ch..."
2,இது ஒரு test comment,"{'text_length': 20, 'word_count': 4, 'tamil_ch..."
3,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க,"{'text_length': 42, 'word_count': 5, 'tamil_ch..."
4,எல்லாமே script thaan,"{'text_length': 20, 'word_count': 3, 'tamil_ch..."
5,தமிழ்123,"{'text_length': 8, 'word_count': 1, 'tamil_cha..."



Expected:
✓ hello world → Latin
✓ test comment → Tamil + Latin
✓ Suganthi ... → Tamil + Latin


In [ ]:
# ============================================================
# CELL 7 — CREATE CORRECT FEATURE DATASETS
# ============================================================

train_features = extract_features_correct(
    train_df["Text"]
)

validation_features = extract_features_correct(
    validation_df["Text"]
)

# ------------------------------------------------------------
# Combine with labels
# ------------------------------------------------------------

train_feature_df = pd.concat(
    [
        train_df[["Text", "Class"]].reset_index(drop=True),
        train_features.reset_index(drop=True)
    ],
    axis=1
)

validation_feature_df = pd.concat(
    [
        validation_df[["Text", "Class"]].reset_index(drop=True),
        validation_features.reset_index(drop=True)
    ],
    axis=1
)

print("=" * 70)
print("CORRECT FEATURE DATASETS")
print("=" * 70)

print("Train:", train_feature_df.shape)
print("Validation:", validation_feature_df.shape)

print()
print("Columns:")
print(train_feature_df.columns.tolist())

CORRECT FEATURE DATASETS
Train: (20709, 13)
Validation: (5178, 13)

Columns:
['Text', 'Class', 'text_length', 'word_count', 'tamil_chars', 'latin_chars', 'digits', 'tamil_ratio', 'latin_ratio', 'emoji_count', 'punctuation_count', 'repeated_char_patterns', 'english_word_count']


In [ ]:
# ============================================================
# DIAGNOSTIC — WHY ARE ENGLISH WORDS ZERO?
# ============================================================

print("=" * 70)
print("ENGLISH / LATIN DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# Test regex independently
# ------------------------------------------------------------

english_test_sentences = [
    "hello world",
    "இது ஒரு test comment",
    "Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க",
    "எல்லாமே script thaan",
    "Coming April Salary",
    "Tik tok nallathu than"
]

print("\nREGEX TEST")
print("-" * 70)

for text in english_test_sentences:

    latin_chars = sum(
        is_latin_char(ch)
        for ch in text
    )

    english_words = re.findall(
        r"\b[A-Za-z]+(?:['-][A-Za-z]+)*\b",
        text
    )

    print("\nText:", text)
    print("Latin chars:", latin_chars)
    print("English words:", english_words)
    print("English count:", len(english_words))


# ------------------------------------------------------------
# Search actual TRAIN data
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("SEARCHING TRAIN DATA")
print("=" * 70)

latin_mask = train_df["Text"].apply(
    lambda x: any(is_latin_char(ch) for ch in str(x))
)

english_mask = train_df["Text"].apply(
    lambda x: len(
        re.findall(
            r"\b[A-Za-z]+(?:['-][A-Za-z]+)*\b",
            str(x)
        )
    ) > 0
)

print("Latin-containing train rows:",
      latin_mask.sum())

print("English-word train rows:",
      english_mask.sum())


print("\nLATIN-CONTAINING TRAIN EXAMPLES")
print("-" * 70)

if latin_mask.sum() > 0:

    display(
        train_df.loc[
            latin_mask,
            ["Text", "Class"]
        ].head(30)
    )

else:

    print("NO LATIN TEXT FOUND")


print("\nENGLISH-WORD TRAIN EXAMPLES")
print("-" * 70)

if english_mask.sum() > 0:

    display(
        train_df.loc[
            english_mask,
            ["Text", "Class"]
        ].head(30)
    )

else:

    print("NO ENGLISH WORDS FOUND")

ENGLISH / LATIN DIAGNOSTIC

REGEX TEST
----------------------------------------------------------------------

Text: hello world
Latin chars: 10
English words: ['hello', 'world']
English count: 2

Text: இது ஒரு test comment
Latin chars: 11
English words: ['test', 'comment']
English count: 2

Text: Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க
Latin chars: 8
English words: ['Suganthi']
English count: 1

Text: எல்லாமே script thaan
Latin chars: 11
English words: ['script', 'thaan']
English count: 2

Text: Coming April Salary
Latin chars: 17
English words: ['Coming', 'April', 'Salary']
English count: 3

Text: Tik tok nallathu than
Latin chars: 18
English words: ['Tik', 'tok', 'nallathu', 'than']
English count: 4


SEARCHING TRAIN DATA
Latin-containing train rows: 4
English-word train rows: 0

LATIN-CONTAINING TRAIN EXAMPLES
----------------------------------------------------------------------


,Text,Class
953,உ எமோஷனல் ஆகி எங்களுக்கும் எமோஷனல் ஆக்கிட்டியேâ€¦,0
8881,டோரா புஜ்ஜியை விட்டா ராகேஷ் ஏன் அவங்க ã ஆ,0
13235,ஹை அண்ணா நான் உங்கள் பெரிய ã,0
15952,ஆம்பள குரல் ஆஆஆ முகம் இவளுக்கு பாக்க முடியல தூ...,0



ENGLISH-WORD TRAIN EXAMPLES
----------------------------------------------------------------------
NO ENGLISH WORDS FOUND


In [ ]:
# ============================================================
# ORIGINAL DATASET LATIN / ENGLISH AUDIT
# ============================================================

print("=" * 70)
print("ORIGINAL TRAIN DATASET AUDIT")
print("=" * 70)

original_latin_mask = train_raw["Text"].apply(
    lambda x: any(is_latin_char(ch) for ch in str(x))
)

original_english_mask = train_raw["Text"].apply(
    lambda x: len(
        re.findall(
            r"\b[A-Za-z]+(?:['-][A-Za-z]+)*\b",
            str(x)
        )
    ) > 0
)

print("Original rows:", len(train_raw))

print(
    "Latin-containing:",
    original_latin_mask.sum()
)

print(
    "English-word samples:",
    original_english_mask.sum()
)

print()
print(
    "Latin percentage:",
    f"{100 * original_latin_mask.mean():.2f}%"
)

print(
    "English-word percentage:",
    f"{100 * original_english_mask.mean():.2f}%"
)

print()
print("Examples:")
display(
    train_raw.loc[
        original_latin_mask,
        ["Text", "Class"]
    ].head(30)
)

ORIGINAL TRAIN DATASET AUDIT
Original rows: 25946
Latin-containing: 5
English-word samples: 0

Latin percentage: 0.02%
English-word percentage: 0.00%

Examples:


,Text,Class
9688,சூப்பர் ஜாரா அக்கா நான் மிஷ்ரியா நான் உங்க ã அ...,0
11519,ஆம்பள குரல் ஆஆஆ முகம் இவளுக்கு பாக்க முடியல தூ...,0
14904,டோரா புஜ்ஜியை விட்டா ராகேஷ் ஏன் அவங்க ã ஆ,0
15608,உ எமோஷனல் ஆகி எங்களுக்கும் எமோஷனல் ஆக்கிட்டியேâ€¦,0
17110,ஹை அண்ணா நான் உங்கள் பெரிய ã,0


In [ ]:
# ============================================================
# ORIGINAL vs CLEAN vs TRAIN vs VALIDATION
# ============================================================

def audit_script(df, name):

    latin_count = df["Text"].apply(
        lambda x: any(
            is_latin_char(ch)
            for ch in str(x)
        )
    ).sum()

    english_count = df["Text"].apply(
        lambda x: len(
            re.findall(
                r"\b[A-Za-z]+(?:['-][A-Za-z]+)*\b",
                str(x)
            )
        ) > 0
    ).sum()

    print(
        f"{name:20s} "
        f"Rows={len(df):6d} | "
        f"Latin={latin_count:5d} | "
        f"English={english_count:5d}"
    )


print("=" * 70)
print("SCRIPT DISTRIBUTION THROUGH PIPELINE")
print("=" * 70)

audit_script(train_raw, "Original")
audit_script(df, "Clean")
audit_script(train_df, "Train")
audit_script(validation_df, "Validation")

SCRIPT DISTRIBUTION THROUGH PIPELINE
Original             Rows= 25946 | Latin=    5 | English=    0
Clean                Rows= 25887 | Latin=    5 | English=    0
Train                Rows= 20709 | Latin=    4 | English=    0
Validation           Rows=  5178 | Latin=    1 | English=    0


In [ ]:
# ============================================================
# DEFINITIVE SCRIPT / TANGLISH DIAGNOSTIC
# ============================================================

import re
import unicodedata
import pandas as pd
import numpy as np

print("=" * 70)
print("DEFINITIVE SCRIPT / TANGLISH DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# Direct Unicode detectors
# ------------------------------------------------------------

def count_tamil_chars(text):
    if not isinstance(text, str):
        return 0
    return sum(
        1 for ch in text
        if '\u0B80' <= ch <= '\u0BFF'
    )


def count_latin_chars(text):
    if not isinstance(text, str):
        return 0

    return sum(
        1 for ch in text
        if (
            ('A' <= ch <= 'Z') or
            ('a' <= ch <= 'z')
        )
    )


def extract_latin_words(text):
    if not isinstance(text, str):
        return []

    return re.findall(
        r'\b[A-Za-z]+\b',
        text
    )


def script_type(text):
    tamil = count_tamil_chars(text)
    latin = count_latin_chars(text)

    if tamil > 0 and latin > 0:
        return "Tamil+Latin"
    elif tamil > 0:
        return "Tamil"
    elif latin > 0:
        return "Latin"
    else:
        return "Other"


# ------------------------------------------------------------
# Apply directly to ORIGINAL text
# ------------------------------------------------------------

def diagnostic(df, name):

    temp = df.copy()

    temp["__tamil_chars"] = temp["Text"].apply(
        count_tamil_chars
    )

    temp["__latin_chars"] = temp["Text"].apply(
        count_latin_chars
    )

    temp["__latin_words"] = temp["Text"].apply(
        extract_latin_words
    )

    temp["__english_word_count"] = temp[
        "__latin_words"
    ].apply(len)

    temp["__script"] = temp["Text"].apply(
        script_type
    )

    print()
    print("-" * 70)
    print(name)
    print("-" * 70)

    print("Rows:", len(temp))

    print()
    print("Script distribution:")
    print(
        temp["__script"].value_counts()
    )

    print()
    print("Latin-containing:",
          (temp["__latin_chars"] > 0).sum())

    print(
        "English-word containing:",
        (temp["__english_word_count"] > 0).sum()
    )

    # --------------------------------------------------------
    # Examples
    # --------------------------------------------------------

    latin_rows = temp[
        temp["__latin_chars"] > 0
    ]

    print()
    print("Latin-containing examples:")

    if len(latin_rows) > 0:

        display(
            latin_rows[
                [
                    "Text",
                    "__script",
                    "__latin_chars",
                    "__english_word_count"
                ]
            ].head(20)
        )

    else:
        print("NONE")

    return temp


# ------------------------------------------------------------
# IMPORTANT:
# Use the ORIGINAL train/validation split files
# ------------------------------------------------------------

TRAIN_SPLIT = (
    "/content/drive/MyDrive/DravidianLangTech_2026/"
    "results/fixed_train_split_seed42.csv"
)

VAL_SPLIT = (
    "/content/drive/MyDrive/DravidianLangTech_2026/"
    "results/fixed_validation_split_seed42.csv"
)

TEST_PATH = (
    "/content/drive/MyDrive/DravidianLangTech_2026/data/"
    "TestV2 - testV2.csv"
)


train_raw = pd.read_csv(TRAIN_SPLIT)
val_raw = pd.read_csv(VAL_SPLIT)
test_raw = pd.read_csv(TEST_PATH)

print("\nLoaded:")
print("Train:", train_raw.shape)
print("Validation:", val_raw.shape)
print("Test:", test_raw.shape)


train_diag = diagnostic(
    train_raw,
    "TRAIN"
)

val_diag = diagnostic(
    val_raw,
    "VALIDATION"
)

test_diag = diagnostic(
    test_raw,
    "OFFICIAL TEST"
)

DEFINITIVE SCRIPT / TANGLISH DIAGNOSTIC

Loaded:
Train: (20709, 2)
Validation: (5178, 2)
Test: (913, 1)

----------------------------------------------------------------------
TRAIN
----------------------------------------------------------------------
Rows: 20709

Script distribution:
__script
Tamil    20699
Other       10
Name: count, dtype: int64

Latin-containing: 0
English-word containing: 0

Latin-containing examples:
NONE

----------------------------------------------------------------------
VALIDATION
----------------------------------------------------------------------
Rows: 5178

Script distribution:
__script
Tamil    5176
Other       2
Name: count, dtype: int64

Latin-containing: 0
English-word containing: 0

Latin-containing examples:
NONE

----------------------------------------------------------------------
OFFICIAL TEST
----------------------------------------------------------------------
Rows: 913

Script distribution:
__script
Tamil          705
Tamil+Latin    208


,Text,__script,__latin_chars,__english_word_count
3,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க கார...,Tamil+Latin,14,2
4,எல்லாமே script thaan. ஷகீலா உங்க scriptum அரும...,Tamil+Latin,37,6
9,நாட்டுக்கு ரொம்ப முக்கியம் இது.... மக்கள கெடு...,Tamil+Latin,17,2
11,மூஞ்சி pundai ய பார்த்தா 1000 சுன்னிய பார்த்து...,Tamil+Latin,6,1
12,Coming April ல இருந்து Salary எல்லாம் குறைய போ...,Tamil+Latin,32,5
20,அவ அழரத விட நீங்க போடுர BGMஅ கேட்டா தான்டா பயம...,Tamil+Latin,3,0
28,இரண்டு வல்கர் ponugala சாக்கடை. கட்டி வச்சி அட...,Tamil+Latin,8,1
30,தயவு செய்து இதை பார்த்தாவது பெண்கள் திருந்தவும...,Tamil+Latin,6,1
49,இரண்டு ponnugalum தெரு நாய்ங்க தூ தூ,Tamil+Latin,10,1
52,Tik tok nallathu than but இந்தியா மக்கள் தான் ...,Tamil+Latin,38,6


In [ ]:
# ============================================================
# SHOW ALL LATIN-CONTAINING TRAIN SAMPLES
# ============================================================

train_latin = train_diag[
    train_diag["__latin_chars"] > 0
]

print("=" * 70)
print("LATIN-CONTAINING TRAIN SAMPLES")
print("=" * 70)

print("Count:", len(train_latin))

display(
    train_latin[
        [
            "Text",
            "Class",
            "__script",
            "__latin_chars",
            "__english_word_count"
        ]
    ].to_string(index=False)
)

LATIN-CONTAINING TRAIN SAMPLES
Count: 0


'Empty DataFrame\nColumns: [Text, Class, __script, __latin_chars, __english_word_count]\nIndex: []'

In [ ]:
# ============================================================
# SHOW LATIN-CONTAINING VALIDATION SAMPLES
# ============================================================

val_latin = val_diag[
    val_diag["__latin_chars"] > 0
]

print("=" * 70)
print("LATIN-CONTAINING VALIDATION SAMPLES")
print("=" * 70)

print("Count:", len(val_latin))

display(
    val_latin[
        [
            "Text",
            "Class",
            "__script",
            "__latin_chars",
            "__english_word_count"
        ]
    ].to_string(index=False)
)

LATIN-CONTAINING VALIDATION SAMPLES
Count: 0


'Empty DataFrame\nColumns: [Text, Class, __script, __latin_chars, __english_word_count]\nIndex: []'

In [ ]:
# ============================================================
# ORIGINAL DATASET SCRIPT AUDIT
# ============================================================

ORIGINAL_PATH = (
    "/content/drive/MyDrive/DravidianLangTech_2026/"
    "data/train2.csv"
)

original_df = pd.read_csv(
    ORIGINAL_PATH
)

original_diag = diagnostic(
    original_df,
    "ORIGINAL train2.csv"
)


----------------------------------------------------------------------
ORIGINAL train2.csv
----------------------------------------------------------------------
Rows: 25946

Script distribution:
__script
Tamil    25934
Other       12
Name: count, dtype: int64

Latin-containing: 0
English-word containing: 0

Latin-containing examples:
NONE


In [ ]:
print("=" * 70)
print("ORIGINAL → CLEAN → TRAIN → VALIDATION")
print("=" * 70)

for name, df in [
    ("Original", original_diag),
    ("Clean", clean_df if "clean_df" in globals() else None),
    ("Train", train_diag),
    ("Validation", val_diag),
]:

    if df is None:
        continue

    latin = (
        df["__latin_chars"] > 0
    ).sum()

    english = (
        df["__english_word_count"] > 0
    ).sum()

    print(
        f"{name:<15}"
        f"Rows={len(df):6d} | "
        f"Latin={latin:6d} | "
        f"English={english:6d}"
    )

ORIGINAL → CLEAN → TRAIN → VALIDATION
Original       Rows= 25946 | Latin=     0 | English=     0
Train          Rows= 20709 | Latin=     0 | English=     0
Validation     Rows=  5178 | Latin=     0 | English=     0
